# 34 — Final Streamlit Dashboard Assets

**Origin:** Consolidates Existing Previous Versions of Notebooks 29 and 34, Pre-refactor\
**Refactor status:** Finished\
**Validation status:** Finished\
**Completion gate passed:** Yes\
**Output root:** `outputs/34_final_streamlit_dashboard_assets/`\
**Depends on:** Notebooks 01–33

## Purpose

Package the validated evidence produced by Notebooks 01–33 into a lightweight, normalized, manifest-driven asset layer for the final Streamlit dashboard.

This notebook does not run restoration inference, recompute scientific metrics, create new statistical tests, or construct a combined quality or trustworthiness score. It selects, aggregates, reshapes, indexes, and validates existing canonical evidence for interactive presentation.

## Approved dashboard architecture

The application contains eight principal pages:

1. **Overview**
2. **Study Design**
3. **Metric Framework**
4. **Model Performance**
5. **Robustness & Uncertainty**
6. **Trustworthiness & XAI**
7. **Case Explorer**
8. **Reports & Reproducibility**

Each page must balance concise conclusions, headline indicators, analytical plots, paintings or diagnostic images, limitations, and expandable technical evidence.

## Evidence population

The dashboard package represents:

- 50 controlled paintings;
- 525 registered experimental cases;
- 410 restoration-eligible cases;
- 1,785 approved candidates;
- three fully evaluated core models;
- one bounded ten-case SDXL feasibility population;
- 130 canonical Stable Diffusion uncertainty groups;
- 35 damage-size uncertainty groups;
- 30 detailed case reports;
- 50 painting reports;
- four model reports;
- the final evaluation report and supporting method reports.

Representative cases determine initial presentation only. The complete applicable candidate and painting populations remain accessible through dashboard filters and indexes.

## Interpretation boundaries

- Visual plausibility is not equivalent to historical or restoration trustworthiness.
- Repeated-seed variability is empirical generative uncertainty, not calibrated confidence.
- Mask-placement variation is input robustness, not generative uncertainty.
- Prompt-arm variation is prompt sensitivity.
- Computational flags are diagnostic rules, not expert or conservation ground truth.
- CLIP and DINOv2 retrieval provides comparison context, not historical proof.
- SDXL remains a bounded feasibility evaluation.
- The dashboard is an inspection and decision-support interface, not a restoration or experiment tool.

## Canonical outputs

```text
data/dashboard_summary.json
data/dashboard_tables/
data/dashboard_indexes/
manifests/dashboard_assets.csv
manifests/run_manifest.json
manifests/artifacts.csv
validation/checks.csv

## Batch 1 — Contract, initialization, and upstream preflight

This batch:

- discovers the repository root without assuming the working directory;
- loads the locked Notebook 34 configuration;
- creates only the declared notebook-owned output directories;
- rejects stale pre-existing output files;
- validates the current inventory and project-path registry;
- verifies the dashboard architecture and governing-document contracts;
- requires completed run manifests from Notebooks 01–33;
- loads and validates all 41 declared canonical input tables;
- confirms exact row counts and required columns;
- applies a blocking gate before dashboard assets are assembled.

Progress is printed after every ten input tables.

This batch does not persist any canonical dashboard table, index, summary, manifest, or validation file.

In [1]:
from __future__ import annotations

import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display


# Discover the repository from either its root or the notebooks directory.
CURRENT_LOCATION = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (CURRENT_LOCATION, *CURRENT_LOCATION.parents)
        if (
            candidate
            / "config"
            / "evaluation"
            / "dashboard_assets.yaml"
        ).is_file()
        and (candidate / "src" / "restoration_eval").is_dir()
        and (candidate / "notebooks").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the painting-restoration-eval repository root."
    )

SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))


from restoration_eval.dashboard_assets import (
    CONFIG_SCHEMA_VERSION,
    DASHBOARD_PACKAGE_SCHEMA_VERSION,
    MODULE_VERSION as DASHBOARD_ASSETS_MODULE_VERSION,
    OUTPUT_SCHEMAS,
    create_output_directories,
    load_dashboard_config,
    load_upstream_manifests,
    resolve_output_paths,
    validate_input_table_contracts,
)
from restoration_eval.paths import (
    PATHS_MODULE_VERSION,
    PROJECT_PATHS_SCHEMA_VERSION,
    validate_project_paths_registry,
)
from restoration_eval.validation import (
    VALIDATION_MODULE_VERSION,
    ValidationCollector,
)


RUN_STARTED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

VALIDATION = ValidationCollector()


def collect_helper_checks(checks: pd.DataFrame) -> None:
    """Transfer normalized helper checks into the notebook collector."""

    if checks.empty:
        return

    for record in checks.to_dict(orient="records"):
        VALIDATION.add(
            validation_stage=str(record["validation_stage"]),
            check_id=str(record["check_id"]),
            check_description=str(record["check_name"]).replace("_", " "),
            severity=str(record["severity"]),
            expected=record["expected"],
            observed=record["observed"],
            passed=bool(record["passed"]),
            details=record.get("issue", ""),
        )


print("Notebook 34 environment initialized.")
print("Project root:", PROJECT_ROOT)
print("Run started:", RUN_STARTED_AT_UTC)
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print(
    "Helper versions:",
    {
        "dashboard_assets": DASHBOARD_ASSETS_MODULE_VERSION,
        "validation": VALIDATION_MODULE_VERSION,
        "paths": PATHS_MODULE_VERSION,
    },
)

Notebook 34 environment initialized.
Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Run started: 2026-09-03T06:34:26Z
Python: 3.12.6
Platform: Windows-11-10.0.26200-SP0
Helper versions: {'dashboard_assets': '1.0.0', 'validation': '1.0.0', 'paths': '1.0.0'}


In [2]:
CONFIG_PATH = (
    PROJECT_ROOT
    / "config"
    / "evaluation"
    / "dashboard_assets.yaml"
)

CONFIG = load_dashboard_config(CONFIG_PATH)
SETTINGS = CONFIG["dashboard_assets"]

POPULATION = SETTINGS["population"]
PRESENTATION = SETTINGS["presentation"]
OUTPUT_SPEC = SETTINGS["output"]
PAGE_PLAN = pd.DataFrame(SETTINGS["pages"])

OUTPUT_PATHS = resolve_output_paths(PROJECT_ROOT, CONFIG)
OUTPUT_ROOT = OUTPUT_PATHS["root"]


# Inspect before creating directories so stale files remain detectable.
OUTPUT_ROOT_PREEXISTED = OUTPUT_ROOT.exists()
PREEXISTING_OUTPUT_FILES = (
    sorted(
        path.relative_to(OUTPUT_ROOT).as_posix()
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    )
    if OUTPUT_ROOT_PREEXISTED
    else []
)

create_output_directories(OUTPUT_PATHS)


EXPECTED_PAGE_IDS = [
    "overview",
    "study_design",
    "metric_framework",
    "model_performance",
    "robustness_uncertainty",
    "trustworthiness_xai",
    "case_explorer",
    "reports_reproducibility",
]

CONTRACT_CHECKS = (
    (
        "config_schema",
        "Configuration uses the approved Notebook 34 schema",
        CONFIG_SCHEMA_VERSION,
        CONFIG["config_schema_version"],
        CONFIG["config_schema_version"] == CONFIG_SCHEMA_VERSION,
    ),
    (
        "dashboard_package_schema",
        "Dashboard package schema is locked",
        DASHBOARD_PACKAGE_SCHEMA_VERSION,
        SETTINGS["dashboard_schema_version"],
        (
            SETTINGS["dashboard_schema_version"]
            == DASHBOARD_PACKAGE_SCHEMA_VERSION
        ),
    ),
    (
        "notebook_identity",
        "Notebook identity is fixed",
        {
            "notebook_id": "34",
            "notebook_stem": "34_final_streamlit_dashboard_assets",
        },
        {
            "notebook_id": SETTINGS["notebook_id"],
            "notebook_stem": SETTINGS["notebook_stem"],
        },
        (
            SETTINGS["notebook_id"] == "34"
            and SETTINGS["notebook_stem"]
            == "34_final_streamlit_dashboard_assets"
        ),
    ),
    (
        "presentation_only_scope",
        "Notebook 34 does not create scientific evidence",
        False,
        SETTINGS["creates_new_scientific_evidence"],
        SETTINGS["creates_new_scientific_evidence"] is False,
    ),
    (
        "output_root",
        "Output root is notebook-owned",
        "outputs/34_final_streamlit_dashboard_assets",
        OUTPUT_SPEC["root"],
        (
            OUTPUT_SPEC["root"]
            == "outputs/34_final_streamlit_dashboard_assets"
        ),
    ),
    (
        "clean_output_root",
        "No stale files existed in the Notebook 34 output root",
        [],
        PREEXISTING_OUTPUT_FILES,
        not PREEXISTING_OUTPUT_FILES,
    ),
    (
        "approved_page_order",
        "The approved eight-page order is unchanged",
        EXPECTED_PAGE_IDS,
        PAGE_PLAN["page_id"].tolist(),
        PAGE_PLAN["page_id"].tolist() == EXPECTED_PAGE_IDS,
    ),
    (
        "principal_page_limit",
        "The dashboard remains below the ten-page limit",
        "<= 10",
        len(PAGE_PLAN),
        len(PAGE_PLAN) <= 10,
    ),
    (
        "dashboard_table_count",
        "Nine normalized dashboard tables are declared",
        9,
        int(POPULATION["dashboard_table_count"]),
        int(POPULATION["dashboard_table_count"]) == 9,
    ),
    (
        "dashboard_index_count",
        "Five dashboard indexes are declared",
        5,
        int(POPULATION["dashboard_index_count"]),
        int(POPULATION["dashboard_index_count"]) == 5,
    ),
    (
        "normalized_dataframe_schemas",
        "Fourteen exact dataframe schemas are registered",
        14,
        len(OUTPUT_SCHEMAS),
        len(OUTPUT_SCHEMAS) == 14,
    ),
    (
        "input_table_contract_count",
        "All canonical input-table contracts are declared",
        41,
        len(SETTINGS["input_table_contracts"]),
        len(SETTINGS["input_table_contracts"]) == 41,
    ),
    (
        "upstream_manifest_count",
        "All upstream run manifests are declared",
        33,
        len(SETTINGS["upstream_manifests"]),
        len(SETTINGS["upstream_manifests"]) == 33,
    ),
)

for check_id, description, expected, observed, passed in CONTRACT_CHECKS:
    VALIDATION.add(
        validation_stage="batch_1_contract",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Notebook 34 differs from the approved preparation contract."
        ),
    )

VALIDATION.raise_for_blocking()

OUTPUT_CONTRACT_VIEW = pd.DataFrame(
    [
        {
            "output_key": key,
            "repository_relative_path": (
                path.relative_to(PROJECT_ROOT).as_posix()
            ),
            "kind": (
                "directory"
                if key.endswith("_dir") or key in {"root", "work_dir"}
                else "file"
            ),
            "exists_after_setup": path.exists(),
        }
        for key, path in OUTPUT_PATHS.items()
    ]
).sort_values(["kind", "output_key"]).reset_index(drop=True)

display(PAGE_PLAN)
display(OUTPUT_CONTRACT_VIEW)

print("Notebook contract passed.")
print("Pre-existing output files:", len(PREEXISTING_OUTPUT_FILES))
print("Declared output paths:", len(OUTPUT_PATHS))

,page_id,display_order,display_name,question
0,overview,1,Overview,What does the complete evaluation support?
1,study_design,2,Study Design,"What was evaluated, and under which controlled..."
2,metric_framework,3,Metric Framework,Why do regions and metric families change the ...
3,model_performance,4,Model Performance,"Which model performs better, under which condi..."
4,robustness_uncertainty,5,Robustness & Uncertainty,Where do restoration results become less depen...
5,trustworthiness_xai,6,Trustworthiness & XAI,"Why was a candidate flagged, and what evidence..."
6,case_explorer,7,Case Explorer,What evidence supports an individual restorati...
7,reports_reproducibility,8,Reports & Reproducibility,Can each conclusion be traced to validated evi...


,output_key,repository_relative_path,kind,exists_after_setup
0,dashboard_indexes_dir,outputs/34_final_streamlit_dashboard_assets/da...,directory,True
1,dashboard_tables_dir,outputs/34_final_streamlit_dashboard_assets/da...,directory,True
2,root,outputs/34_final_streamlit_dashboard_assets,directory,True
3,work_dir,outputs/34_final_streamlit_dashboard_assets/work,directory,True
4,artifacts_path,outputs/34_final_streamlit_dashboard_assets/ma...,file,False
5,case_index_path,outputs/34_final_streamlit_dashboard_assets/da...,file,False
6,compute_summary_path,outputs/34_final_streamlit_dashboard_assets/da...,file,False
7,dashboard_assets_path,outputs/34_final_streamlit_dashboard_assets/ma...,file,False
8,dashboard_summary_path,outputs/34_final_streamlit_dashboard_assets/da...,file,False
9,filter_options_path,outputs/34_final_streamlit_dashboard_assets/da...,file,False


Notebook contract passed.
Pre-existing output files: 0
Declared output paths: 23


In [3]:
INPUT_SPEC = SETTINGS["inputs"]

INVENTORY_PATH = PROJECT_ROOT / INPUT_SPEC["inventory_path"]
INVENTORY_RUN_PATH = PROJECT_ROOT / INPUT_SPEC["inventory_run_path"]
PROJECT_PATHS_PATH = PROJECT_ROOT / INPUT_SPEC["project_paths_path"]

with INVENTORY_RUN_PATH.open("r", encoding="utf-8-sig") as handle:
    INVENTORY_RUN = json.load(handle)

INVENTORY = pd.read_csv(INVENTORY_PATH, low_memory=False)

with PROJECT_PATHS_PATH.open("r", encoding="utf-8-sig") as handle:
    PROJECT_PATHS_REGISTRY = json.load(handle)

PROJECT_PATH_ERRORS = validate_project_paths_registry(
    PROJECT_PATHS_REGISTRY
)

IMPLEMENTATION_TEXT = (
    PROJECT_ROOT / INPUT_SPEC["implementation_guidelines_path"]
).read_text(encoding="utf-8-sig")

ROADMAP_TEXT = (
    PROJECT_ROOT / INPUT_SPEC["notebook_roadmap_path"]
).read_text(encoding="utf-8-sig")

AUDIT_TEXT = (
    PROJECT_ROOT / INPUT_SPEC["evidence_dependency_audit_path"]
).read_text(encoding="utf-8-sig")

COVERAGE_TEXT = (
    PROJECT_ROOT / INPUT_SPEC["evidence_coverage_path"]
).read_text(encoding="utf-8-sig")


PREPARATION_FILES = {
    "config/evaluation/dashboard_assets.yaml",
    "config/evaluation/evidence_coverage.yaml",
    "docs/evidence_dependency_audit.md",
    "docs/final_notebook_roadmap.md",
    "docs/refactoring_implementation_guidelines.md",
    "notebooks/34_final_streamlit_dashboard_assets.ipynb",
    "src/restoration_eval/dashboard_assets.py",
}

INVENTORY_PATHS = set(INVENTORY["relative_path"].astype(str))

CURRENT_PYTHON = (
    sys.version_info.major,
    sys.version_info.minor,
)
SUPPORTED_PYTHON = {(3, 11), (3, 12)}

INVENTORY_AND_GOVERNANCE_CHECKS = (
    (
        "inventory_status",
        "Inventory refresh completed",
        "completed",
        INVENTORY_RUN.get("status"),
        INVENTORY_RUN.get("status") == "completed",
    ),
    (
        "inventory_read_errors",
        "Inventory contains no read errors",
        0,
        int(INVENTORY_RUN["summary"]["read_error_count"]),
        int(INVENTORY_RUN["summary"]["read_error_count"]) == 0,
    ),
    (
        "inventory_repository_root",
        "Inventory belongs to this repository",
        str(PROJECT_ROOT),
        INVENTORY_RUN.get("repository_root"),
        (
            Path(INVENTORY_RUN["repository_root"]).resolve()
            == PROJECT_ROOT
        ),
    ),
    (
        "supported_python",
        "Kernel uses Python 3.11 or 3.12",
        ["3.11", "3.12"],
        f"{CURRENT_PYTHON[0]}.{CURRENT_PYTHON[1]}",
        CURRENT_PYTHON in SUPPORTED_PYTHON,
    ),
    (
        "preparation_files_in_inventory",
        "All Notebook 34 preparation files are indexed",
        sorted(PREPARATION_FILES),
        sorted(PREPARATION_FILES & INVENTORY_PATHS),
        PREPARATION_FILES.issubset(INVENTORY_PATHS),
    ),
    (
        "project_paths_schema",
        "Project paths use the canonical schema",
        PROJECT_PATHS_SCHEMA_VERSION,
        PROJECT_PATHS_REGISTRY.get("registry_schema_version"),
        (
            PROJECT_PATHS_REGISTRY.get("registry_schema_version")
            == PROJECT_PATHS_SCHEMA_VERSION
            and not PROJECT_PATH_ERRORS
        ),
    ),
    (
        "manual_notebook_editing_rule",
        "The assistant must not edit notebook cells directly",
        True,
        (
            "The assistant must not insert Batch 1 or any later cells"
            in IMPLEMENTATION_TEXT
        ),
        (
            "The assistant must not insert Batch 1 or any later cells"
            in IMPLEMENTATION_TEXT
        ),
    ),
    (
        "dashboard_mock_contract",
        "The dashboard mock-fidelity contract remains present",
        True,
        (
            "Dashboard design, mock fidelity, and implementation boundary"
            in IMPLEMENTATION_TEXT
        ),
        (
            "Dashboard design, mock fidelity, and implementation boundary"
            in IMPLEMENTATION_TEXT
        ),
    ),
    (
        "roadmap_entry",
        "Notebook 34 remains in the final roadmap",
        True,
        (
            "34_final_streamlit_dashboard_assets.ipynb"
            in ROADMAP_TEXT
        ),
        (
            "34_final_streamlit_dashboard_assets.ipynb"
            in ROADMAP_TEXT
        ),
    ),
    (
        "dependency_audit_contract",
        "The Notebook 34 preparation contract is recorded",
        True,
        (
            "Notebook 34 now has an approved preparation contract"
            in AUDIT_TEXT
        ),
        (
            "Notebook 34 now has an approved preparation contract"
            in AUDIT_TEXT
        ),
    ),
    (
        "evidence_coverage_contract",
        "The detailed Notebook 34 evidence gate is recorded",
        True,
        (
            'stem: "34_final_streamlit_dashboard_assets"'
            in COVERAGE_TEXT
            and "principal_page_count: 8" in COVERAGE_TEXT
            and "candidate_index_rows: 1785" in COVERAGE_TEXT
        ),
        (
            'stem: "34_final_streamlit_dashboard_assets"'
            in COVERAGE_TEXT
            and "principal_page_count: 8" in COVERAGE_TEXT
            and "candidate_index_rows: 1785" in COVERAGE_TEXT
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    INVENTORY_AND_GOVERNANCE_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_1_inventory_governance",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Inventory or governing-document preflight failed."
        ),
    )

VALIDATION.raise_for_blocking()

PREPARATION_INVENTORY = (
    INVENTORY.loc[
        INVENTORY["relative_path"].isin(PREPARATION_FILES),
        [
            "relative_path",
            "file_kind",
            "format",
            "size_bytes",
            "read_error_count",
        ],
    ]
    .sort_values("relative_path")
    .reset_index(drop=True)
)

display(PREPARATION_INVENTORY)

print("Inventory and governance preflight passed.")
print("Inventory run ID:", INVENTORY_RUN["inventory_run_id"])
print("Indexed files:", INVENTORY_RUN["summary"]["file_count"])
print("Preparation files indexed:", len(PREPARATION_INVENTORY))

,relative_path,file_kind,format,size_bytes,read_error_count
0,config/evaluation/dashboard_assets.yaml,yaml,yaml,20148,0
1,config/evaluation/evidence_coverage.yaml,yaml,yaml,72273,0
2,docs/evidence_dependency_audit.md,markdown,markdown,40359,0
3,docs/final_notebook_roadmap.md,markdown,markdown,82719,0
4,docs/refactoring_implementation_guidelines.md,markdown,markdown,68129,0
5,notebooks/34_final_streamlit_dashboard_assets....,notebook,notebook,640,0
6,src/restoration_eval/dashboard_assets.py,python,python,24431,0


Inventory and governance preflight passed.
Inventory run ID: inventory_20260902T231016Z_d7ee71cf
Indexed files: 18942
Preparation files indexed: 7


In [4]:
UPSTREAM_MANIFESTS, MANIFEST_CHECKS = load_upstream_manifests(
    PROJECT_ROOT,
    CONFIG,
)

collect_helper_checks(MANIFEST_CHECKS)
VALIDATION.raise_for_blocking()


def input_progress(
    completed: int,
    total: int,
    input_key: str,
) -> None:
    if completed % 10 == 0 or completed == total:
        print(
            f"Input contract preflight: "
            f"{completed}/{total} tables checked."
        )


INPUT_TABLES, INPUT_TABLE_CHECKS = validate_input_table_contracts(
    PROJECT_ROOT,
    CONFIG,
    progress_callback=input_progress,
)

collect_helper_checks(INPUT_TABLE_CHECKS)
VALIDATION.raise_for_blocking()


UPSTREAM_MANIFEST_SUMMARY = pd.DataFrame(
    [
        {
            "notebook_id": notebook_id,
            "notebook_name": manifest.get("notebook_name"),
            "run_id": manifest.get("run_id"),
            "run_status": manifest.get("run_status"),
            "git_commit": manifest.get("git_commit"),
        }
        for notebook_id, manifest in sorted(
            UPSTREAM_MANIFESTS.items()
        )
    ]
)

INPUT_TABLE_SUMMARY = pd.DataFrame(
    [
        {
            "input_key": input_key,
            "rows": len(frame),
            "columns": len(frame.columns),
            "expected_rows": int(
                SETTINGS["input_table_contracts"][input_key]["rows"]
            ),
            "status": (
                "passed"
                if len(frame)
                == int(
                    SETTINGS[
                        "input_table_contracts"
                    ][input_key]["rows"]
                )
                else "failed"
            ),
        }
        for input_key, frame in INPUT_TABLES.items()
    ]
).sort_values("input_key").reset_index(drop=True)

display(UPSTREAM_MANIFEST_SUMMARY)
display(INPUT_TABLE_SUMMARY)

print("Upstream manifests loaded:", len(UPSTREAM_MANIFESTS))
print("Canonical input tables loaded:", len(INPUT_TABLES))
print(
    "Total loaded table rows:",
    sum(len(frame) for frame in INPUT_TABLES.values()),
)

Input contract preflight: 10/41 tables checked.
Input contract preflight: 20/41 tables checked.
Input contract preflight: 30/41 tables checked.
Input contract preflight: 40/41 tables checked.
Input contract preflight: 41/41 tables checked.


,notebook_id,notebook_name,run_id,run_status,git_commit
0,01,Dataset Verification,run_424a69cf0c2c4a228d0919dc2d66258a,completed,4148e151e712f5bca81813f65c090c1b5b07e651
1,02,Image Preprocessing,run_5e54b2de32934e1c9303e7250b932404,completed,1bafcfa9e5d371c30d69862b26e0bb547a0344fb
2,03,Canonical Mask Generation,run_16536d2f39c54081b0dd58a00b5e9f01,completed,e897fb5632b244abd5b5bd51a4751475d57fce71
3,04,Canonical Damaged-Image Generation,run_bf312c29726843a4b3c5319e05bf49f0,completed,fdfc45ca3a00872a6970e8d221eac66757be67f8
4,05,Damage-Size Sensitivity Dataset Generation,run_519fd38cca4c45d69dc8f43e90cae621,completed,d4533d00a8b8de8a2354552b1c8abe1bd77abf9a
5,06,Mask Robustness Dataset Generation,run_3d910e4c145e44d18ba19233125634d6,completed,0b57f464b9d081d174e47ebb46f8c57c1bbe0606
6,07,Synthetic Degradation Dataset Generation,run_306a79f17cfb48edb73fa90d174317f0,completed,f15a4881ac53b04e530f44d444450323e79eade3
7,08,Experiment Contracts and Region Policy,run_a3734bc2121f401b87de4894717366ab,completed,2af42ac0ecf72fb716398a7a2cea6f299b8caf95
8,09,OpenCV Telea Restoration,run_eafa3b4f805e472688f1df65b894b96e,completed,c46feee50144854d0b7598df7cb24450ed4e98e8
9,10,LaMa Restoration,run_25e38d80461643fb91ba60cb551a9a2f,completed,609f1ac8bc39cbfd4e3e44560293d75bca496269


,input_key,rows,columns,expected_rows,status
0,ablation_results_path,7710,38,7710,passed
1,artworks_path,50,36,50,passed
2,canonical_cases_path,250,34,250,passed
3,canonical_masks_path,250,89,250,passed
4,canonical_uncertainty_path,130,69,130,passed
5,case_neighbors_path,100,35,100,passed
6,case_registry_path,525,14,525,passed
7,case_report_index_path,30,21,30,passed
8,compute_scalability_path,35,46,35,passed
9,damage_size_analysis_path,1901,49,1901,passed


Upstream manifests loaded: 33
Canonical input tables loaded: 41
Total loaded table rows: 232640


In [5]:
EXPECTED_MODEL_IDS = {
    "opencv_telea",
    "lama",
    "stable_diffusion_inpainting",
    "sdxl_inpainting",
}

OBSERVED_MODEL_IDS = set(
    INPUT_TABLES["model_cards_path"]["model_id"]
    .dropna()
    .astype(str)
)

FINAL_PREFLIGHT_CHECKS = (
    (
        "upstream_manifest_population",
        "All 33 upstream manifests were loaded",
        33,
        len(UPSTREAM_MANIFESTS),
        len(UPSTREAM_MANIFESTS) == 33,
    ),
    (
        "input_table_population",
        "All 41 canonical input tables were loaded",
        41,
        len(INPUT_TABLES),
        len(INPUT_TABLES) == 41,
    ),
    (
        "painting_population",
        "Artwork table contains all controlled paintings",
        int(POPULATION["painting_count"]),
        len(INPUT_TABLES["artworks_path"]),
        (
            len(INPUT_TABLES["artworks_path"])
            == int(POPULATION["painting_count"])
        ),
    ),
    (
        "approved_candidate_population",
        "Explanation catalog contains every approved candidate",
        int(POPULATION["approved_candidate_count"]),
        len(INPUT_TABLES["explanation_cases_path"]),
        (
            len(INPUT_TABLES["explanation_cases_path"])
            == int(POPULATION["approved_candidate_count"])
        ),
    ),
    (
        "case_report_population",
        "Detailed case-report index is complete",
        int(POPULATION["selected_case_report_count"]),
        len(INPUT_TABLES["case_report_index_path"]),
        (
            len(INPUT_TABLES["case_report_index_path"])
            == int(POPULATION["selected_case_report_count"])
        ),
    ),
    (
        "painting_report_population",
        "Painting-report index covers all paintings",
        int(POPULATION["painting_report_count"]),
        len(INPUT_TABLES["painting_report_index_path"]),
        (
            len(INPUT_TABLES["painting_report_index_path"])
            == int(POPULATION["painting_report_count"])
        ),
    ),
    (
        "model_report_population",
        "Model-report index covers all approved models",
        int(POPULATION["model_report_count"]),
        len(INPUT_TABLES["model_report_index_path"]),
        (
            len(INPUT_TABLES["model_report_index_path"])
            == int(POPULATION["model_report_count"])
        ),
    ),
    (
        "model_identity_population",
        "Only the four approved model identities are present",
        sorted(EXPECTED_MODEL_IDS),
        sorted(OBSERVED_MODEL_IDS),
        OBSERVED_MODEL_IDS == EXPECTED_MODEL_IDS,
    ),
    (
        "canonical_uncertainty_population",
        "Canonical uncertainty groups are complete",
        int(POPULATION["canonical_uncertainty_group_count"]),
        INPUT_TABLES[
            "canonical_uncertainty_path"
        ]["uncertainty_group_id"].nunique(),
        (
            INPUT_TABLES[
                "canonical_uncertainty_path"
            ]["uncertainty_group_id"].nunique()
            == int(POPULATION["canonical_uncertainty_group_count"])
        ),
    ),
    (
        "damage_size_uncertainty_population",
        "Damage-size uncertainty groups are complete",
        int(POPULATION["damage_size_uncertainty_group_count"]),
        INPUT_TABLES[
            "damage_size_uncertainty_path"
        ]["uncertainty_group_id"].nunique(),
        (
            INPUT_TABLES[
                "damage_size_uncertainty_path"
            ]["uncertainty_group_id"].nunique()
            == int(POPULATION["damage_size_uncertainty_group_count"])
        ),
    ),
    (
        "no_canonical_files_written",
        "Batch 1 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    FINAL_PREFLIGHT_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_1_final_preflight",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "The loaded population differs from the approved contract."
        ),
    )

VALIDATION.raise_for_blocking()

VALIDATION_FRAME = VALIDATION.to_dataframe()

VALIDATION_STAGE_SUMMARY = (
    VALIDATION_FRAME
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

VALIDATION_STAGE_SUMMARY["failed"] = (
    VALIDATION_STAGE_SUMMARY["checks"]
    - VALIDATION_STAGE_SUMMARY["passed"]
)

display(VALIDATION_STAGE_SUMMARY)

print("Batch 1 preflight passed.")
print("Validation checks:", len(VALIDATION_FRAME))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Dashboard pages:", len(PAGE_PLAN))
print("Normalized dataframe schemas:", len(OUTPUT_SCHEMAS))
print("Output root:", OUTPUT_ROOT)
print("Canonical files persisted by this batch: 0")

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,13,13,0
1,batch_1_final_preflight,blocking,11,11,0
2,batch_1_inventory_governance,blocking,11,11,0
3,batch_1_preflight,blocking,222,222,0


Batch 1 preflight passed.
Validation checks: 257
Blocking failures: 0
Dashboard pages: 8
Normalized dataframe schemas: 14
Output root: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\34_final_streamlit_dashboard_assets
Canonical files persisted by this batch: 0


## Batch 2 — Population and filter normalization

This batch constructs the two complete dashboard browsing populations:

- one candidate-level index covering all 1,785 approved candidates;
- one painting-level index covering all 50 controlled paintings.

It joins validated report availability without restricting the population to the 30 selected detailed cases. It also constructs the shared model, experiment, metric, region, failure-category, and filter vocabularies used by later dashboard pages.

The batch validates:

- candidate, case, painting, model, and experiment populations;
- candidate and painting identifier uniqueness;
- model-specific candidate counts;
- complete original, damaged, mask, and restoration paths;
- all 50 painting-report links;
- all 30 selected case-report links;
- JSON evidence fields;
- controlled vocabulary values;
- exact normalized output schemas.

No canonical dashboard files are persisted by this batch.

In [6]:
from restoration_eval.dashboard_assets import (
    as_bool,
    is_repo_relative,
    json_list,
    parse_json_list,
    stable_id,
    validate_output_frame,
)


def normalized_text(series: pd.Series) -> pd.Series:
    """Return consistently blank-safe strings."""

    return series.fillna("").astype(str).str.strip()


def normalize_relative_paths(series: pd.Series) -> pd.Series:
    """Normalize repository paths without resolving or changing ownership."""

    return (
        normalized_text(series)
        .str.replace("\\", "/", regex=False)
    )


def repository_file_exists(value: object) -> bool:
    """Validate one non-empty repository-relative file path."""

    text = str(value).strip()

    if not text or not is_repo_relative(text):
        return False

    return (PROJECT_ROOT / text).is_file()


def all_json_lists_valid(
    frame: pd.DataFrame,
    columns: list[str],
) -> tuple[bool, list[str]]:
    """Check JSON-list fields without mutating their contents."""

    invalid: list[str] = []

    for column in columns:
        for row_index, value in frame[column].items():
            try:
                parse_json_list(value)
            except (TypeError, ValueError, json.JSONDecodeError) as exc:
                invalid.append(
                    f"{column}[{row_index}]: "
                    f"{type(exc).__name__}: {exc}"
                )

                if len(invalid) >= 20:
                    return False, invalid

    return not invalid, invalid


print("Batch 2 normalization helpers initialized.")

Batch 2 normalization helpers initialized.


In [7]:
EXPLANATION_SOURCE = (
    INPUT_TABLES["explanation_cases_path"]
    .copy()
    .reset_index(drop=True)
)

CASE_REPORT_LOOKUP = (
    INPUT_TABLES["case_report_index_path"][
        ["case_id", "report_path"]
    ]
    .rename(columns={"report_path": "case_report_path"})
    .copy()
)

PAINTING_REPORT_LOOKUP = (
    INPUT_TABLES["painting_report_index_path"][
        ["painting_id", "report_path"]
    ]
    .rename(columns={"report_path": "painting_report_path"})
    .copy()
)

if CASE_REPORT_LOOKUP["case_id"].duplicated().any():
    raise ValueError("Case-report lookup contains duplicate case IDs.")

if PAINTING_REPORT_LOOKUP["painting_id"].duplicated().any():
    raise ValueError(
        "Painting-report lookup contains duplicate painting IDs."
    )


CASE_SOURCE = (
    EXPLANATION_SOURCE
    .merge(
        CASE_REPORT_LOOKUP,
        on="case_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        PAINTING_REPORT_LOOKUP,
        on="painting_id",
        how="left",
        validate="many_to_one",
    )
)

if len(CASE_SOURCE) != len(EXPLANATION_SOURCE):
    raise ValueError(
        "Report joins changed the approved candidate population."
    )


def case_text(column: str) -> pd.Series:
    return normalized_text(CASE_SOURCE[column])


CASE_INDEX = pd.DataFrame(
    {
        "case_index_id": [
            stable_id("dashboard_case", candidate_id)
            for candidate_id in CASE_SOURCE["candidate_id"]
        ],
        "candidate_id": case_text("candidate_id"),
        "case_id": case_text("case_id"),
        "painting_id": case_text("painting_id"),
        "model_id": case_text("model_id"),
        "experiment_id": case_text("experiment_id"),
        "prompt_variant_id": case_text("prompt_variant_id"),
        "population_role": case_text("population_role"),
        "category": case_text("category"),
        "style_or_period": case_text("style_or_period"),
        "degradation_family": case_text("degradation_family"),
        "severity": case_text("severity"),
        "clean_path": case_text("clean_path"),
        "damaged_path": case_text("damaged_path"),
        "mask_path": case_text("mask_path"),
        "restored_path": case_text("restored_path"),
        "difference_paths_json": case_text(
            "difference_paths_json"
        ),
        "uncertainty_paths_json": case_text(
            "uncertainty_paths_json"
        ),
        "seam_paths_json": case_text("seam_paths_json"),
        "colour_paths_json": case_text("colour_paths_json"),
        "texture_paths_json": case_text("texture_paths_json"),
        "semantic_paths_json": case_text("semantic_paths_json"),
        "mask_boundary_paths_json": case_text(
            "mask_boundary_paths_json"
        ),
        "recommendation_category": case_text(
            "recommendation_category"
        ),
        "manual_review_required": (
            CASE_SOURCE["manual_review_required"].map(as_bool)
        ),
        "triggered_flag_ids_json": case_text(
            "triggered_flag_ids_json"
        ),
        "triggered_category_ids_json": case_text(
            "triggered_category_ids_json"
        ),
        "affected_regions_json": case_text(
            "affected_regions_json"
        ),
        "recommended_actions_json": case_text(
            "recommended_actions_json"
        ),
        "metric_disagreement_ids_json": case_text(
            "metric_disagreement_ids_json"
        ),
        "uncertainty_group_id": case_text(
            "uncertainty_group_id"
        ),
        "uncertainty_applicability": case_text(
            "uncertainty_applicability"
        ),
        "report_selected": (
            CASE_SOURCE["report_selected"].map(as_bool)
        ),
        "case_report_path": case_text("case_report_path"),
        "painting_report_path": case_text(
            "painting_report_path"
        ),
        "report_selection_roles_json": case_text(
            "report_selection_roles_json"
        ),
        "evidence_source_notebook_ids_json": case_text(
            "evidence_source_notebook_ids_json"
        ),
        "evidence_coverage_status": case_text(
            "evidence_coverage_status"
        ),
        "scope_status": case_text("scope_status"),
        "scope_note": case_text("scope_note"),
        "schema_version": SETTINGS[
            "expected_output_schemas"
        ]["case_index"],
        "status": case_text("status"),
        "issue": case_text("issue"),
    }
)

PATH_COLUMNS = [
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "case_report_path",
    "painting_report_path",
]

for column in PATH_COLUMNS:
    CASE_INDEX[column] = normalize_relative_paths(
        CASE_INDEX[column]
    )

CASE_INDEX = (
    validate_output_frame(CASE_INDEX, "case_index")
    .sort_values(
        [
            "painting_id",
            "case_id",
            "model_id",
            "prompt_variant_id",
            "candidate_id",
        ]
    )
    .reset_index(drop=True)
)

print("Candidate-level dashboard index constructed.")
print("Rows:", len(CASE_INDEX))
print("Cases:", CASE_INDEX["case_id"].nunique())
print("Paintings:", CASE_INDEX["painting_id"].nunique())
print("Models:", CASE_INDEX["model_id"].nunique())

Candidate-level dashboard index constructed.
Rows: 1785
Cases: 410
Paintings: 50
Models: 4


In [8]:
PAINTING_AGGREGATES = (
    CASE_INDEX
    .groupby("painting_id", sort=True)
    .agg(
        case_count=("case_id", "nunique"),
        candidate_count=("candidate_id", "size"),
        model_count=("model_id", "nunique"),
        has_uncertainty=(
            "uncertainty_group_id",
            lambda values: (
                values.astype(str).str.len().gt(0).any()
            ),
        ),
        has_selected_case_report=(
            "case_report_path",
            lambda values: (
                values.astype(str).str.len().gt(0).any()
            ),
        ),
    )
    .reset_index()
)

PAINTING_REPORT_METADATA = (
    INPUT_TABLES["painting_report_index_path"][
        [
            "painting_id",
            "report_path",
            "report_sha256",
            "self_contained",
        ]
    ]
    .rename(
        columns={
            "report_path": "painting_report_path",
            "report_sha256": "painting_report_sha256",
            "self_contained": "painting_report_self_contained",
        }
    )
)

PAINTING_SOURCE = (
    INPUT_TABLES["artworks_path"]
    .merge(
        PAINTING_AGGREGATES,
        on="painting_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        PAINTING_REPORT_METADATA,
        on="painting_id",
        how="left",
        validate="one_to_one",
    )
)


def painting_text(column: str) -> pd.Series:
    return normalized_text(PAINTING_SOURCE[column])


PAINTING_INDEX = pd.DataFrame(
    {
        "painting_index_id": [
            stable_id("dashboard_painting", painting_id)
            for painting_id in PAINTING_SOURCE["painting_id"]
        ],
        "painting_id": painting_text("painting_id"),
        "dataset_sort_index": (
            pd.to_numeric(
                PAINTING_SOURCE["dataset_sort_index"],
                errors="raise",
            ).astype(int)
        ),
        "title": painting_text("title"),
        "artist": painting_text("artist"),
        "date_or_period": painting_text("date_or_period"),
        "style_or_period": painting_text("style_or_period"),
        "category": painting_text("category"),
        "medium": painting_text("medium"),
        "source": painting_text("source"),
        "license": painting_text("license"),
        "rights_status": painting_text("rights_status"),
        "raw_image_path": normalize_relative_paths(
            PAINTING_SOURCE["raw_image_path"]
        ),
        "metadata_completeness_pct": pd.to_numeric(
            PAINTING_SOURCE["metadata_completeness_pct"],
            errors="coerce",
        ),
        "case_count": PAINTING_SOURCE["case_count"].astype(int),
        "candidate_count": (
            PAINTING_SOURCE["candidate_count"].astype(int)
        ),
        "model_count": PAINTING_SOURCE["model_count"].astype(int),
        "has_uncertainty": (
            PAINTING_SOURCE["has_uncertainty"].map(as_bool)
        ),
        "has_selected_case_report": (
            PAINTING_SOURCE[
                "has_selected_case_report"
            ].map(as_bool)
        ),
        "painting_report_path": normalize_relative_paths(
            PAINTING_SOURCE["painting_report_path"]
        ),
        "painting_report_sha256": painting_text(
            "painting_report_sha256"
        ),
        "painting_report_self_contained": (
            PAINTING_SOURCE[
                "painting_report_self_contained"
            ].map(as_bool)
        ),
        "source_notebook_ids_json": json_list(
            ["01", "29", "32"]
        ),
        "source_paths_json": json_list(
            [
                SETTINGS["inputs"]["artworks_path"],
                SETTINGS["inputs"]["explanation_cases_path"],
                SETTINGS["inputs"]["painting_report_index_path"],
            ]
        ),
        "schema_version": SETTINGS[
            "expected_output_schemas"
        ]["painting_index"],
        "status": "ok",
        "issue": "",
    }
)

PAINTING_INDEX = (
    validate_output_frame(PAINTING_INDEX, "painting_index")
    .sort_values(["dataset_sort_index", "painting_id"])
    .reset_index(drop=True)
)

print("Painting-level dashboard index constructed.")
print("Rows:", len(PAINTING_INDEX))
print(
    "Painting reports:",
    PAINTING_INDEX["painting_report_path"].ne("").sum(),
)

Painting-level dashboard index constructed.
Rows: 50
Painting reports: 50


In [9]:
MODEL_CARDS = (
    INPUT_TABLES["model_cards_path"]
    .copy()
    .sort_values("model_id")
    .reset_index(drop=True)
)

MODEL_DIMENSION = MODEL_CARDS[
    [
        "model_id",
        "display_name",
        "evaluation_status",
        "model_family",
        "methodological_role",
        "deterministic",
        "stochastic",
        "prompt_dependent",
        "evaluated_painting_count",
        "evaluated_case_count",
        "evaluated_candidate_count",
        "strengths_json",
        "weaknesses_json",
        "known_limitations_json",
        "status",
    ]
].copy()

EXPERIMENT_DIMENSION = (
    CASE_INDEX
    .groupby("experiment_id", sort=True)
    .agg(
        painting_count=("painting_id", "nunique"),
        case_count=("case_id", "nunique"),
        candidate_count=("candidate_id", "size"),
        model_count=("model_id", "nunique"),
    )
    .reset_index()
)

REGION_POLICY = INPUT_TABLES["region_policy_path"]
FAILURE_TAXONOMY = INPUT_TABLES["failure_taxonomy_path"]

FILTER_OPTIONS = {
    "schema_version": SETTINGS[
        "expected_output_schemas"
    ]["filter_options"],
    "page_ids": PAGE_PLAN["page_id"].tolist(),
    "pages": PAGE_PLAN.to_dict(orient="records"),
    "painting_ids": PAINTING_INDEX["painting_id"].tolist(),
    "models": (
        MODEL_DIMENSION[
            ["model_id", "display_name", "evaluation_status"]
        ]
        .to_dict(orient="records")
    ),
    "experiment_ids": sorted(
        CASE_INDEX["experiment_id"].dropna().unique().tolist()
    ),
    "categories": sorted(
        CASE_INDEX["category"].dropna().unique().tolist()
    ),
    "styles_or_periods": sorted(
        value
        for value in CASE_INDEX[
            "style_or_period"
        ].dropna().unique().tolist()
        if str(value).strip()
    ),
    "degradation_families": sorted(
        value
        for value in CASE_INDEX[
            "degradation_family"
        ].dropna().unique().tolist()
        if str(value).strip()
    ),
    "severities": sorted(
        value
        for value in CASE_INDEX[
            "severity"
        ].dropna().unique().tolist()
        if str(value).strip()
    ),
    "recommendation_categories": sorted(
        CASE_INDEX[
            "recommendation_category"
        ].dropna().unique().tolist()
    ),
    "uncertainty_applicability": sorted(
        CASE_INDEX[
            "uncertainty_applicability"
        ].dropna().unique().tolist()
    ),
    "metric_families": sorted(
        REGION_POLICY["metric_family"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),
    "region_ids": sorted(
        REGION_POLICY["region_id"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),
    "failure_categories": (
        FAILURE_TAXONOMY[
            ["category_id", "display_name", "is_proxy"]
        ]
        .sort_values("category_id")
        .to_dict(orient="records")
    ),
    "status": "ok",
}

display(MODEL_DIMENSION)
display(EXPERIMENT_DIMENSION)

print("Shared dashboard vocabularies constructed.")
print("Filter groups:", len(FILTER_OPTIONS) - 2)
print("Painting IDs:", len(FILTER_OPTIONS["painting_ids"]))
print("Metric families:", len(FILTER_OPTIONS["metric_families"]))
print("Regions:", len(FILTER_OPTIONS["region_ids"]))
print(
    "Failure categories:",
    len(FILTER_OPTIONS["failure_categories"]),
)

,model_id,display_name,evaluation_status,model_family,methodological_role,deterministic,stochastic,prompt_dependent,evaluated_painting_count,evaluated_case_count,evaluated_candidate_count,strengths_json,weaknesses_json,known_limitations_json,status
0,lama,LaMa,fully_evaluated,learned_fourier_convolution_inpainting,learned_deterministic_inpainting_baseline,True,False,False,50,410,410,"[""broad learned context"",""practical large-mask...","[""general-scene rather than painting-specific ...","[""Runtime and memory observations describe one...",ok
1,opencv_telea,OpenCV Telea,fully_evaluated,classical_fast_marching_inpainting,classical_deterministic_baseline,True,False,False,50,410,410,"[""fast and deterministic"",""transparent classic...","[""no semantic understanding"",""limited large-ma...","[""Runtime and memory observations describe one...",ok
2,sdxl_inpainting,SDXL Inpainting,partial_evaluation,prompt_conditioned_sdxl_latent_diffusion_inpai...,bounded_higher_capacity_diffusion_candidate,False,True,True,5,10,10,"[""higher-capacity diffusion lineage"",""technica...","[""only ten purposively selected cases"",""very h...","[""Runtime and memory observations describe one...",ok
3,stable_diffusion_inpainting,Stable Diffusion Inpainting,fully_evaluated,prompt_conditioned_latent_diffusion_inpainting,prompt_conditioned_stochastic_inpainting_baseline,False,True,True,50,410,1330,"[""prompt-conditioned generative completion"",""s...","[""high hallucination and prompt-sensitivity ri...","[""Runtime and memory observations describe one...",ok


,experiment_id,painting_count,case_count,candidate_count,model_count
0,canonical_missing_region,50,250,1194,4
1,damage_size_sensitivity,5,35,210,3
2,mask_robustness,5,75,225,3
3,synthetic_degradation,5,50,156,4


Shared dashboard vocabularies constructed.
Filter groups: 14
Painting IDs: 50
Metric families: 13
Regions: 11
Failure categories: 14


In [10]:
EXPECTED_MODEL_COUNTS = {
    "lama": 410,
    "opencv_telea": 410,
    "sdxl_inpainting": 10,
    "stable_diffusion_inpainting": 955,
}

EXPECTED_EXPERIMENT_COUNTS = {
    "canonical_missing_region": 1194,
    "damage_size_sensitivity": 210,
    "mask_robustness": 225,
    "synthetic_degradation": 156,
}

OBSERVED_MODEL_COUNTS = (
    CASE_INDEX.groupby("model_id")
    .size()
    .astype(int)
    .to_dict()
)

OBSERVED_EXPERIMENT_COUNTS = (
    CASE_INDEX.groupby("experiment_id")
    .size()
    .astype(int)
    .to_dict()
)

DIRECT_CASE_PATH_COLUMNS = [
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
]

DIRECT_PATH_FAILURES = {
    column: int(
        (~CASE_INDEX[column].map(repository_file_exists)).sum()
    )
    for column in DIRECT_CASE_PATH_COLUMNS
}

REPORT_PATH_FAILURES = {
    "case_report_path": int(
        (
            CASE_INDEX.loc[
                CASE_INDEX["case_report_path"].ne(""),
                "case_report_path",
            ]
            .map(repository_file_exists)
            .eq(False)
        ).sum()
    ),
    "painting_report_path": int(
        (
            PAINTING_INDEX.loc[
                PAINTING_INDEX["painting_report_path"].ne(""),
                "painting_report_path",
            ]
            .map(repository_file_exists)
            .eq(False)
        ).sum()
    ),
}

JSON_LIST_COLUMNS = [
    "difference_paths_json",
    "uncertainty_paths_json",
    "seam_paths_json",
    "colour_paths_json",
    "texture_paths_json",
    "semantic_paths_json",
    "mask_boundary_paths_json",
    "triggered_flag_ids_json",
    "triggered_category_ids_json",
    "affected_regions_json",
    "recommended_actions_json",
    "metric_disagreement_ids_json",
    "report_selection_roles_json",
    "evidence_source_notebook_ids_json",
]

JSON_LISTS_VALID, JSON_LIST_ERRORS = all_json_lists_valid(
    CASE_INDEX,
    JSON_LIST_COLUMNS,
)

BATCH_2_CHECKS = (
    (
        "candidate_row_count",
        "Candidate index covers the approved population",
        1785,
        len(CASE_INDEX),
        len(CASE_INDEX) == 1785,
    ),
    (
        "candidate_id_uniqueness",
        "Every candidate ID appears exactly once",
        1785,
        CASE_INDEX["candidate_id"].nunique(),
        (
            CASE_INDEX["candidate_id"].nunique()
            == len(CASE_INDEX)
        ),
    ),
    (
        "case_population",
        "Candidate index covers 410 restoration cases",
        410,
        CASE_INDEX["case_id"].nunique(),
        CASE_INDEX["case_id"].nunique() == 410,
    ),
    (
        "painting_population",
        "Candidate index and painting index cover 50 paintings",
        50,
        {
            "candidate_index": CASE_INDEX[
                "painting_id"
            ].nunique(),
            "painting_index": len(PAINTING_INDEX),
        },
        (
            CASE_INDEX["painting_id"].nunique() == 50
            and len(PAINTING_INDEX) == 50
        ),
    ),
    (
        "painting_id_uniqueness",
        "Each painting appears once in the painting index",
        50,
        PAINTING_INDEX["painting_id"].nunique(),
        (
            PAINTING_INDEX["painting_id"].nunique()
            == len(PAINTING_INDEX)
        ),
    ),
    (
        "model_candidate_counts",
        "Model-specific candidate counts match the contract",
        EXPECTED_MODEL_COUNTS,
        OBSERVED_MODEL_COUNTS,
        OBSERVED_MODEL_COUNTS == EXPECTED_MODEL_COUNTS,
    ),
    (
        "experiment_candidate_counts",
        "Experiment-specific candidate counts match the contract",
        EXPECTED_EXPERIMENT_COUNTS,
        OBSERVED_EXPERIMENT_COUNTS,
        (
            OBSERVED_EXPERIMENT_COUNTS
            == EXPECTED_EXPERIMENT_COUNTS
        ),
    ),
    (
        "direct_case_paths",
        "Every direct candidate image path resolves",
        {
            column: 0
            for column in DIRECT_CASE_PATH_COLUMNS
        },
        DIRECT_PATH_FAILURES,
        not any(DIRECT_PATH_FAILURES.values()),
    ),
    (
        "case_report_coverage",
        "All 30 selected cases expose detailed reports",
        30,
        CASE_INDEX.loc[
            CASE_INDEX["case_report_path"].ne(""),
            "case_id",
        ].nunique(),
        (
            CASE_INDEX.loc[
                CASE_INDEX["case_report_path"].ne(""),
                "case_id",
            ].nunique()
            == 30
        ),
    ),
    (
        "painting_report_coverage",
        "All 50 paintings expose painting reports",
        50,
        PAINTING_INDEX["painting_report_path"].ne("").sum(),
        (
            PAINTING_INDEX[
                "painting_report_path"
            ].ne("").sum()
            == 50
        ),
    ),
    (
        "report_paths",
        "Every joined report path resolves",
        {
            "case_report_path": 0,
            "painting_report_path": 0,
        },
        REPORT_PATH_FAILURES,
        not any(REPORT_PATH_FAILURES.values()),
    ),
    (
        "self_contained_painting_reports",
        "Every painting report is self-contained",
        50,
        int(
            PAINTING_INDEX[
                "painting_report_self_contained"
            ].sum()
        ),
        PAINTING_INDEX[
            "painting_report_self_contained"
        ].all(),
    ),
    (
        "json_list_contracts",
        "Candidate evidence fields contain valid JSON lists",
        [],
        JSON_LIST_ERRORS,
        JSON_LISTS_VALID,
    ),
    (
        "candidate_status",
        "All candidate-index rows retain successful status",
        ["ok"],
        sorted(CASE_INDEX["status"].unique().tolist()),
        CASE_INDEX["status"].eq("ok").all(),
    ),
    (
        "filter_page_order",
        "Filter metadata retains the approved page order",
        EXPECTED_PAGE_IDS,
        FILTER_OPTIONS["page_ids"],
        FILTER_OPTIONS["page_ids"] == EXPECTED_PAGE_IDS,
    ),
    (
        "no_canonical_files_written",
        "Batch 2 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_2_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_2_population_normalization",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Normalized dashboard population validation failed."
        ),
    )

VALIDATION.raise_for_blocking()

MODEL_PROFILE = (
    CASE_INDEX
    .groupby(
        [
            "model_id",
            "recommendation_category",
            "uncertainty_applicability",
        ],
        dropna=False,
    )
    .agg(
        candidates=("candidate_id", "size"),
        cases=("case_id", "nunique"),
        paintings=("painting_id", "nunique"),
        manual_review_candidates=(
            "manual_review_required",
            "sum",
        ),
    )
    .reset_index()
)

EXPERIMENT_PROFILE = (
    CASE_INDEX
    .groupby(
        ["experiment_id", "model_id"],
        dropna=False,
    )
    .agg(
        candidates=("candidate_id", "size"),
        cases=("case_id", "nunique"),
        paintings=("painting_id", "nunique"),
    )
    .reset_index()
)

BATCH_2_VALIDATION = (
    VALIDATION.to_dataframe()
    .query(
        "validation_stage == "
        "'batch_2_population_normalization'"
    )
    .reset_index(drop=True)
)

display(MODEL_PROFILE)
display(EXPERIMENT_PROFILE)
display(PAINTING_INDEX.head(10))
display(
    BATCH_2_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 2 population normalization passed.")
print("Candidate-index rows:", len(CASE_INDEX))
print("Painting-index rows:", len(PAINTING_INDEX))
print("Case-report case coverage:", 30)
print("Painting-report coverage:", 50)
print("Direct candidate path failures:", DIRECT_PATH_FAILURES)
print("Report path failures:", REPORT_PATH_FAILURES)
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,model_id,recommendation_category,uncertainty_applicability,candidates,cases,paintings,manual_review_candidates
0,lama,do_not_rely_automatically,not_applicable_deterministic_method,68,68,50,68
1,lama,specialist_review_required,not_applicable_deterministic_method,295,295,50,295
2,lama,suitable_for_preliminary_inspection,not_applicable_deterministic_method,47,47,5,0
3,opencv_telea,do_not_rely_automatically,not_applicable_deterministic_method,98,98,50,98
4,opencv_telea,specialist_review_required,not_applicable_deterministic_method,278,278,50,278
5,opencv_telea,suitable_for_preliminary_inspection,not_applicable_deterministic_method,34,34,5,0
6,sdxl_inpainting,do_not_rely_automatically,not_applicable_insufficient_seed_coverage,1,1,1,1
7,sdxl_inpainting,specialist_review_required,not_applicable_insufficient_seed_coverage,9,9,5,9
8,stable_diffusion_inpainting,do_not_rely_automatically,applicable_complete_repeated_seed_group,66,16,13,66
9,stable_diffusion_inpainting,do_not_rely_automatically,not_available_single_candidate_scope,64,64,50,64


,experiment_id,model_id,candidates,cases,paintings
0,canonical_missing_region,lama,250,250,50
1,canonical_missing_region,opencv_telea,250,250,50
2,canonical_missing_region,sdxl_inpainting,4,4,4
3,canonical_missing_region,stable_diffusion_inpainting,690,250,50
4,damage_size_sensitivity,lama,35,35,5
5,damage_size_sensitivity,opencv_telea,35,35,5
6,damage_size_sensitivity,stable_diffusion_inpainting,140,35,5
7,mask_robustness,lama,75,75,5
8,mask_robustness,opencv_telea,75,75,5
9,mask_robustness,stable_diffusion_inpainting,75,75,5


,painting_index_id,painting_id,dataset_sort_index,title,artist,date_or_period,style_or_period,category,medium,source,...,has_uncertainty,has_selected_case_report,painting_report_path,painting_report_sha256,painting_report_self_contained,source_notebook_ids_json,source_paths_json,schema_version,status,issue
0,dashboard_painting_476add61ae307328cfb24a7b,p001,1,Juan de Pareja,Diego Velázquez,1650,Baroque,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,True,outputs/32_case_and_painting_report_generation...,91e4dfe7745666237f00e4207617f6dec5811489bfa25d...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
1,dashboard_painting_fca5855dd15c06fe63155be3,p002,2,Madame X (Madame Pierre Gautreau),John Singer Sargent,1883-84,19th century portraiture,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,True,outputs/32_case_and_painting_report_generation...,e0b702270eef06dec09ff7fecfa53be446f24cb4c4cb74...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
2,dashboard_painting_77092275327207a576bf6e8c,p003,3,Marie Joséphine Charlotte du Val d'Ognes,Marie-Denise Villers,1801,Neoclassical / early 19th century,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,False,outputs/32_case_and_painting_report_generation...,c96299d5d6495febeca2de6719119010adcb90fcdb0f3e...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
3,dashboard_painting_76139481155891aa1d44c69e,p004,4,Madame Georges Charpentier and Her Children,Pierre-Auguste Renoir,1878,Impressionism,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,False,outputs/32_case_and_painting_report_generation...,57f79bd7caca921c506049e2a9959fe49f465a362786b2...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
4,dashboard_painting_dc663a6ccdeaccfe9b5413d3,p005,5,Self-Portrait with Two Pupils,Adélaïde Labille-Guiard,1785,18th century portraiture,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,True,outputs/32_case_and_painting_report_generation...,47595d9cf26157121e714de5807e3027f10ad742e89052...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
5,dashboard_painting_819e4f8e1d8ef0eb2645fc8f,p006,6,Boy with a Sword,Édouard Manet,1861,Realism / early modern French painting,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,True,outputs/32_case_and_painting_report_generation...,714464c16197dc9775f54c83bc759ea4fb02b972cdace4...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
6,dashboard_painting_8d9a2795e7e1ecf1ddffeeeb,p007,7,Princesse de Broglie,Jean-Auguste-Dominique Ingres,1851-53,Neoclassical,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,False,outputs/32_case_and_painting_report_generation...,299f206a8956ca1e38539a4070265a09ed9342a9d0c905...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
7,dashboard_painting_4b5b00948503e357f652fc35,p008,8,Manuel Osorio Manrique de Zuñiga,Francisco de Goya y Lucientes,1787-88,Spanish portraiture,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,False,outputs/32_case_and_painting_report_generation...,fec44c8247e1f4e1b355f78c32da0aaff67ec569b9e650...,True,"[""01"", ""29"", ""32""]","[""outputs/01_dataset_verification/data/artwork...",dashboard_painting_index.v1,ok,
8,dashboard_painting_93caf3d8bdbd38c76284ac6f,p009,9,Madame Roulin and Her Baby,Vincent van Gogh,1888,Post-Impressionism,portrait_figure,Oil on canvas,The Metropolitan Museum of Art,...,True,True,outputs/32_case_and_painting_report_generation...,3b66827187987ed5878ecb2f584fab3c6b0c143edc4

,check_id,severity,passed,observed
0,candidate_row_count,blocking,True,1785
1,candidate_id_uniqueness,blocking,True,1785
2,case_population,blocking,True,410
3,painting_population,blocking,True,"{""candidate_index"": 50, ""painting_index"": 50}"
4,painting_id_uniqueness,blocking,True,50
5,model_candidate_counts,blocking,True,"{""lama"": 410, ""opencv_telea"": 410, ""sdxl_inpai..."
6,experiment_candidate_counts,blocking,True,"{""canonical_missing_region"": 1194, ""damage_siz..."
7,direct_case_paths,blocking,True,"{""clean_path"": 0, ""damaged_path"": 0, ""mask_pat..."
8,case_report_coverage,blocking,True,30
9,painting_report_coverage,blocking,True,50


Batch 2 population normalization passed.
Candidate-index rows: 1785
Painting-index rows: 50
Case-report case coverage: 30
Painting-report coverage: 50
Direct candidate path failures: {'clean_path': 0, 'damaged_path': 0, 'mask_path': 0, 'restored_path': 0}
Report path failures: {'case_report_path': 0, 'painting_report_path': 0}
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 3 — Overview and study-design evidence

This batch constructs the dashboard's opening evidence layer from completed upstream results.

It creates:

- concise headline findings for the Overview page;
- the complete controlled-dataset and experiment design summary;
- explicit coverage and limitation statements;
- source references for every displayed claim.

The dashboard does not introduce a new score or reinterpret partial SDXL evidence as a full benchmark. All conclusions remain tied to validated Notebook 33 evidence and the original dataset-generation notebooks.

In [11]:
THESIS_TABLES = (
    INPUT_TABLES["thesis_tables_path"]
    .copy()
    .reset_index(drop=True)
)

ARTWORKS = (
    INPUT_TABLES["artworks_path"]
    .copy()
    .reset_index(drop=True)
)

CANONICAL_MASKS = (
    INPUT_TABLES["canonical_masks_path"]
    .copy()
    .reset_index(drop=True)
)

DAMAGE_SIZE_CASES = (
    INPUT_TABLES["damage_size_cases_path"]
    .copy()
    .reset_index(drop=True)
)

ROBUSTNESS_CASES = (
    INPUT_TABLES["mask_robustness_cases_path"]
    .copy()
    .reset_index(drop=True)
)

DEGRADATION_CASES = (
    INPUT_TABLES["degradation_cases_path"]
    .copy()
    .reset_index(drop=True)
)

CASE_REGISTRY = (
    INPUT_TABLES["case_registry_path"]
    .copy()
    .reset_index(drop=True)
)


def parse_values_json(value: object) -> dict:
    """Parse a Notebook 33 values_json object strictly."""

    if not isinstance(value, str) or not value.strip():
        raise ValueError("Expected a non-empty values_json string.")

    parsed = json.loads(value)

    if not isinstance(parsed, dict):
        raise TypeError("values_json must contain a JSON object.")

    return parsed


METRIC_DISAGREEMENT_SOURCE = (
    THESIS_TABLES.loc[
        THESIS_TABLES["table_id"].eq("t05_metric_disagreement"),
        [
            "row_key",
            "row_label",
            "values_json",
            "interpretation",
            "status",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

ANCHOR_WINNERS = pd.DataFrame(
    [
        {
            "anchor_id": str(row.row_key),
            "winner_model_id": parsed["winner_model_id"],
            "majority_vote_winner_model_id": (
                parsed["majority_vote_winner_model_id"]
            ),
            "agrees_with_majority_vote": bool(
                parsed["agrees_with_majority_vote"]
            ),
            "loo_winner_stability_fraction": float(
                parsed["loo_winner_stability_fraction"]
            ),
        }
        for row in METRIC_DISAGREEMENT_SOURCE.itertuples(index=False)
        for parsed in [parse_values_json(row.values_json)]
    ]
)

WINNER_COUNTS = (
    ANCHOR_WINNERS["winner_model_id"]
    .value_counts()
    .sort_index()
    .to_dict()
)

CATEGORY_COUNTS = (
    ARTWORKS["category"]
    .astype(str)
    .value_counts()
    .sort_index()
)

STYLE_STATUS_COUNTS = (
    ARTWORKS["prompt_metadata_status"]
    .astype(str)
    .value_counts()
    .to_dict()
)

EXPERIMENT_CASE_COUNTS = (
    CASE_REGISTRY.groupby("experiment_id")
    .size()
    .astype(int)
    .to_dict()
)

RECOMMENDATION_COUNTS = (
    CASE_INDEX["recommendation_category"]
    .astype(str)
    .value_counts()
    .to_dict()
)

MANUAL_REVIEW_COUNT = int(
    CASE_INDEX["manual_review_required"].sum()
)

MANUAL_REVIEW_FRACTION = (
    MANUAL_REVIEW_COUNT / len(CASE_INDEX)
)

TOTAL_UNCERTAINTY_GROUPS = (
    INPUT_TABLES[
        "canonical_uncertainty_path"
    ]["uncertainty_group_id"].nunique()
    + INPUT_TABLES[
        "damage_size_uncertainty_path"
    ]["uncertainty_group_id"].nunique()
)

print("Validated overview evidence extracted.")
print("Quality-anchor winners:", WINNER_COUNTS)
print("Dataset categories:", CATEGORY_COUNTS.to_dict())
print("Experiment cases:", EXPERIMENT_CASE_COUNTS)
print(
    "Manual-review candidates:",
    f"{MANUAL_REVIEW_COUNT}/{len(CASE_INDEX)}",
)
print("Uncertainty groups:", TOTAL_UNCERTAINTY_GROUPS)

Validated overview evidence extracted.
Quality-anchor winners: {'lama': 10, 'opencv_telea': 1}
Dataset categories: {'abstraction_surrealism': 10, 'architecture_structured': 10, 'high_texture_brushwork': 10, 'landscape_natural': 10, 'portrait_figure': 10}
Experiment cases: {'canonical_missing_region': 250, 'damage_size_sensitivity': 35, 'mask_robustness': 75, 'synthetic_degradation': 165}
Manual-review candidates: 1703/1785
Uncertainty groups: 165


In [12]:
def headline_row(
    display_order: int,
    finding_type: str,
    title: str,
    value: object,
    value_unit: str,
    conclusion: str,
    evidence_strength: str,
    tone: str,
    scope: str,
    denominator: object,
    source_notebook_ids: list[str],
    source_paths: list[str],
    limitation: str,
) -> dict:
    """Construct one exact-schema headline finding."""

    return {
        "finding_id": stable_id(
            "finding",
            "overview",
            display_order,
            title,
        ),
        "page_id": "overview",
        "display_order": display_order,
        "finding_type": finding_type,
        "title": title,
        "value": value,
        "value_unit": value_unit,
        "conclusion": conclusion,
        "evidence_strength": evidence_strength,
        "tone": tone,
        "scope": scope,
        "denominator": denominator,
        "source_notebook_ids_json": json_list(
            source_notebook_ids
        ),
        "source_paths_json": json_list(source_paths),
        "limitation": limitation,
        "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
        "status": "ok",
        "issue": "",
    }


HEADLINE_ROWS = [
    headline_row(
        display_order=1,
        finding_type="scope",
        title="Controlled painting collection",
        value=len(ARTWORKS),
        value_unit="paintings",
        conclusion=(
            "The evaluation covers a balanced 50-painting collection "
            "with ten paintings in each of five visual categories."
        ),
        evidence_strength="validated_scope",
        tone="info",
        scope="controlled_50 dataset",
        denominator=len(ARTWORKS),
        source_notebook_ids=["01"],
        source_paths=[
            SETTINGS["inputs"]["artworks_path"],
        ],
        limitation=(
            "The controlled collection does not establish "
            "real-world conservation generality."
        ),
    ),
    headline_row(
        display_order=2,
        finding_type="scope",
        title="Approved restoration evidence",
        value=len(CASE_INDEX),
        value_unit="candidates",
        conclusion=(
            "All 1,785 approved candidates remain available for "
            "filtering and case-level inspection."
        ),
        evidence_strength="validated_complete_catalog",
        tone="info",
        scope="approved candidate population",
        denominator=len(CASE_INDEX),
        source_notebook_ids=["29", "34"],
        source_paths=[
            SETTINGS["inputs"]["explanation_cases_path"],
        ],
        limitation=(
            "Candidate count is not an independent sample-size claim "
            "because repeated seeds and models share cases."
        ),
    ),
    headline_row(
        display_order=3,
        finding_type="conclusion",
        title="Strongest general benchmark result",
        value=int(WINNER_COUNTS.get("lama", 0)),
        value_unit="of 11 quality anchors",
        conclusion=(
            "LaMa ranks first on 10 of 11 quality anchors. Within "
            "this controlled benchmark, it is the strongest general "
            "restoration baseline across error, colour, perceptual, "
            "feature, seam, texture, and structural-affinity evidence."
        ),
        evidence_strength="validated_comparative",
        tone="success",
        scope="three fully evaluated models across 410 cases",
        denominator=len(ANCHOR_WINNERS),
        source_notebook_ids=["21", "33"],
        source_paths=[
            SETTINGS["inputs"]["metric_disagreement_path"],
            SETTINGS["inputs"]["thesis_tables_path"],
        ],
        limitation=(
            "Anchor wins are metric-specific benchmark results, "
            "not conservation approval or historical authenticity."
        ),
    ),
    headline_row(
        display_order=4,
        finding_type="conclusion",
        title="One metric changes the winner",
        value=int(
            (
                ANCHOR_WINNERS["winner_model_id"]
                == "opencv_telea"
            ).sum()
        ),
        value_unit="of 11 quality anchors",
        conclusion=(
            "OpenCV Telea narrowly leads crop SSIM, while LaMa leads "
            "the other ten anchors. SSIM alone would therefore give "
            "an incomplete account of restoration quality."
        ),
        evidence_strength="validated_metric_disagreement",
        tone="warning",
        scope="quality-anchor comparison",
        denominator=len(ANCHOR_WINNERS),
        source_notebook_ids=["21", "33"],
        source_paths=[
            SETTINGS["inputs"]["metric_disagreement_path"],
            SETTINGS["inputs"]["thesis_tables_path"],
        ],
        limitation=(
            "The result demonstrates metric disagreement; it does "
            "not make either model universally correct."
        ),
    ),
    headline_row(
        display_order=5,
        finding_type="coverage",
        title="Repeated-run uncertainty coverage",
        value=int(TOTAL_UNCERTAINTY_GROUPS),
        value_unit="uncertainty groups",
        conclusion=(
            "Repeated-seed evidence covers 130 canonical groups and "
            "35 damage-size groups for Stable Diffusion."
        ),
        evidence_strength="validated_supported_population",
        tone="info",
        scope="supported Stable Diffusion repeated-seed groups",
        denominator=int(TOTAL_UNCERTAINTY_GROUPS),
        source_notebook_ids=["18", "22"],
        source_paths=[
            SETTINGS["inputs"]["canonical_uncertainty_path"],
            SETTINGS["inputs"]["damage_size_uncertainty_path"],
        ],
        limitation=(
            "Seed variability is an empirical uncertainty proxy, "
            "not calibrated confidence."
        ),
    ),
    headline_row(
        display_order=6,
        finding_type="coverage",
        title="SDXL remains bounded evidence",
        value=int(
            (
                CASE_INDEX["model_id"]
                == "sdxl_inpainting"
            ).sum()
        ),
        value_unit="candidates",
        conclusion=(
            "SDXL contributes ten completed cases for qualitative "
            "and bounded comparison, not a fourth full benchmark."
        ),
        evidence_strength="validated_partial_coverage",
        tone="warning",
        scope="partial SDXL feasibility population",
        denominator=int(POPULATION["eligible_restoration_case_count"]),
        source_notebook_ids=["12", "30", "33"],
        source_paths=[
            SETTINGS["inputs"]["model_cards_path"],
            SETTINGS["inputs"]["thesis_tables_path"],
        ],
        limitation=(
            "SDXL results must not be generalized to all 410 "
            "restoration-eligible cases."
        ),
    ),
    headline_row(
        display_order=7,
        finding_type="trustworthiness",
        title="Human review remains central",
        value=round(100.0 * MANUAL_REVIEW_FRACTION, 1),
        value_unit="% of candidates",
        conclusion=(
            f"{MANUAL_REVIEW_COUNT:,} of {len(CASE_INDEX):,} candidates "
            "trigger conservative review guidance. Automated restoration "
            "should support inspection, not replace expert judgement."
        ),
        evidence_strength="validated_computational_flags",
        tone="danger",
        scope="approved candidate population",
        denominator=len(CASE_INDEX),
        source_notebook_ids=["27", "29", "33"],
        source_paths=[
            SETTINGS["inputs"]["trustworthiness_flags_path"],
            SETTINGS["inputs"]["explanation_cases_path"],
        ],
        limitation=(
            "Computational review flags are diagnostic rules, "
            "not expert annotations or conservation ground truth."
        ),
    ),
    headline_row(
        display_order=8,
        finding_type="principle",
        title="No universal restoration score",
        value=0,
        value_unit="approved combined scores",
        conclusion=(
            "The evaluation keeps metric families, regions, "
            "uncertainty, and failure evidence separate so that "
            "important disagreements remain visible."
        ),
        evidence_strength="approved_methodological_boundary",
        tone="info",
        scope="complete evaluation framework",
        denominator=0,
        source_notebook_ids=["08", "21", "28", "33"],
        source_paths=[
            SETTINGS["inputs"]["region_policy_path"],
            SETTINGS["inputs"]["ablation_results_path"],
            SETTINGS["inputs"]["thesis_tables_path"],
        ],
        limitation=(
            "The dashboard supports evidence review and does not "
            "produce conservation approval."
        ),
    ),
]

HEADLINE_FINDINGS = validate_output_frame(
    pd.DataFrame(HEADLINE_ROWS),
    "headline_findings",
)

display(
    HEADLINE_FINDINGS[
        [
            "display_order",
            "title",
            "value",
            "value_unit",
            "conclusion",
            "tone",
        ]
    ]
)

print("Headline findings constructed:", len(HEADLINE_FINDINGS))

,display_order,title,value,value_unit,conclusion,tone
0,1,Controlled painting collection,50.0,paintings,The evaluation covers a balanced 50-painting c...,info
1,2,Approved restoration evidence,1785.0,candidates,"All 1,785 approved candidates remain available...",info
2,3,Strongest general benchmark result,10.0,of 11 quality anchors,LaMa ranks first on 10 of 11 quality anchors. ...,success
3,4,One metric changes the winner,1.0,of 11 quality anchors,"OpenCV Telea narrowly leads crop SSIM, while L...",warning
4,5,Repeated-run uncertainty coverage,165.0,uncertainty groups,Repeated-seed evidence covers 130 canonical gr...,info
5,6,SDXL remains bounded evidence,10.0,candidates,SDXL contributes ten completed cases for quali...,warning
6,7,Human review remains central,95.4,% of candidates,"1,703 of 1,785 candidates trigger conservative...",danger
7,8,No universal restoration score,0.0,approved combined scores,"The evaluation keeps metric families, regions,...",info


Headline findings constructed: 8


In [13]:
STUDY_DESIGN_ROWS: list[dict] = []


def add_study_design_row(
    section_id: str,
    record_type: str,
    display_name: str,
    value: object,
    value_unit: str,
    scope: str,
    denominator: object,
    source_notebook_ids: list[str],
    source_paths: list[str],
    interpretation: str,
    *,
    experiment_id: str = "",
    group_field: str = "",
    group_value: str = "",
) -> None:
    """Append one exact-schema study-design record."""

    display_order = len(STUDY_DESIGN_ROWS) + 1

    STUDY_DESIGN_ROWS.append(
        {
            "design_row_id": stable_id(
                "design",
                section_id,
                record_type,
                display_name,
                experiment_id,
                group_field,
                group_value,
            ),
            "page_id": "study_design",
            "section_id": section_id,
            "display_order": display_order,
            "record_type": record_type,
            "experiment_id": experiment_id,
            "display_name": display_name,
            "group_field": group_field,
            "group_value": group_value,
            "value": value,
            "value_unit": value_unit,
            "scope": scope,
            "denominator": denominator,
            "source_notebook_ids_json": json_list(
                source_notebook_ids
            ),
            "source_paths_json": json_list(source_paths),
            "interpretation": interpretation,
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )


ARTWORKS_PATH = SETTINGS["inputs"]["artworks_path"]
MASKS_PATH = SETTINGS["inputs"]["canonical_masks_path"]
SIZE_PATH = SETTINGS["inputs"]["damage_size_cases_path"]
ROBUSTNESS_PATH = SETTINGS["inputs"]["mask_robustness_cases_path"]
DEGRADATION_PATH = SETTINGS["inputs"]["degradation_cases_path"]
REGISTRY_PATH = SETTINGS["inputs"]["case_registry_path"]
EXPLANATION_PATH = SETTINGS["inputs"]["explanation_cases_path"]

add_study_design_row(
    "dataset_scope",
    "population",
    "Controlled paintings",
    len(ARTWORKS),
    "paintings",
    "controlled_50 dataset",
    len(ARTWORKS),
    ["01"],
    [ARTWORKS_PATH],
    "The study uses a fixed, versioned collection of 50 paintings.",
)

add_study_design_row(
    "dataset_scope",
    "group_count",
    "Visual categories",
    ARTWORKS["category"].nunique(),
    "categories",
    "controlled_50 dataset",
    len(ARTWORKS),
    ["01"],
    [ARTWORKS_PATH],
    "Five broad visual categories support balanced descriptive comparisons.",
)

add_study_design_row(
    "dataset_scope",
    "metadata",
    "Mean metadata completeness",
    round(float(ARTWORKS["metadata_completeness_pct"].mean()), 1),
    "%",
    "controlled_50 dataset",
    len(ARTWORKS),
    ["01"],
    [ARTWORKS_PATH],
    "Metadata is useful for context but remains incomplete for some paintings.",
)

add_study_design_row(
    "dataset_scope",
    "metadata",
    "Documented style or period",
    int(STYLE_STATUS_COUNTS.get("complete", 0)),
    "paintings",
    "controlled_50 dataset",
    len(ARTWORKS),
    ["01", "33"],
    [
        ARTWORKS_PATH,
        SETTINGS["inputs"]["thesis_tables_path"],
    ],
    "Only 18 paintings have complete prompt-relevant style or period metadata.",
)

add_study_design_row(
    "dataset_scope",
    "metadata",
    "Incomplete style or period",
    int(STYLE_STATUS_COUNTS.get("partial", 0)),
    "paintings",
    "controlled_50 dataset",
    len(ARTWORKS),
    ["01", "33"],
    [
        ARTWORKS_PATH,
        SETTINGS["inputs"]["thesis_tables_path"],
    ],
    "Style-level conclusions must remain limited because 32 paintings lack complete documentation.",
)

for category, count in CATEGORY_COUNTS.items():
    add_study_design_row(
        "dataset_categories",
        "category_population",
        category.replace("_", " ").title(),
        int(count),
        "paintings",
        "controlled_50 dataset",
        len(ARTWORKS),
        ["01"],
        [ARTWORKS_PATH],
        "Each category contributes equally to the controlled collection.",
        group_field="category",
        group_value=category,
    )

add_study_design_row(
    "experiment_registry",
    "population",
    "Registered experimental cases",
    len(CASE_REGISTRY),
    "cases",
    "unified experiment registry",
    len(CASE_REGISTRY),
    ["08"],
    [REGISTRY_PATH],
    "The registry unifies canonical, damage-size, robustness, and synthetic-degradation cases.",
)

add_study_design_row(
    "canonical_experiment",
    "case_population",
    "Canonical cases",
    len(CANONICAL_MASKS),
    "cases",
    "canonical missing-region experiment",
    len(CANONICAL_MASKS),
    ["03", "04"],
    [MASKS_PATH],
    "Canonical coverage includes four damage masks plus a zero-damage control.",
    experiment_id="canonical_missing_region",
)

ZERO_CONTROL_COUNT = int(
    CANONICAL_MASKS["mask_type"].eq("zero_control").sum()
)

add_study_design_row(
    "canonical_experiment",
    "control_population",
    "Zero-damage controls",
    ZERO_CONTROL_COUNT,
    "cases",
    "canonical missing-region experiment",
    len(CANONICAL_MASKS),
    ["03", "04"],
    [MASKS_PATH],
    "One zero-damage control per painting checks identity preservation.",
    experiment_id="canonical_missing_region",
    group_field="mask_type",
    group_value="zero_control",
)

add_study_design_row(
    "canonical_experiment",
    "damaged_population",
    "Nonzero canonical cases",
    len(CANONICAL_MASKS) - ZERO_CONTROL_COUNT,
    "cases",
    "canonical missing-region experiment",
    len(CANONICAL_MASKS),
    ["03", "04"],
    [MASKS_PATH],
    "The 200 damaged cases provide the primary controlled restoration comparison.",
    experiment_id="canonical_missing_region",
    group_field="damage_state",
    group_value="nonzero",
)

add_study_design_row(
    "canonical_experiment",
    "group_count",
    "Canonical mask types",
    CANONICAL_MASKS["mask_type"].nunique(),
    "mask types",
    "canonical missing-region experiment",
    len(CANONICAL_MASKS),
    ["03"],
    [MASKS_PATH],
    "Equal mask-type coverage prevents one canonical damage geometry from dominating.",
    experiment_id="canonical_missing_region",
)

add_study_design_row(
    "damage_size_experiment",
    "case_population",
    "Damage-size cases",
    len(DAMAGE_SIZE_CASES),
    "cases",
    "focused damage-size extension",
    len(DAMAGE_SIZE_CASES),
    ["05"],
    [SIZE_PATH],
    "Seven nested damage levels are evaluated on five representative paintings.",
    experiment_id="damage_size_sensitivity",
)

add_study_design_row(
    "damage_size_experiment",
    "painting_population",
    "Damage-size paintings",
    DAMAGE_SIZE_CASES["painting_id"].nunique(),
    "paintings",
    "focused damage-size extension",
    len(ARTWORKS),
    ["05"],
    [SIZE_PATH],
    "The extension uses one painting from each broad category.",
    experiment_id="damage_size_sensitivity",
)

add_study_design_row(
    "damage_size_experiment",
    "level_count",
    "Damage-size levels",
    DAMAGE_SIZE_CASES["level_id"].nunique(),
    "levels",
    "2% to 20% target damage",
    DAMAGE_SIZE_CASES["level_id"].nunique(),
    ["05"],
    [SIZE_PATH],
    "Nested levels support within-painting sensitivity analysis.",
    experiment_id="damage_size_sensitivity",
)

add_study_design_row(
    "mask_robustness_experiment",
    "case_population",
    "Mask-robustness cases",
    len(ROBUSTNESS_CASES),
    "cases",
    "focused mask-geometry extension",
    len(ROBUSTNESS_CASES),
    ["06"],
    [ROBUSTNESS_PATH],
    "Controlled variants test sensitivity to mask geometry.",
    experiment_id="mask_robustness",
)

add_study_design_row(
    "mask_robustness_experiment",
    "painting_population",
    "Mask-robustness paintings",
    ROBUSTNESS_CASES["painting_id"].nunique(),
    "paintings",
    "focused mask-geometry extension",
    len(ARTWORKS),
    ["06"],
    [ROBUSTNESS_PATH],
    "The focused extension cannot separate painting effects from category effects.",
    experiment_id="mask_robustness",
)

add_study_design_row(
    "mask_robustness_experiment",
    "group_count",
    "Robustness groups",
    ROBUSTNESS_CASES["robustness_group_id"].nunique(),
    "groups",
    "focused mask-geometry extension",
    len(ROBUSTNESS_CASES),
    ["06"],
    [ROBUSTNESS_PATH],
    "Each group holds damage family and painting constant while mask geometry changes.",
    experiment_id="mask_robustness",
)

ROBUSTNESS_VARIANTS_PER_GROUP = int(
    ROBUSTNESS_CASES.groupby("robustness_group_id")["variant_id"]
    .nunique()
    .min()
)

add_study_design_row(
    "mask_robustness_experiment",
    "variant_count",
    "Variants per robustness group",
    ROBUSTNESS_VARIANTS_PER_GROUP,
    "variants",
    "each robustness group",
    ROBUSTNESS_VARIANTS_PER_GROUP,
    ["06"],
    [ROBUSTNESS_PATH],
    "Five variants provide a consistent within-group robustness comparison.",
    experiment_id="mask_robustness",
)

add_study_design_row(
    "degradation_experiment",
    "case_population",
    "Synthetic-degradation cases",
    len(DEGRADATION_CASES),
    "cases",
    "focused synthetic-degradation extension",
    len(DEGRADATION_CASES),
    ["07"],
    [DEGRADATION_PATH],
    "The extension broadens testing beyond missing-region masks.",
    experiment_id="synthetic_degradation",
)

add_study_design_row(
    "degradation_experiment",
    "painting_population",
    "Synthetic-degradation paintings",
    DEGRADATION_CASES["painting_id"].nunique(),
    "paintings",
    "focused synthetic-degradation extension",
    len(ARTWORKS),
    ["07"],
    [DEGRADATION_PATH],
    "The five-painting design supports controlled sensitivity evidence, not population generality.",
    experiment_id="synthetic_degradation",
)

add_study_design_row(
    "degradation_experiment",
    "family_count",
    "Synthetic-degradation families",
    DEGRADATION_CASES["degradation_family"].nunique(),
    "families",
    "focused synthetic-degradation extension",
    len(DEGRADATION_CASES),
    ["07"],
    [DEGRADATION_PATH],
    "Thirteen single and combined degradation families are represented.",
    experiment_id="synthetic_degradation",
)

for severity in ["mild", "moderate", "severe"]:
    severity_count = int(
        DEGRADATION_CASES["severity"].eq(severity).sum()
    )

    add_study_design_row(
        "degradation_experiment",
        "severity_population",
        f"{severity.title()} degradation cases",
        severity_count,
        "cases",
        "focused synthetic-degradation extension",
        len(DEGRADATION_CASES),
        ["07"],
        [DEGRADATION_PATH],
        "Severity counts describe generated test coverage rather than real conservation prevalence.",
        experiment_id="synthetic_degradation",
        group_field="severity",
        group_value=severity,
    )

add_study_design_row(
    "evaluation_population",
    "case_population",
    "Restoration-eligible cases",
    CASE_INDEX["case_id"].nunique(),
    "cases",
    "approved restoration evidence",
    len(CASE_REGISTRY),
    ["08", "29", "33"],
    [
        REGISTRY_PATH,
        EXPLANATION_PATH,
        SETTINGS["inputs"]["thesis_tables_path"],
    ],
    "410 of 525 registered cases enter the approved restoration evidence population.",
)

PRIMARY_CANDIDATE_COUNT = int(
    CASE_INDEX["population_role"].isin(
        [
            "primary_comparison",
            "primary_and_uncertainty",
        ]
    ).sum()
)

add_study_design_row(
    "evaluation_population",
    "candidate_population",
    "Primary three-model candidates",
    PRIMARY_CANDIDATE_COUNT,
    "candidates",
    "three fully evaluated models",
    3 * CASE_INDEX["case_id"].nunique(),
    ["09", "10", "11", "33"],
    [
        EXPLANATION_PATH,
        SETTINGS["inputs"]["thesis_tables_path"],
    ],
    "Each eligible case has primary evidence from Telea, LaMa, and Stable Diffusion.",
)

STABLE_DIFFUSION_COUNT = int(
    CASE_INDEX["model_id"]
    .eq("stable_diffusion_inpainting")
    .sum()
)

add_study_design_row(
    "evaluation_population",
    "candidate_population",
    "Stable Diffusion candidates",
    STABLE_DIFFUSION_COUNT,
    "candidates",
    "completed Stable Diffusion evidence",
    len(CASE_INDEX),
    ["11", "29"],
    [EXPLANATION_PATH],
    "Repeated seeds and the scratch-prompt ablation increase the diffusion candidate count.",
)

SDXL_COUNT = int(
    CASE_INDEX["model_id"].eq("sdxl_inpainting").sum()
)

add_study_design_row(
    "evaluation_population",
    "candidate_population",
    "Bounded SDXL candidates",
    SDXL_COUNT,
    "candidates",
    "partial SDXL feasibility evidence",
    CASE_INDEX["case_id"].nunique(),
    ["12", "29", "33"],
    [
        EXPLANATION_PATH,
        SETTINGS["inputs"]["thesis_tables_path"],
    ],
    "Ten SDXL candidates are retained as partial evidence and are not treated as full coverage.",
)

add_study_design_row(
    "evaluation_population",
    "candidate_population",
    "All approved candidates",
    len(CASE_INDEX),
    "candidates",
    "complete dashboard case index",
    len(CASE_INDEX),
    ["29", "34"],
    [EXPLANATION_PATH],
    "The dashboard keeps every approved candidate accessible even when reports show selected examples.",
)

add_study_design_row(
    "uncertainty_population",
    "uncertainty_group_population",
    "Canonical uncertainty groups",
    INPUT_TABLES[
        "canonical_uncertainty_path"
    ]["uncertainty_group_id"].nunique(),
    "groups",
    "supported canonical repeated-seed evidence",
    int(POPULATION["canonical_uncertainty_group_count"]),
    ["18"],
    [SETTINGS["inputs"]["canonical_uncertainty_path"]],
    "Canonical uncertainty is available only for prompt-specific Stable Diffusion groups with complete seed coverage.",
)

add_study_design_row(
    "uncertainty_population",
    "uncertainty_group_population",
    "Damage-size uncertainty groups",
    INPUT_TABLES[
        "damage_size_uncertainty_path"
    ]["uncertainty_group_id"].nunique(),
    "groups",
    "supported damage-size repeated-seed evidence",
    int(POPULATION["damage_size_uncertainty_group_count"]),
    ["22"],
    [SETTINGS["inputs"]["damage_size_uncertainty_path"]],
    "The extension supplies the damage-size uncertainty evidence absent from the frozen Notebook 18 population.",
)

STUDY_DESIGN = validate_output_frame(
    pd.DataFrame(STUDY_DESIGN_ROWS),
    "study_design",
)

display(
    STUDY_DESIGN[
        [
            "section_id",
            "display_name",
            "value",
            "value_unit",
            "interpretation",
        ]
    ]
)

print("Study-design rows constructed:", len(STUDY_DESIGN))

,section_id,display_name,value,value_unit,interpretation
0,dataset_scope,Controlled paintings,50.0,paintings,"The study uses a fixed, versioned collection o..."
1,dataset_scope,Visual categories,5.0,categories,Five broad visual categories support balanced ...
2,dataset_scope,Mean metadata completeness,76.0,%,Metadata is useful for context but remains inc...
3,dataset_scope,Documented style or period,18.0,paintings,Only 18 paintings have complete prompt-relevan...
4,dataset_scope,Incomplete style or period,32.0,paintings,Style-level conclusions must remain limited be...
5,dataset_categories,Abstraction Surrealism,10.0,paintings,Each category contributes equally to the contr...
6,dataset_categories,Architecture Structured,10.0,paintings,Each category contributes equally to the contr...
7,dataset_categories,High Texture Brushwork,10.0,paintings,Each category contributes equally to the contr...
8,dataset_categories,Landscape Natural,10.0,paintings,Each category contributes equally to the contr...
9,dataset_categories,Portrait Figure,10.0,paintings,Each category contributes equally to the contr...


Study-design rows constructed: 35


In [14]:
def all_source_paths_are_relative(series: pd.Series) -> bool:
    """Check every JSON-encoded source path in a series."""

    return all(
        is_repo_relative(path)
        for value in series
        for path in parse_json_list(value)
    )


EXPECTED_EXPERIMENT_CASE_COUNTS = {
    "canonical_missing_region": 250,
    "damage_size_sensitivity": 35,
    "mask_robustness": 75,
    "synthetic_degradation": 165,
}

EXPECTED_WINNER_COUNTS = {
    "lama": 10,
    "opencv_telea": 1,
}

BATCH_3_CHECKS = (
    (
        "headline_row_count",
        "Eight concise overview findings were constructed",
        8,
        len(HEADLINE_FINDINGS),
        len(HEADLINE_FINDINGS) == 8,
    ),
    (
        "headline_identifier_uniqueness",
        "Headline finding identifiers are unique",
        8,
        HEADLINE_FINDINGS["finding_id"].nunique(),
        (
            HEADLINE_FINDINGS["finding_id"].nunique()
            == len(HEADLINE_FINDINGS)
        ),
    ),
    (
        "quality_anchor_population",
        "All eleven quality anchors are represented",
        11,
        len(ANCHOR_WINNERS),
        len(ANCHOR_WINNERS) == 11,
    ),
    (
        "quality_anchor_winners",
        "Validated anchor-winner counts are preserved",
        EXPECTED_WINNER_COUNTS,
        WINNER_COUNTS,
        WINNER_COUNTS == EXPECTED_WINNER_COUNTS,
    ),
    (
        "balanced_category_population",
        "Every dataset category contains ten paintings",
        [10],
        sorted(CATEGORY_COUNTS.unique().tolist()),
        sorted(CATEGORY_COUNTS.unique().tolist()) == [10],
    ),
    (
        "style_metadata_population",
        "Style-metadata coverage remains explicit",
        {"complete": 18, "partial": 32},
        STYLE_STATUS_COUNTS,
        STYLE_STATUS_COUNTS == {"complete": 18, "partial": 32},
    ),
    (
        "experiment_case_population",
        "All four experiment populations match the registry",
        EXPECTED_EXPERIMENT_CASE_COUNTS,
        EXPERIMENT_CASE_COUNTS,
        EXPERIMENT_CASE_COUNTS
        == EXPECTED_EXPERIMENT_CASE_COUNTS,
    ),
    (
        "study_design_row_count",
        "The complete study-design summary contains 35 rows",
        35,
        len(STUDY_DESIGN),
        len(STUDY_DESIGN) == 35,
    ),
    (
        "study_design_identifier_uniqueness",
        "Study-design identifiers are unique",
        len(STUDY_DESIGN),
        STUDY_DESIGN["design_row_id"].nunique(),
        (
            STUDY_DESIGN["design_row_id"].nunique()
            == len(STUDY_DESIGN)
        ),
    ),
    (
        "primary_candidate_population",
        "The primary comparison contains one candidate per model and case",
        1230,
        int(PRIMARY_CANDIDATE_COUNT),
        int(PRIMARY_CANDIDATE_COUNT) == 1230,
    ),
    (
        "candidate_population",
        "All approved candidates remain represented",
        int(POPULATION["approved_candidate_count"]),
        len(CASE_INDEX),
        len(CASE_INDEX)
        == int(POPULATION["approved_candidate_count"]),
    ),
    (
        "manual_review_population",
        "Review guidance covers every candidate",
        len(CASE_INDEX),
        sum(RECOMMENDATION_COUNTS.values()),
        sum(RECOMMENDATION_COUNTS.values())
        == len(CASE_INDEX),
    ),
    (
        "uncertainty_group_population",
        "Both supported uncertainty populations are retained",
        165,
        int(TOTAL_UNCERTAINTY_GROUPS),
        int(TOTAL_UNCERTAINTY_GROUPS) == 165,
    ),
    (
        "headline_sources_relative",
        "Headline source paths are repository-relative",
        True,
        all_source_paths_are_relative(
            HEADLINE_FINDINGS["source_paths_json"]
        ),
        all_source_paths_are_relative(
            HEADLINE_FINDINGS["source_paths_json"]
        ),
    ),
    (
        "study_design_sources_relative",
        "Study-design source paths are repository-relative",
        True,
        all_source_paths_are_relative(
            STUDY_DESIGN["source_paths_json"]
        ),
        all_source_paths_are_relative(
            STUDY_DESIGN["source_paths_json"]
        ),
    ),
    (
        "batch_3_status",
        "All Batch 3 records have valid status",
        {"ok"},
        set(HEADLINE_FINDINGS["status"])
        | set(STUDY_DESIGN["status"]),
        (
            set(HEADLINE_FINDINGS["status"])
            | set(STUDY_DESIGN["status"])
            == {"ok"}
        ),
    ),
    (
        "no_canonical_files_written",
        "Batch 3 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_3_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_3_overview_study_design",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Overview or study-design evidence differs from its validated source."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_3_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_3_overview_study_design"
        )
    ]
    .reset_index(drop=True)
)

OVERVIEW_PREVIEW = HEADLINE_FINDINGS[
    [
        "display_order",
        "title",
        "value",
        "value_unit",
        "conclusion",
    ]
].copy()

STUDY_DESIGN_SECTION_SUMMARY = (
    STUDY_DESIGN.groupby("section_id", sort=False)
    .agg(
        rows=("design_row_id", "size"),
        first_display_order=("display_order", "min"),
    )
    .reset_index()
    .sort_values("first_display_order")
    .reset_index(drop=True)
)

display(OVERVIEW_PREVIEW)
display(STUDY_DESIGN_SECTION_SUMMARY)
display(
    BATCH_3_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 3 completed.")
print("Headline findings:", len(HEADLINE_FINDINGS))
print("Study-design rows:", len(STUDY_DESIGN))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,display_order,title,value,value_unit,conclusion
0,1,Controlled painting collection,50.0,paintings,The evaluation covers a balanced 50-painting c...
1,2,Approved restoration evidence,1785.0,candidates,"All 1,785 approved candidates remain available..."
2,3,Strongest general benchmark result,10.0,of 11 quality anchors,LaMa ranks first on 10 of 11 quality anchors. ...
3,4,One metric changes the winner,1.0,of 11 quality anchors,"OpenCV Telea narrowly leads crop SSIM, while L..."
4,5,Repeated-run uncertainty coverage,165.0,uncertainty groups,Repeated-seed evidence covers 130 canonical gr...
5,6,SDXL remains bounded evidence,10.0,candidates,SDXL contributes ten completed cases for quali...
6,7,Human review remains central,95.4,% of candidates,"1,703 of 1,785 candidates trigger conservative..."
7,8,No universal restoration score,0.0,approved combined scores,"The evaluation keeps metric families, regions,..."


,section_id,rows,first_display_order
0,dataset_scope,5,1
1,dataset_categories,5,6
2,experiment_registry,1,11
3,canonical_experiment,4,12
4,damage_size_experiment,3,16
5,mask_robustness_experiment,4,19
6,degradation_experiment,6,23
7,evaluation_population,5,29
8,uncertainty_population,2,34


,check_id,severity,passed,observed
0,headline_row_count,blocking,True,8
1,headline_identifier_uniqueness,blocking,True,8
2,quality_anchor_population,blocking,True,11
3,quality_anchor_winners,blocking,True,"{""lama"": 10, ""opencv_telea"": 1}"
4,balanced_category_population,blocking,True,[10]
5,style_metadata_population,blocking,True,"{""complete"": 18, ""partial"": 32}"
6,experiment_case_population,blocking,True,"{""canonical_missing_region"": 250, ""damage_size..."
7,study_design_row_count,blocking,True,35
8,study_design_identifier_uniqueness,blocking,True,35
9,primary_candidate_population,blocking,True,1230


Batch 3 completed.
Headline findings: 8
Study-design rows: 35
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 4 — Metric framework and canonical region policy

This batch prepares the dashboard explanation of what each metric measures, where it can be computed, and why metric choice changes the interpretation.

It preserves:

- all 143 canonical metric-family and region-policy combinations;
- compatible and prohibited region combinations;
- summaries for all 13 policy metric families and 11 canonical regions;
- the 11 approved model-comparison quality anchors;
- metric direction, feature backbone, region, source, and interpretation limits.

A compatible metric-region combination means that the computation has an approved interpretation. It does not mean that the metric is sufficient by itself or that a high-performing candidate is conservation appropriate.

In [15]:
REGION_POLICY_SOURCE = (
    INPUT_TABLES["region_policy_path"]
    .copy()
    .sort_values(["metric_family", "region_id"])
    .reset_index(drop=True)
)

REGION_POLICY_SOURCE["compatible_bool"] = (
    REGION_POLICY_SOURCE["compatible"].map(as_bool)
)

METRIC_FAMILY_LABELS = {
    "classical_pixel": "Classical pixel error",
    "clip": "CLIP feature similarity",
    "colour": "Perceptual colour difference",
    "dinov2": "DINOv2 feature similarity",
    "lpips": "LPIPS perceptual distance",
    "seam": "Boundary and seam continuity",
    "semantic_patch": "Local semantic and structural consistency",
    "spatial_diagnostics": "Spatial error diagnostics",
    "ssim": "Structural similarity",
    "texture_descriptor": "Texture descriptors",
    "texture_map": "Local texture maps",
    "uncertainty_perceptual": "Perceptual seed uncertainty",
    "uncertainty_pixelwise": "Pixelwise seed uncertainty",
}

REGION_LABELS = {
    "boundary_ring": "Boundary ring",
    "content_region": "Content region",
    "degradation_support": "Degradation support",
    "full_image": "Full image",
    "inner_boundary_band": "Inner boundary band",
    "mask_bbox_crop": "Mask bounding-box crop",
    "masked_region": "Masked region",
    "outer_boundary_band": "Outer boundary band",
    "outside_boundary_ring": "Outside boundary ring",
    "outside_mask_content": "Outside-mask content",
    "patch_window": "Patch window",
}

REGION_INTERPRETATIONS = {
    "boundary_ring": (
        "Tests whether the repaired area joins its surroundings cleanly."
    ),
    "content_region": (
        "Measures the complete valid painting area while excluding padding."
    ),
    "degradation_support": (
        "Targets pixels affected by a synthetic degradation operator."
    ),
    "full_image": (
        "Provides broad context but can dilute small local restoration errors."
    ),
    "inner_boundary_band": (
        "Measures continuity immediately inside the repaired boundary."
    ),
    "mask_bbox_crop": (
        "Provides a contiguous crop around the damaged area for perceptual "
        "and feature models."
    ),
    "masked_region": (
        "Measures the pixels that the restoration method was asked to repair."
    ),
    "outer_boundary_band": (
        "Measures continuity immediately outside the repaired boundary."
    ),
    "outside_boundary_ring": (
        "Tests whether changes extend beyond the intended repair boundary."
    ),
    "outside_mask_content": (
        "Checks whether undamaged painting content was preserved."
    ),
    "patch_window": (
        "Supports local sliding-window evidence without reducing the "
        "painting to one global value."
    ),
}

EXPECTED_POLICY_FAMILIES = set(METRIC_FAMILY_LABELS)
EXPECTED_REGION_IDS = set(REGION_LABELS)

OBSERVED_POLICY_FAMILIES = set(
    REGION_POLICY_SOURCE["metric_family"].astype(str)
)

OBSERVED_REGION_IDS = set(
    REGION_POLICY_SOURCE["region_id"].astype(str)
)

if OBSERVED_POLICY_FAMILIES != EXPECTED_POLICY_FAMILIES:
    raise ValueError(
        "Unexpected metric-family population: "
        f"{sorted(OBSERVED_POLICY_FAMILIES)}"
    )

if OBSERVED_REGION_IDS != EXPECTED_REGION_IDS:
    raise ValueError(
        "Unexpected canonical-region population: "
        f"{sorted(OBSERVED_REGION_IDS)}"
    )

POLICY_ROWS = []

for row in REGION_POLICY_SOURCE.itertuples(index=False):
    compatible = bool(row.compatible_bool)

    POLICY_ROWS.append(
        {
            "metric_row_id": stable_id(
                "metric",
                "region_policy",
                row.policy_id,
            ),
            "page_id": "metric_framework",
            "section_id": "region_policy",
            "display_order": 0,
            "record_type": "region_policy",
            "policy_id": str(row.policy_id),
            "metric_family": str(row.metric_family),
            "metric_name": "",
            "feature_model_id": "",
            "region_id": str(row.region_id),
            "region_type": str(row.region_type),
            "compatible": compatible,
            "primary_role": str(row.primary_role),
            "comparison_direction": "",
            "ablation_policy_ids_json": str(
                row.ablation_policy_ids_json
            ),
            "value": int(compatible),
            "value_unit": "compatibility_flag",
            "scope": str(row.case_semantics),
            "source_notebook_ids_json": json_list(["08"]),
            "source_paths_json": json_list(
                [SETTINGS["inputs"]["region_policy_path"]]
            ),
            "interpretation": str(row.compatibility_reason),
            "limitation": (
                "Compatibility approves this metric-region meaning; "
                "it does not establish restoration quality."
                if compatible
                else
                "This combination is prohibited because its spatial "
                "support or interpretation is not defensible."
            ),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

POLICY_FRAME = pd.DataFrame(POLICY_ROWS)

print("Canonical region policies normalized:", len(POLICY_FRAME))
print(
    "Compatibility counts:",
    POLICY_FRAME["compatible"].value_counts().to_dict(),
)

Canonical region policies normalized: 143
Compatibility counts: {True: 86, False: 57}


In [16]:
SUMMARY_ROWS = []

for metric_family, group in REGION_POLICY_SOURCE.groupby(
    "metric_family",
    sort=True,
):
    compatible_count = int(group["compatible_bool"].sum())
    total_count = len(group)

    ablation_ids = sorted(
        {
            policy_id
            for value in group["ablation_policy_ids_json"]
            for policy_id in parse_json_list(value)
        }
    )

    SUMMARY_ROWS.append(
        {
            "metric_row_id": stable_id(
                "metric",
                "metric_family_summary",
                metric_family,
            ),
            "page_id": "metric_framework",
            "section_id": "metric_families",
            "display_order": 0,
            "record_type": "metric_family_summary",
            "policy_id": "",
            "metric_family": metric_family,
            "metric_name": "",
            "feature_model_id": "",
            "region_id": "",
            "region_type": "",
            "compatible": compatible_count > 0,
            "primary_role": "summary",
            "comparison_direction": "",
            "ablation_policy_ids_json": json_list(
                ablation_ids
            ),
            "value": compatible_count,
            "value_unit": "approved regions",
            "scope": "canonical region policy",
            "source_notebook_ids_json": json_list(["08"]),
            "source_paths_json": json_list(
                [SETTINGS["inputs"]["region_policy_path"]]
            ),
            "interpretation": (
                f"{METRIC_FAMILY_LABELS[metric_family]} has "
                f"{compatible_count} approved regions out of "
                f"{total_count}. Region choice therefore limits what "
                "the metric can validly claim."
            ),
            "limitation": (
                "Approved-region count describes policy coverage, "
                "not model performance or metric importance."
            ),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

for region_id, group in REGION_POLICY_SOURCE.groupby(
    "region_id",
    sort=True,
):
    compatible_count = int(group["compatible_bool"].sum())
    total_count = len(group)

    SUMMARY_ROWS.append(
        {
            "metric_row_id": stable_id(
                "metric",
                "region_summary",
                region_id,
            ),
            "page_id": "metric_framework",
            "section_id": "canonical_regions",
            "display_order": 0,
            "record_type": "region_summary",
            "policy_id": "",
            "metric_family": "",
            "metric_name": "",
            "feature_model_id": "",
            "region_id": region_id,
            "region_type": str(group["region_type"].iloc[0]),
            "compatible": compatible_count > 0,
            "primary_role": "summary",
            "comparison_direction": "",
            "ablation_policy_ids_json": json_list(
                sorted(
                    {
                        policy_id
                        for value in group[
                            "ablation_policy_ids_json"
                        ]
                        for policy_id in parse_json_list(value)
                    }
                )
            ),
            "value": compatible_count,
            "value_unit": "compatible metric families",
            "scope": "canonical region policy",
            "source_notebook_ids_json": json_list(["08"]),
            "source_paths_json": json_list(
                [SETTINGS["inputs"]["region_policy_path"]]
            ),
            "interpretation": REGION_INTERPRETATIONS[region_id],
            "limitation": (
                f"{compatible_count} of {total_count} metric families "
                "have an approved interpretation for this region. "
                "Unsupported metrics must not be forced onto it."
            ),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

SUMMARY_FRAME = pd.DataFrame(SUMMARY_ROWS)

print("Metric-family summaries:", len(METRIC_FAMILY_LABELS))
print("Canonical-region summaries:", len(REGION_LABELS))
print("Total framework summaries:", len(SUMMARY_FRAME))

Metric-family summaries: 13
Canonical-region summaries: 11
Total framework summaries: 24


In [17]:
MODEL_COMPARISON_SOURCE = (
    INPUT_TABLES["model_comparison_path"]
    .copy()
    .reset_index(drop=True)
)

CORE_ANCHORS = (
    MODEL_COMPARISON_SOURCE.loc[
        MODEL_COMPARISON_SOURCE["population_id"].eq(
            "core_three_model"
        )
        & MODEL_COMPARISON_SOURCE["analysis_scope"].eq("overall")
        & normalized_text(
            MODEL_COMPARISON_SOURCE["anchor_id"]
        ).ne(""),
        [
            "anchor_id",
            "evidence_family",
            "metric_family",
            "metric_name",
            "feature_model_id",
            "region_id",
            "summary_statistic",
            "value_unit",
            "comparison_direction",
        ],
    ]
    .drop_duplicates(subset=["anchor_id"])
    .sort_values("anchor_id")
    .reset_index(drop=True)
)

ANCHOR_POLICY_FAMILY = {
    "classical_masked_mae": "classical_pixel",
    "colour_masked_delta_e": "colour",
    "feature_clip_crop": "clip",
    "feature_dino_crop": "dinov2",
    "perceptual_crop_lpips": "lpips",
    "seam_boundary_gradient": "seam",
    "semantic_local_dino": "semantic_patch",
    "spatial_masked_error": "spatial_diagnostics",
    "structural_affinity_correlation": "semantic_patch",
    "structural_crop_ssim": "ssim",
    "texture_crop_p95": "texture_map",
}

ANCHOR_INTERPRETATIONS = {
    "classical_masked_mae": (
        "Measures average pixel error inside the repaired region. "
        "Lower values mean closer colour values at repaired pixels."
    ),
    "colour_masked_delta_e": (
        "Measures perceptual colour difference inside the repair. "
        "Lower values indicate better colour agreement."
    ),
    "feature_clip_crop": (
        "Measures broad CLIP feature continuity around the repair. "
        "Higher similarity means the crop remains more visually aligned."
    ),
    "feature_dino_crop": (
        "Measures DINOv2 feature continuity around the repair. "
        "Higher similarity indicates better retained visual structure."
    ),
    "perceptual_crop_lpips": (
        "Measures learned perceptual distance around the repair. "
        "Lower distance indicates a closer perceptual match."
    ),
    "seam_boundary_gradient": (
        "Measures gradient mismatch along the repair boundary. "
        "Lower values indicate a less visible seam."
    ),
    "semantic_local_dino": (
        "Measures local semantic continuity in the repaired area. "
        "Higher similarity indicates better local feature preservation."
    ),
    "spatial_masked_error": (
        "Summarizes spatial error inside the repaired pixels. "
        "Lower values indicate less concentrated repair error."
    ),
    "structural_affinity_correlation": (
        "Measures whether relationships between image regions are "
        "preserved. Higher correlation indicates stronger layout continuity."
    ),
    "structural_crop_ssim": (
        "Measures structural similarity around the repair. "
        "Higher values indicate closer local luminance and structure."
    ),
    "texture_crop_p95": (
        "Measures high-end local texture error around the repair. "
        "Lower values indicate fewer severe texture mismatches."
    ),
}

ANCHOR_LIMITATIONS = {
    "classical_masked_mae": (
        "Pixel agreement can reward smooth but visually unconvincing repairs."
    ),
    "colour_masked_delta_e": (
        "Colour agreement does not establish correct texture or structure."
    ),
    "feature_clip_crop": (
        "CLIP similarity is not historical authenticity or expert judgement."
    ),
    "feature_dino_crop": (
        "DINOv2 similarity is not proof that generated detail is correct."
    ),
    "perceptual_crop_lpips": (
        "LPIPS is a learned proxy and not a conservator rating."
    ),
    "seam_boundary_gradient": (
        "A clean boundary does not guarantee a correct repair interior."
    ),
    "semantic_local_dino": (
        "Feature continuity cannot verify iconography or historical detail."
    ),
    "spatial_masked_error": (
        "A scalar summary can hide where the largest errors occur."
    ),
    "structural_affinity_correlation": (
        "Layout preservation does not establish fine-detail correctness."
    ),
    "structural_crop_ssim": (
        "SSIM can disagree with colour, perceptual, texture, and seam evidence."
    ),
    "texture_crop_p95": (
        "Texture agreement does not establish semantic correctness."
    ),
}

POLICY_LOOKUP = (
    REGION_POLICY_SOURCE
    .set_index("policy_id", verify_integrity=True)
)

ANCHOR_ROWS = []

for row in CORE_ANCHORS.itertuples(index=False):
    anchor_id = str(row.anchor_id)
    policy_family = ANCHOR_POLICY_FAMILY[anchor_id]
    policy_id = f"{policy_family}::{row.region_id}"
    policy_record = POLICY_LOOKUP.loc[policy_id]

    if not bool(policy_record["compatible_bool"]):
        raise ValueError(
            f"Quality anchor uses a prohibited policy: {policy_id}"
        )

    ANCHOR_ROWS.append(
        {
            "metric_row_id": stable_id(
                "metric",
                "quality_anchor",
                anchor_id,
            ),
            "page_id": "metric_framework",
            "section_id": "quality_anchors",
            "display_order": 0,
            "record_type": "quality_anchor",
            "policy_id": policy_id,
            "metric_family": str(row.metric_family),
            "metric_name": str(row.metric_name),
            "feature_model_id": (
                ""
                if pd.isna(row.feature_model_id)
                else str(row.feature_model_id)
            ),
            "region_id": str(row.region_id),
            "region_type": str(policy_record["region_type"]),
            "compatible": True,
            "primary_role": "quality_ranking_anchor",
            "comparison_direction": str(
                row.comparison_direction
            ),
            "ablation_policy_ids_json": str(
                policy_record["ablation_policy_ids_json"]
            ),
            "value": 1,
            "value_unit": "selected quality anchor",
            "scope": "core_three_model overall comparison",
            "source_notebook_ids_json": json_list(
                ["08", "21", "33"]
            ),
            "source_paths_json": json_list(
                [
                    SETTINGS["inputs"]["region_policy_path"],
                    SETTINGS["inputs"]["model_comparison_path"],
                    SETTINGS["inputs"]["thesis_tables_path"],
                ]
            ),
            "interpretation": ANCHOR_INTERPRETATIONS[anchor_id],
            "limitation": ANCHOR_LIMITATIONS[anchor_id],
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

ANCHOR_FRAME = pd.DataFrame(ANCHOR_ROWS)

display(
    ANCHOR_FRAME[
        [
            "metric_name",
            "feature_model_id",
            "region_id",
            "comparison_direction",
            "interpretation",
            "limitation",
        ]
    ]
)

print("Approved quality anchors:", len(ANCHOR_FRAME))

,metric_name,feature_model_id,region_id,comparison_direction,interpretation,limitation
0,mae,,masked_region,lower_is_better,Measures average pixel error inside the repair...,Pixel agreement can reward smooth but visually...
1,delta_e_ciede2000_mean,,masked_region,lower_is_better,Measures perceptual colour difference inside t...,Colour agreement does not establish correct te...
2,clip_cosine_similarity,clip_vit_b32,mask_bbox_crop,higher_is_better,Measures broad CLIP feature continuity around ...,CLIP similarity is not historical authenticity...
3,dinov2_cosine_similarity,dinov2_vits14,mask_bbox_crop,higher_is_better,Measures DINOv2 feature continuity around the ...,DINOv2 similarity is not proof that generated ...
4,lpips,,mask_bbox_crop,lower_is_better,Measures learned perceptual distance around th...,LPIPS is a learned proxy and not a conservator...
5,boundary_gradient_mismatch,,boundary_ring,lower_is_better,Measures gradient mismatch along the repair bo...,A clean boundary does not guarantee a correct ...
6,local_patch_cosine_similarity,dinov2_vits14,mask_bbox_crop,higher_is_better,Measures local semantic continuity in the repa...,Feature continuity cannot verify iconography o...
7,restored_error_mean,,masked_region,lower_is_better,Summarizes spatial error inside the repaired p...,A scalar summary can hide where the largest er...
8,reference_affinity_map_correlation,dinov2_vits14,content_region,higher_is_better,Measures whether relationships between image r...,Layout preservation does not establish fine-de...
9,ssim,,mask_bbox_crop,higher_is_better,Measures structural similarity around the repa...,"SSIM can disagree with colour, perceptual, tex..."


Approved quality anchors: 11


In [18]:
METRIC_FRAMEWORK = pd.concat(
    [
        POLICY_FRAME,
        SUMMARY_FRAME,
        ANCHOR_FRAME,
    ],
    ignore_index=True,
)

SECTION_ORDER = {
    "quality_anchors": 1,
    "metric_families": 2,
    "canonical_regions": 3,
    "region_policy": 4,
}

METRIC_FRAMEWORK["_section_order"] = (
    METRIC_FRAMEWORK["section_id"].map(SECTION_ORDER)
)

METRIC_FRAMEWORK = (
    METRIC_FRAMEWORK
    .sort_values(
        [
            "_section_order",
            "metric_family",
            "region_id",
            "policy_id",
        ],
        kind="stable",
    )
    .drop(columns="_section_order")
    .reset_index(drop=True)
)

METRIC_FRAMEWORK["display_order"] = (
    METRIC_FRAMEWORK.index + 1
)

METRIC_FRAMEWORK = validate_output_frame(
    METRIC_FRAMEWORK,
    "metric_framework",
)

COMPATIBILITY_MATRIX = (
    POLICY_FRAME.pivot(
        index="metric_family",
        columns="region_id",
        values="compatible",
    )
    .reindex(
        index=sorted(METRIC_FAMILY_LABELS),
        columns=sorted(REGION_LABELS),
    )
)

COMPATIBILITY_DISPLAY = (
    COMPATIBILITY_MATRIX
    .replace({True: "approved", False: "prohibited"})
)

METRIC_FAMILY_SUMMARY_VIEW = (
    METRIC_FRAMEWORK.loc[
        METRIC_FRAMEWORK["record_type"].eq(
            "metric_family_summary"
        ),
        [
            "metric_family",
            "value",
            "value_unit",
            "interpretation",
        ],
    ]
    .sort_values("metric_family")
    .reset_index(drop=True)
)

display(METRIC_FAMILY_SUMMARY_VIEW)
display(COMPATIBILITY_DISPLAY)

print("Metric-framework rows:", len(METRIC_FRAMEWORK))
print(
    "Record types:",
    METRIC_FRAMEWORK["record_type"]
    .value_counts()
    .to_dict(),
)

,metric_family,value,value_unit,interpretation
0,classical_pixel,11,approved regions,Classical pixel error has 11 approved regions ...
1,clip,4,approved regions,CLIP feature similarity has 4 approved regions...
2,colour,11,approved regions,Perceptual colour difference has 11 approved r...
3,dinov2,4,approved regions,DINOv2 feature similarity has 4 approved regio...
4,lpips,4,approved regions,LPIPS perceptual distance has 4 approved regio...
5,seam,4,approved regions,Boundary and seam continuity has 4 approved re...
6,semantic_patch,4,approved regions,Local semantic and structural consistency has ...
7,spatial_diagnostics,11,approved regions,Spatial error diagnostics has 11 approved regi...
8,ssim,4,approved regions,Structural similarity has 4 approved regions o...
9,texture_descriptor,4,approved regions,Texture descriptors has 4 approved regions out...


region_id,boundary_ring,content_region,degradation_support,full_image,inner_boundary_band,mask_bbox_crop,masked_region,outer_boundary_band,outside_boundary_ring,outside_mask_content,patch_window
metric_family,,,,,,,,,,,
classical_pixel,approved,approved,approved,approved,approved,approved,approved,approved,approved,approved,approved
clip,prohibited,approved,prohibited,approved,prohibited,approved,prohibited,prohibited,prohibited,prohibited,approved
colour,approved,approved,approved,approved,approved,approved,approved,approved,approved,approved,approved
dinov2,prohibited,approved,prohibited,approved,prohibited,approved,prohibited,prohibited,prohibited,prohibited,approved
lpips,prohibited,approved,prohibited,approved,prohibited,approved,prohibited,prohibited,prohibited,prohibited,approved
seam,approved,prohibited,prohibited,prohibited,approved,prohibited,prohibited,approved,approved,prohibited,prohibited
semantic_patch,prohibited,approved,prohibited,approved,prohibited,approved,prohibited,prohibited,prohibited,prohibited,approved
spatial_diagnostics,approved,approved,approved,approved,approved,approved,approved,approved,approved,approved,approved
ssim,prohibited,approved,prohibited,approved,prohibited,approved,prohibited,prohibited,prohibited,prohibited,approved


Metric-framework rows: 178
Record types: {'region_policy': 143, 'metric_family_summary': 13, 'quality_anchor': 11, 'region_summary': 11}


In [19]:
EXPECTED_METRIC_FRAMEWORK_COUNTS = {
    "region_policy": 143,
    "metric_family_summary": 13,
    "region_summary": 11,
    "quality_anchor": 11,
}

OBSERVED_METRIC_FRAMEWORK_COUNTS = (
    METRIC_FRAMEWORK["record_type"]
    .value_counts()
    .to_dict()
)

POLICY_COMPATIBILITY_COUNTS = (
    POLICY_FRAME["compatible"]
    .value_counts()
    .to_dict()
)

ANCHOR_POLICY_IDS = set(
    ANCHOR_FRAME["policy_id"].astype(str)
)

APPROVED_POLICY_IDS = set(
    POLICY_FRAME.loc[
        POLICY_FRAME["compatible"],
        "policy_id",
    ].astype(str)
)

BATCH_4_CHECKS = (
    (
        "metric_framework_row_count",
        "The metric framework contains 178 normalized rows",
        178,
        len(METRIC_FRAMEWORK),
        len(METRIC_FRAMEWORK) == 178,
    ),
    (
        "record_type_counts",
        "Every framework record type has its expected population",
        EXPECTED_METRIC_FRAMEWORK_COUNTS,
        OBSERVED_METRIC_FRAMEWORK_COUNTS,
        (
            OBSERVED_METRIC_FRAMEWORK_COUNTS
            == EXPECTED_METRIC_FRAMEWORK_COUNTS
        ),
    ),
    (
        "policy_population",
        "All 143 canonical region policies are retained",
        143,
        len(POLICY_FRAME),
        len(POLICY_FRAME) == 143,
    ),
    (
        "policy_compatibility_counts",
        "Approved and prohibited policy counts are preserved",
        {True: 86, False: 57},
        POLICY_COMPATIBILITY_COUNTS,
        (
            POLICY_COMPATIBILITY_COUNTS
            == {True: 86, False: 57}
        ),
    ),
    (
        "metric_family_population",
        "All 13 policy metric families are summarized",
        13,
        len(METRIC_FAMILY_LABELS),
        len(METRIC_FAMILY_LABELS) == 13,
    ),
    (
        "region_population",
        "All 11 canonical regions are summarized",
        11,
        len(REGION_LABELS),
        len(REGION_LABELS) == 11,
    ),
    (
        "quality_anchor_population",
        "All 11 quality anchors are documented",
        11,
        len(ANCHOR_FRAME),
        len(ANCHOR_FRAME) == 11,
    ),
    (
        "quality_anchor_policy_support",
        "Every quality anchor uses an approved region policy",
        sorted(ANCHOR_POLICY_IDS),
        sorted(ANCHOR_POLICY_IDS & APPROVED_POLICY_IDS),
        ANCHOR_POLICY_IDS.issubset(APPROVED_POLICY_IDS),
    ),
    (
        "metric_identifier_uniqueness",
        "Metric-framework identifiers are unique",
        len(METRIC_FRAMEWORK),
        METRIC_FRAMEWORK["metric_row_id"].nunique(),
        (
            METRIC_FRAMEWORK["metric_row_id"].nunique()
            == len(METRIC_FRAMEWORK)
        ),
    ),
    (
        "metric_display_order",
        "Metric-framework display order is contiguous",
        list(range(1, len(METRIC_FRAMEWORK) + 1)),
        METRIC_FRAMEWORK["display_order"].tolist(),
        (
            METRIC_FRAMEWORK["display_order"].tolist()
            == list(range(1, len(METRIC_FRAMEWORK) + 1))
        ),
    ),
    (
        "ablation_json_valid",
        "Every ablation-policy field is valid JSON",
        True,
        all(
            isinstance(parse_json_list(value), list)
            for value in METRIC_FRAMEWORK[
                "ablation_policy_ids_json"
            ]
        ),
        all(
            isinstance(parse_json_list(value), list)
            for value in METRIC_FRAMEWORK[
                "ablation_policy_ids_json"
            ]
        ),
    ),
    (
        "metric_sources_relative",
        "Every metric-framework source path is repository-relative",
        True,
        all_source_paths_are_relative(
            METRIC_FRAMEWORK["source_paths_json"]
        ),
        all_source_paths_are_relative(
            METRIC_FRAMEWORK["source_paths_json"]
        ),
    ),
    (
        "metric_framework_status",
        "All metric-framework records have valid status",
        {"ok"},
        set(METRIC_FRAMEWORK["status"]),
        set(METRIC_FRAMEWORK["status"]) == {"ok"},
    ),
    (
        "no_canonical_files_written",
        "Batch 4 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_4_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_4_metric_framework",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Metric-framework evidence differs from its approved source."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_4_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_4_metric_framework"
        )
    ]
    .reset_index(drop=True)
)

display(
    BATCH_4_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 4 completed.")
print("Metric-framework rows:", len(METRIC_FRAMEWORK))
print("Approved metric-region policies: 86")
print("Prohibited metric-region policies: 57")
print("Quality anchors:", len(ANCHOR_FRAME))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,check_id,severity,passed,observed
0,metric_framework_row_count,blocking,True,178
1,record_type_counts,blocking,True,"{""metric_family_summary"": 13, ""quality_anchor""..."
2,policy_population,blocking,True,143
3,policy_compatibility_counts,blocking,True,"{""false"": 57, ""true"": 86}"
4,metric_family_population,blocking,True,13
5,region_population,blocking,True,11
6,quality_anchor_population,blocking,True,11
7,quality_anchor_policy_support,blocking,True,"[""classical_pixel::masked_region"", ""clip::mask..."
8,metric_identifier_uniqueness,blocking,True,178
9,metric_display_order,blocking,True,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."


Batch 4 completed.
Metric-framework rows: 178
Approved metric-region policies: 86
Prohibited metric-region policies: 57
Quality anchors: 11
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 5 — Model-performance evidence

This batch prepares the main model-comparison table used by the Model Performance dashboard page.

It retains:

- the 11 approved quality anchors;
- the complete overall three-model comparison;
- category, damage-type, experiment, degradation-type, severity, damage-size, and zero-control slices;
- descriptive quartiles and model ranks;
- the bounded ten-case SDXL comparison as a clearly separate population.

Style or period comparisons are not promoted to the dashboard because only 18 of 50 paintings have complete style or period metadata.

The lower and upper values stored here are descriptive first and third quartiles. They are not confidence intervals. Model winners remain specific to a metric, region, population, and condition.

In [20]:
SUPPORTED_CORE_PERFORMANCE_SCOPES = [
    "overall",
    "category",
    "damage_type",
    "experiment",
    "degradation_type",
    "severity",
    "target_damage_fraction",
    "zero_control",
]

PERFORMANCE_SOURCE = (
    MODEL_COMPARISON_SOURCE.loc[
        normalized_text(
            MODEL_COMPARISON_SOURCE["anchor_id"]
        ).ne("")
        & (
            (
                MODEL_COMPARISON_SOURCE["population_id"].eq(
                    "core_three_model"
                )
                & MODEL_COMPARISON_SOURCE[
                    "analysis_scope"
                ].isin(SUPPORTED_CORE_PERFORMANCE_SCOPES)
            )
            |
            (
                MODEL_COMPARISON_SOURCE["population_id"].eq(
                    "sdxl_four_model_subset"
                )
                & MODEL_COMPARISON_SOURCE[
                    "analysis_scope"
                ].eq("overall")
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

MODEL_DISPLAY_NAMES = dict(
    zip(
        MODEL_DIMENSION["model_id"].astype(str),
        MODEL_DIMENSION["display_name"].astype(str),
    )
)


def readable_identifier(value: object) -> str:
    """Convert a machine identifier into a concise display label."""

    return (
        str(value)
        .replace("_", " ")
        .strip()
        .title()
    )


def performance_interpretation(row: object) -> str:
    """Create a scoped, metric-specific result statement."""

    model_id = str(row.model_id)
    model_name = MODEL_DISPLAY_NAMES.get(
        model_id,
        readable_identifier(model_id),
    )
    winner_id = str(row.winner_model_id)
    winner_name = MODEL_DISPLAY_NAMES.get(
        winner_id,
        readable_identifier(winner_id),
    )

    rank_value = float(row.aggregate_rank)
    rank_text = (
        str(int(rank_value))
        if rank_value.is_integer()
        else f"{rank_value:.2f}"
    )

    if model_id == winner_id:
        result = (
            f"{model_name} ranks first for this metric and condition."
        )
    else:
        result = (
            f"{model_name} ranks {rank_text}; "
            f"{winner_name} ranks first for this metric and condition."
        )

    if str(row.population_id) == "sdxl_four_model_subset":
        result += (
            " This ordering applies only to the bounded ten-case "
            "SDXL comparison and must not be generalized."
        )
    else:
        result += (
            " The result is valid only for the displayed metric, "
            "region, and evidence slice."
        )

    return result


print("Approved performance rows selected:", len(PERFORMANCE_SOURCE))
print(
    "Population counts:",
    PERFORMANCE_SOURCE["population_id"]
    .value_counts()
    .to_dict(),
)
print(
    "Core scope counts:",
    PERFORMANCE_SOURCE.loc[
        PERFORMANCE_SOURCE["population_id"].eq(
            "core_three_model"
        ),
        "analysis_scope",
    ]
    .value_counts()
    .sort_index()
    .to_dict(),
)

Approved performance rows selected: 1241
Population counts: {'core_three_model': 1197, 'sdxl_four_model_subset': 44}
Core scope counts: {'category': 165, 'damage_type': 168, 'degradation_type': 165, 'experiment': 132, 'overall': 33, 'severity': 132, 'target_damage_fraction': 366, 'zero_control': 36}


In [21]:
PERFORMANCE_ROWS = []

for row in PERFORMANCE_SOURCE.itertuples(index=False):
    population_id = str(row.population_id)
    analysis_scope = str(row.analysis_scope)
    scope_value = str(row.scope_value)

    is_overall = analysis_scope == "overall"
    is_sdxl_subset = (
        population_id == "sdxl_four_model_subset"
    )

    section_id = (
        "sdxl_bounded_comparison"
        if is_sdxl_subset
        else (
            "core_overall_performance"
            if is_overall
            else "core_conditional_performance"
        )
    )

    PERFORMANCE_ROWS.append(
        {
            "performance_row_id": stable_id(
                "performance",
                row.comparison_row_id,
            ),
            "page_id": "model_performance",
            "section_id": section_id,
            "display_order": 0,
            "record_type": "quality_anchor_summary",
            "population_id": population_id,
            "analysis_scope": analysis_scope,
            "scope_value": scope_value,
            "experiment_id": (
                scope_value
                if analysis_scope == "experiment"
                else ""
            ),
            "condition_field": (
                ""
                if is_overall
                else analysis_scope
            ),
            "condition_value": (
                ""
                if is_overall
                else scope_value
            ),
            "evidence_family": str(row.evidence_family),
            "metric_family": str(row.metric_family),
            "metric_id": str(row.metric_id),
            "metric_name": str(row.metric_name),
            "feature_model_id": (
                ""
                if pd.isna(row.feature_model_id)
                else str(row.feature_model_id)
            ),
            "region_id": str(row.region_id),
            "summary_statistic": str(row.summary_statistic),
            "comparison_direction": str(
                row.comparison_direction
            ),
            "model_id": str(row.model_id),
            "estimate": float(row.restored_mean),
            "interval_low": float(row.restored_q25),
            "interval_high": float(row.restored_q75),
            "rank": float(row.aggregate_rank),
            "winner_model_id": str(row.winner_model_id),
            "case_count": int(float(row.paired_case_count)),
            "painting_count": int(
                float(row.paired_painting_count)
            ),
            "coverage_fraction": float(row.coverage_fraction),
            "applicability_status": (
                "bounded_partial_evaluation"
                if is_sdxl_subset
                else "fully_evaluated_core_comparison"
            ),
            "source_notebook_ids_json": json_list(["21"]),
            "source_paths_json": json_list(
                [SETTINGS["inputs"]["model_comparison_path"]]
            ),
            "interpretation": performance_interpretation(row),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

PERFORMANCE_SUMMARY = pd.DataFrame(PERFORMANCE_ROWS)

SECTION_ORDER = {
    "core_overall_performance": 1,
    "core_conditional_performance": 2,
    "sdxl_bounded_comparison": 3,
}

PERFORMANCE_SUMMARY["_section_order"] = (
    PERFORMANCE_SUMMARY["section_id"].map(SECTION_ORDER)
)

PERFORMANCE_SUMMARY = (
    PERFORMANCE_SUMMARY
    .sort_values(
        [
            "_section_order",
            "analysis_scope",
            "scope_value",
            "evidence_family",
            "metric_name",
            "region_id",
            "rank",
            "model_id",
        ],
        kind="stable",
    )
    .drop(columns="_section_order")
    .reset_index(drop=True)
)

PERFORMANCE_SUMMARY["display_order"] = (
    PERFORMANCE_SUMMARY.index + 1
)

PERFORMANCE_SUMMARY = validate_output_frame(
    PERFORMANCE_SUMMARY,
    "performance_summary",
)

print("Performance-summary rows:", len(PERFORMANCE_SUMMARY))
print(
    "Section counts:",
    PERFORMANCE_SUMMARY["section_id"]
    .value_counts()
    .to_dict(),
)

Performance-summary rows: 1241
Section counts: {'core_conditional_performance': 1164, 'sdxl_bounded_comparison': 44, 'core_overall_performance': 33}


In [22]:
CORE_OVERALL_VIEW = (
    PERFORMANCE_SUMMARY.loc[
        PERFORMANCE_SUMMARY["section_id"].eq(
            "core_overall_performance"
        ),
        [
            "metric_name",
            "feature_model_id",
            "region_id",
            "comparison_direction",
            "model_id",
            "estimate",
            "interval_low",
            "interval_high",
            "rank",
            "winner_model_id",
        ],
    ]
    .sort_values(["metric_name", "rank"])
    .reset_index(drop=True)
)

OVERALL_WINNER_VIEW = (
    CORE_OVERALL_VIEW[
        [
            "metric_name",
            "feature_model_id",
            "region_id",
            "comparison_direction",
            "winner_model_id",
        ]
    ]
    .drop_duplicates()
    .sort_values(["metric_name", "region_id"])
    .reset_index(drop=True)
)

CONDITIONAL_COVERAGE_VIEW = (
    PERFORMANCE_SUMMARY.loc[
        PERFORMANCE_SUMMARY["population_id"].eq(
            "core_three_model"
        )
    ]
    .groupby(
        ["analysis_scope", "scope_value"],
        dropna=False,
        sort=True,
    )
    .agg(
        rows=("performance_row_id", "size"),
        anchors=("metric_name", "nunique"),
        models=("model_id", "nunique"),
        cases=("case_count", "max"),
        paintings=("painting_count", "max"),
    )
    .reset_index()
)

SDXL_BOUNDED_VIEW = (
    PERFORMANCE_SUMMARY.loc[
        PERFORMANCE_SUMMARY["section_id"].eq(
            "sdxl_bounded_comparison"
        ),
        [
            "metric_name",
            "model_id",
            "estimate",
            "rank",
            "winner_model_id",
            "case_count",
            "applicability_status",
        ],
    ]
    .sort_values(["metric_name", "rank"])
    .reset_index(drop=True)
)

display(CORE_OVERALL_VIEW)
display(OVERALL_WINNER_VIEW)
display(CONDITIONAL_COVERAGE_VIEW)
display(SDXL_BOUNDED_VIEW)

print(
    "Core overall rows:",
    len(CORE_OVERALL_VIEW),
)
print(
    "Conditional slices:",
    len(CONDITIONAL_COVERAGE_VIEW),
)
print(
    "Bounded SDXL-subset rows:",
    len(SDXL_BOUNDED_VIEW),
)

,metric_name,feature_model_id,region_id,comparison_direction,model_id,estimate,interval_low,interval_high,rank,winner_model_id
0,boundary_gradient_mismatch,,boundary_ring,lower_is_better,lama,0.006414,0.004697,0.007702,1.0,lama
1,boundary_gradient_mismatch,,boundary_ring,lower_is_better,opencv_telea,0.007586,0.005270,0.009315,2.0,lama
2,boundary_gradient_mismatch,,boundary_ring,lower_is_better,stable_diffusion_inpainting,0.019966,0.011598,0.024711,3.0,lama
3,clip_cosine_similarity,clip_vit_b32,mask_bbox_crop,higher_is_better,lama,0.956540,0.935779,0.998813,1.0,lama
4,clip_cosine_similarity,clip_vit_b32,mask_bbox_crop,higher_is_better,stable_diffusion_inpainting,0.943051,0.919180,0.988880,2.0,lama
5,clip_cosine_similarity,clip_vit_b32,mask_bbox_crop,higher_is_better,opencv_telea,0.886278,0.773813,0.997944,3.0,lama
6,delta_e_ciede2000_mean,,masked_region,lower_is_better,lama,6.930979,4.094912,9.281761,1.0,lama
7,delta_e_ciede2000_mean,,masked_region,lower_is_better,opencv_telea,7.858020,4.310648,10.617390,2.0,lama
8,delta_e_ciede2000_mean,,masked_region,lower_is_better,stable_diffusion_inpainting,12.956593,9.144358,16.065949,3.0,lama
9,dinov2_cosine_similarity,dinov2_vits14,mask_bbox_crop,higher_is_better,lama,0.871590,0.774841,0.997295,1.0,lama


,metric_name,feature_model_id,region_id,comparison_direction,winner_model_id
0,boundary_gradient_mismatch,,boundary_ring,lower_is_better,lama
1,clip_cosine_similarity,clip_vit_b32,mask_bbox_crop,higher_is_better,lama
2,delta_e_ciede2000_mean,,masked_region,lower_is_better,lama
3,dinov2_cosine_similarity,dinov2_vits14,mask_bbox_crop,higher_is_better,lama
4,local_patch_cosine_similarity,dinov2_vits14,mask_bbox_crop,higher_is_better,lama
5,local_texture_error_p95,,mask_bbox_crop,lower_is_better,lama
6,lpips,,mask_bbox_crop,lower_is_better,lama
7,mae,,masked_region,lower_is_better,lama
8,reference_affinity_map_correlation,dinov2_vits14,content_region,higher_is_better,lama
9,restored_error_mean,,masked_region,lower_is_better,lama


,analysis_scope,scope_value,rows,anchors,models,cases,paintings
0,category,abstraction_surrealism,33,11,3,82,10
1,category,architecture_structured,33,11,3,82,10
2,category,high_texture_brushwork,33,11,3,82,10
3,category,landscape_natural,33,11,3,82,10
4,category,portrait_figure,33,11,3,82,10
5,damage_type,loss_large,33,11,3,110,50
6,damage_type,loss_small,33,11,3,75,50
7,damage_type,mixed_damage,33,11,3,50,50
8,damage_type,not_applicable,33,11,3,50,5
9,damage_type,scratch_thin,33,11,3,75,50


,metric_name,model_id,estimate,rank,winner_model_id,case_count,applicability_status
0,boundary_gradient_mismatch,lama,0.005445,1.0,lama,10,bounded_partial_evaluation
1,boundary_gradient_mismatch,opencv_telea,0.006761,2.0,lama,10,bounded_partial_evaluation
2,boundary_gradient_mismatch,stable_diffusion_inpainting,0.016991,3.0,lama,10,bounded_partial_evaluation
3,boundary_gradient_mismatch,sdxl_inpainting,0.050275,4.0,lama,10,bounded_partial_evaluation
4,clip_cosine_similarity,stable_diffusion_inpainting,0.898785,1.0,stable_diffusion_inpainting,10,bounded_partial_evaluation
5,clip_cosine_similarity,lama,0.876445,2.0,stable_diffusion_inpainting,10,bounded_partial_evaluation
6,clip_cosine_similarity,sdxl_inpainting,0.823591,3.0,stable_diffusion_inpainting,10,bounded_partial_evaluation
7,clip_cosine_similarity,opencv_telea,0.814445,4.0,stable_diffusion_inpainting,10,bounded_partial_evaluation
8,delta_e_ciede2000_mean,lama,8.685043,1.0,lama,10,bounded_partial_evaluation
9,delta_e_ciede2000_mean,opencv_telea,10.404084,2.0,lama,10,bounded_partial_evaluation


Core overall rows: 33
Conditional slices: 39
Bounded SDXL-subset rows: 44


In [23]:
EXPECTED_CORE_SCOPE_COUNTS = {
    "overall": 33,
    "category": 165,
    "damage_type": 168,
    "experiment": 132,
    "degradation_type": 165,
    "severity": 132,
    "target_damage_fraction": 366,
    "zero_control": 36,
}

OBSERVED_CORE_SCOPE_COUNTS = (
    PERFORMANCE_SUMMARY.loc[
        PERFORMANCE_SUMMARY["population_id"].eq(
            "core_three_model"
        ),
        "analysis_scope",
    ]
    .value_counts()
    .to_dict()
)

OVERALL_ANCHOR_WINNERS = (
    PERFORMANCE_SUMMARY.loc[
        PERFORMANCE_SUMMARY["section_id"].eq(
            "core_overall_performance"
        ),
        ["metric_name", "region_id", "winner_model_id"],
    ]
    .drop_duplicates()
)

OVERALL_WINNER_COUNTS = (
    OVERALL_ANCHOR_WINNERS["winner_model_id"]
    .value_counts()
    .to_dict()
)

NUMERIC_PERFORMANCE_COLUMNS = [
    "estimate",
    "interval_low",
    "interval_high",
    "rank",
    "case_count",
    "painting_count",
    "coverage_fraction",
]

ALL_PERFORMANCE_NUMERIC_FINITE = bool(
    PERFORMANCE_SUMMARY[
        NUMERIC_PERFORMANCE_COLUMNS
    ]
    .apply(pd.to_numeric, errors="coerce")
    .notna()
    .all()
    .all()
)

BATCH_5_CHECKS = (
    (
        "performance_row_count",
        "The approved performance table contains 1,241 rows",
        1241,
        len(PERFORMANCE_SUMMARY),
        len(PERFORMANCE_SUMMARY) == 1241,
    ),
    (
        "core_performance_population",
        "Core three-model summaries contain 1,197 rows",
        1197,
        int(
            PERFORMANCE_SUMMARY["population_id"]
            .eq("core_three_model")
            .sum()
        ),
        int(
            PERFORMANCE_SUMMARY["population_id"]
            .eq("core_three_model")
            .sum()
        ) == 1197,
    ),
    (
        "sdxl_bounded_population",
        "The bounded SDXL comparison contains 44 rows",
        44,
        int(
            PERFORMANCE_SUMMARY["population_id"]
            .eq("sdxl_four_model_subset")
            .sum()
        ),
        int(
            PERFORMANCE_SUMMARY["population_id"]
            .eq("sdxl_four_model_subset")
            .sum()
        ) == 44,
    ),
    (
        "core_scope_counts",
        "Every approved core comparison scope is complete",
        EXPECTED_CORE_SCOPE_COUNTS,
        OBSERVED_CORE_SCOPE_COUNTS,
        (
            OBSERVED_CORE_SCOPE_COUNTS
            == EXPECTED_CORE_SCOPE_COUNTS
        ),
    ),
    (
        "style_scope_excluded",
        "Incomplete style metadata is not promoted to the dashboard",
        0,
        int(
            PERFORMANCE_SUMMARY["analysis_scope"]
            .eq("style_or_period")
            .sum()
        ),
        not PERFORMANCE_SUMMARY[
            "analysis_scope"
        ].eq("style_or_period").any(),
    ),
    (
        "core_overall_population",
        "Overall comparison contains 11 anchors and three models",
        {
            "rows": 33,
            "anchors": 11,
            "models": 3,
        },
        {
            "rows": len(CORE_OVERALL_VIEW),
            "anchors": len(OVERALL_ANCHOR_WINNERS),
            "models": CORE_OVERALL_VIEW[
                "model_id"
            ].nunique(),
        },
        (
            len(CORE_OVERALL_VIEW) == 33
            and len(OVERALL_ANCHOR_WINNERS) == 11
            and CORE_OVERALL_VIEW["model_id"].nunique() == 3
        ),
    ),
    (
        "overall_winner_counts",
        "Overall metric-specific winners match validated evidence",
        {
            "lama": 10,
            "opencv_telea": 1,
        },
        OVERALL_WINNER_COUNTS,
        (
            OVERALL_WINNER_COUNTS
            == {
                "lama": 10,
                "opencv_telea": 1,
            }
        ),
    ),
    (
        "performance_identifier_uniqueness",
        "Performance identifiers are unique",
        len(PERFORMANCE_SUMMARY),
        PERFORMANCE_SUMMARY[
            "performance_row_id"
        ].nunique(),
        (
            PERFORMANCE_SUMMARY[
                "performance_row_id"
            ].nunique()
            == len(PERFORMANCE_SUMMARY)
        ),
    ),
    (
        "performance_numeric_values",
        "Required performance values are finite",
        True,
        ALL_PERFORMANCE_NUMERIC_FINITE,
        ALL_PERFORMANCE_NUMERIC_FINITE,
    ),
    (
        "descriptive_interval_order",
        "First quartiles do not exceed third quartiles",
        True,
        bool(
            (
                PERFORMANCE_SUMMARY["interval_low"]
                <= PERFORMANCE_SUMMARY["interval_high"]
            ).all()
        ),
        bool(
            (
                PERFORMANCE_SUMMARY["interval_low"]
                <= PERFORMANCE_SUMMARY["interval_high"]
            ).all()
        ),
    ),
    (
        "sdxl_applicability",
        "Every SDXL-subset row remains explicitly bounded",
        {"bounded_partial_evaluation"},
        set(
            PERFORMANCE_SUMMARY.loc[
                PERFORMANCE_SUMMARY["population_id"].eq(
                    "sdxl_four_model_subset"
                ),
                "applicability_status",
            ]
        ),
        (
            set(
                PERFORMANCE_SUMMARY.loc[
                    PERFORMANCE_SUMMARY[
                        "population_id"
                    ].eq("sdxl_four_model_subset"),
                    "applicability_status",
                ]
            )
            == {"bounded_partial_evaluation"}
        ),
    ),
    (
        "performance_sources_relative",
        "Every performance source path is repository-relative",
        True,
        all_source_paths_are_relative(
            PERFORMANCE_SUMMARY["source_paths_json"]
        ),
        all_source_paths_are_relative(
            PERFORMANCE_SUMMARY["source_paths_json"]
        ),
    ),
    (
        "performance_status",
        "All performance rows have valid status",
        {"ok"},
        set(PERFORMANCE_SUMMARY["status"]),
        set(PERFORMANCE_SUMMARY["status"]) == {"ok"},
    ),
    (
        "no_canonical_files_written",
        "Batch 5 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_5_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_5_model_performance",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Model-performance evidence differs from its approved source."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_5_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_5_model_performance"
        )
    ]
    .reset_index(drop=True)
)

display(
    BATCH_5_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 5 completed.")
print("Performance rows:", len(PERFORMANCE_SUMMARY))
print("Core comparison rows: 1,197")
print("Bounded SDXL-subset rows: 44")
print("Overall winner counts:", OVERALL_WINNER_COUNTS)
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,check_id,severity,passed,observed
0,performance_row_count,blocking,True,1241
1,core_performance_population,blocking,True,1197
2,sdxl_bounded_population,blocking,True,44
3,core_scope_counts,blocking,True,"{""category"": 165, ""damage_type"": 168, ""degrada..."
4,style_scope_excluded,blocking,True,0
5,core_overall_population,blocking,True,"{""anchors"": 11, ""models"": 3, ""rows"": 33}"
6,overall_winner_counts,blocking,True,"{""lama"": 10, ""opencv_telea"": 1}"
7,performance_identifier_uniqueness,blocking,True,1241
8,performance_numeric_values,blocking,True,True
9,descriptive_interval_order,blocking,True,True


Batch 5 completed.
Performance rows: 1241
Core comparison rows: 1,197
Bounded SDXL-subset rows: 44
Overall winner counts: {'lama': 10, 'opencv_telea': 1}
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 6 — Sensitivity, robustness, and uncertainty

This batch prepares the Robustness & Uncertainty dashboard evidence.

It retains every analytical result from:

- damage-size sensitivity;
- mask-geometry robustness;
- synthetic-degradation sensitivity.

It also creates a group-level uncertainty table containing:

- canonical Stable Diffusion repeated-seed summaries;
- damage-size pixelwise uncertainty summaries;
- explicit non-applicability records for deterministic methods and the single-seed SDXL subset.

No uncertainty value is invented for Telea, LaMa, or SDXL. Empty applicability records remain empty rather than being encoded as zero uncertainty.

In [24]:
DAMAGE_SIZE_ANALYSIS_SOURCE = (
    INPUT_TABLES["damage_size_analysis_path"]
    .copy()
    .reset_index(drop=True)
)

MASK_ROBUSTNESS_ANALYSIS_SOURCE = (
    INPUT_TABLES["mask_robustness_analysis_path"]
    .copy()
    .reset_index(drop=True)
)

DEGRADATION_ANALYSIS_SOURCE = (
    INPUT_TABLES["degradation_analysis_path"]
    .copy()
    .reset_index(drop=True)
)


def blank_safe_text(value: object) -> str:
    """Return an empty string for missing scalar values."""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


def optional_float(value: object) -> float:
    """Convert a scalar to float while preserving missing values."""

    text = blank_safe_text(value)
    return float(text) if text else float("nan")


def optional_count(value: object) -> int:
    """Convert a count to int, using zero only for absent counts."""

    text = blank_safe_text(value)
    return int(float(text)) if text else 0


def source_condition(
    record: dict,
    candidate_fields: list[str],
) -> tuple[str, str]:
    """Encode the non-empty source dimensions without losing context."""

    pairs = []

    for field in candidate_fields:
        value = blank_safe_text(record.get(field))

        if value and value.lower() not in {
            "not_applicable",
            "not_recorded",
        }:
            pairs.append((field, value))

    if not pairs:
        return "overall", "all"

    return (
        "|".join(field for field, _ in pairs),
        "|".join(value for _, value in pairs),
    )


def source_condition_order(record: dict) -> float:
    """Return a sortable damage or severity order when available."""

    for field in [
        "target_damage_fraction",
        "severity_rank",
    ]:
        value = optional_float(record.get(field))

        if pd.notna(value):
            return float(value)

    variant_id = blank_safe_text(record.get("variant_id"))

    if variant_id.startswith("variant_"):
        suffix = variant_id.removeprefix("variant_")

        if suffix.isdigit():
            return float(int(suffix))

    return float("nan")


ANALYSIS_SOURCES = [
    {
        "analysis_family": "damage_size_sensitivity",
        "experiment_id": "damage_size_sensitivity",
        "source_notebook_id": "23",
        "source_path": SETTINGS["inputs"][
            "damage_size_analysis_path"
        ],
        "frame": DAMAGE_SIZE_ANALYSIS_SOURCE,
        "condition_fields": [
            "level_id",
            "target_damage_fraction",
            "painting_id",
            "scope_type",
            "scope_value",
            "exposure_definition",
        ],
    },
    {
        "analysis_family": "mask_robustness",
        "experiment_id": "mask_robustness",
        "source_notebook_id": "24",
        "source_path": SETTINGS["inputs"][
            "mask_robustness_analysis_path"
        ],
        "frame": MASK_ROBUSTNESS_ANALYSIS_SOURCE,
        "condition_fields": [
            "mask_family",
            "robustness_group_id",
            "variant_id",
            "morphology_field",
            "painting_id",
            "scope_type",
            "scope_value",
        ],
    },
    {
        "analysis_family": "synthetic_degradation",
        "experiment_id": "synthetic_degradation",
        "source_notebook_id": "25",
        "source_path": SETTINGS["inputs"][
            "degradation_analysis_path"
        ],
        "frame": DEGRADATION_ANALYSIS_SOURCE,
        "condition_fields": [
            "degradation_family",
            "severity",
            "component_degradation",
            "painting_id",
            "scope_type",
            "scope_value",
        ],
    },
]

print(
    "Sensitivity source rows:",
    {
        item["analysis_family"]: len(item["frame"])
        for item in ANALYSIS_SOURCES
    },
)

Sensitivity source rows: {'damage_size_sensitivity': 1901, 'mask_robustness': 5373, 'synthetic_degradation': 4695}


In [25]:
SENSITIVITY_ROWS = []

for source in ANALYSIS_SOURCES:
    for row in source["frame"].itertuples(index=False):
        record = row._asdict()

        condition_field, condition_value = source_condition(
            record,
            source["condition_fields"],
        )

        analysis_kind = blank_safe_text(
            record.get("analysis_kind")
        )

        applicability_status = blank_safe_text(
            record.get("applicability_status")
        )

        independent_unit = blank_safe_text(
            record.get("independent_unit")
        )

        SENSITIVITY_ROWS.append(
            {
                "sensitivity_row_id": stable_id(
                    "sensitivity",
                    source["source_notebook_id"],
                    record["analysis_row_id"],
                ),
                "page_id": "robustness_uncertainty",
                "section_id": source["analysis_family"],
                "display_order": 0,
                "analysis_family": source["analysis_family"],
                "analysis_kind": analysis_kind,
                "experiment_id": source["experiment_id"],
                "condition_field": condition_field,
                "condition_value": condition_value,
                "condition_order": source_condition_order(record),
                "evidence_family": blank_safe_text(
                    record.get("evidence_family")
                ),
                "metric_family": blank_safe_text(
                    record.get("metric_family")
                ),
                "metric_name": blank_safe_text(
                    record.get("metric_name")
                ),
                "feature_model_id": blank_safe_text(
                    record.get("feature_model_id")
                ),
                "region_id": blank_safe_text(
                    record.get("region_id")
                ),
                "summary_statistic": blank_safe_text(
                    record.get("summary_statistic")
                ),
                "comparison_direction": blank_safe_text(
                    record.get("comparison_direction")
                ),
                "model_id": blank_safe_text(
                    record.get("model_id")
                ),
                "comparison_model_id": blank_safe_text(
                    record.get("comparison_model_id")
                ),
                "estimate_name": blank_safe_text(
                    record.get("estimate_name")
                ),
                "estimate": float(record["estimate"]),
                "interval_low": optional_float(
                    record.get("ci_lower")
                ),
                "interval_high": optional_float(
                    record.get("ci_upper")
                ),
                "effect_size_name": blank_safe_text(
                    record.get("effect_size_name")
                ),
                "effect_size": optional_float(
                    record.get("effect_size")
                ),
                "p_value": optional_float(
                    record.get("p_value")
                ),
                "q_value": optional_float(
                    record.get("q_value")
                ),
                "independent_unit": independent_unit,
                "n_paintings": optional_count(
                    record.get("n_paintings")
                ),
                "n_cases": optional_count(
                    record.get("n_cases")
                ),
                "n_observations": optional_count(
                    record.get("n_observations")
                ),
                "applicability_status": applicability_status,
                "source_notebook_ids_json": json_list(
                    [source["source_notebook_id"]]
                ),
                "source_paths_json": json_list(
                    [source["source_path"]]
                ),
                "interpretation": (
                    f"{readable_identifier(analysis_kind)} result from "
                    f"Notebook {source['source_notebook_id']}. "
                    f"The declared independent unit is "
                    f"{independent_unit or 'not applicable'}; "
                    f"applicability is {applicability_status}."
                ),
                "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
                "status": "ok",
                "issue": "",
            }
        )

SENSITIVITY_SUMMARY = pd.DataFrame(SENSITIVITY_ROWS)

SENSITIVITY_SUMMARY = (
    SENSITIVITY_SUMMARY
    .sort_values(
        [
            "analysis_family",
            "analysis_kind",
            "condition_order",
            "condition_value",
            "metric_name",
            "region_id",
            "model_id",
            "comparison_model_id",
        ],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

SENSITIVITY_SUMMARY["display_order"] = (
    SENSITIVITY_SUMMARY.index + 1
)

SENSITIVITY_SUMMARY = validate_output_frame(
    SENSITIVITY_SUMMARY,
    "sensitivity_summary",
)

print("Sensitivity rows constructed:", len(SENSITIVITY_SUMMARY))
print(
    "Analysis-family counts:",
    SENSITIVITY_SUMMARY["analysis_family"]
    .value_counts()
    .to_dict(),
)

Sensitivity rows constructed: 11969
Analysis-family counts: {'mask_robustness': 5373, 'synthetic_degradation': 4695, 'damage_size_sensitivity': 1901}


In [26]:
SENSITIVITY_KIND_SUMMARY = (
    SENSITIVITY_SUMMARY.groupby(
        [
            "analysis_family",
            "analysis_kind",
            "applicability_status",
        ],
        dropna=False,
        sort=True,
    )
    .agg(
        rows=("sensitivity_row_id", "size"),
        models=("model_id", "nunique"),
        metrics=("metric_name", "nunique"),
        regions=("region_id", "nunique"),
        maximum_paintings=("n_paintings", "max"),
        maximum_cases=("n_cases", "max"),
        maximum_observations=("n_observations", "max"),
    )
    .reset_index()
)

SENSITIVITY_INFERENCE_SUMMARY = (
    SENSITIVITY_SUMMARY.groupby(
        "analysis_family",
        sort=True,
    )
    .agg(
        rows=("sensitivity_row_id", "size"),
        estimates=("estimate", "count"),
        interval_rows=("interval_low", "count"),
        effect_size_rows=("effect_size", "count"),
        p_value_rows=("p_value", "count"),
        q_value_rows=("q_value", "count"),
    )
    .reset_index()
)

display(SENSITIVITY_KIND_SUMMARY)
display(SENSITIVITY_INFERENCE_SUMMARY)

print(
    "Distinct sensitivity analysis kinds:",
    SENSITIVITY_SUMMARY["analysis_kind"].nunique(),
)

,analysis_family,analysis_kind,applicability_status,rows,models,metrics,regions,maximum_paintings,maximum_cases,maximum_observations
0,damage_size_sensitivity,adjacent_change,applicable,198,3,11,4,5,10,5
1,damage_size_sensitivity,damage_integrity,applicable,49,1,7,1,5,5,5
2,damage_size_sensitivity,damage_trend,applicable,72,3,12,5,5,35,35
3,damage_size_sensitivity,evaluated_baseline_type_contrast,applicable,22,1,11,4,5,35,5
4,damage_size_sensitivity,level_summary,applicable,252,3,12,5,5,5,5
5,damage_size_sensitivity,morphology_association,applicable,228,3,16,4,5,35,35
6,damage_size_sensitivity,painting_trajectory,applicable,660,3,19,7,1,7,7
7,damage_size_sensitivity,paired_model_contrast,applicable,66,2,11,4,5,35,5
8,damage_size_sensitivity,ranking_by_level,applicable,21,3,1,1,5,5,165
9,damage_size_sensitivity,ranking_stability,applicable,63,3,3,1,5,5,5


,analysis_family,rows,estimates,interval_rows,effect_size_rows,p_value_rows,q_value_rows
0,damage_size_sensitivity,1901,1901,929,667,646,646
1,mask_robustness,5373,5373,573,669,429,429
2,synthetic_degradation,4695,4695,939,801,627,627


Distinct sensitivity analysis kinds: 39


In [27]:
CANONICAL_UNCERTAINTY_SOURCE = (
    INPUT_TABLES["canonical_uncertainty_path"]
    .copy()
    .reset_index(drop=True)
)

CANONICAL_UNCERTAINTY_METRICS = {
    "rgb_std_mean_masked": {
        "metric_family": "pixel_variability",
        "metric_name": "pixel_rgb_std_mean",
        "region_id": "masked_region",
        "summary_statistic": "mean",
        "value_unit": "rgb_8bit",
    },
    "rgb_std_p95_masked": {
        "metric_family": "pixel_variability",
        "metric_name": "pixel_rgb_std_p95",
        "region_id": "masked_region",
        "summary_statistic": "p95",
        "value_unit": "rgb_8bit",
    },
    "rgb_pair_mae_mean_masked": {
        "metric_family": "pixel_pairwise",
        "metric_name": "pairwise_rgb_mae",
        "region_id": "masked_region",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "rgb_8bit",
    },
    "rgb_pair_rmse_mean_masked": {
        "metric_family": "pixel_pairwise",
        "metric_name": "pairwise_rgb_rmse",
        "region_id": "masked_region",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "rgb_8bit",
    },
    "lpips_pair_mean_content": {
        "metric_family": "perceptual_pairwise",
        "metric_name": "pairwise_lpips_distance",
        "region_id": "content_region",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "unitless_distance",
    },
    "lpips_pair_mean_crop": {
        "metric_family": "perceptual_pairwise",
        "metric_name": "pairwise_lpips_distance",
        "region_id": "mask_bbox_crop",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "unitless_distance",
    },
    "clip_pair_distance_mean_content": {
        "metric_family": "feature_pairwise",
        "metric_name": "pairwise_clip_cosine_distance",
        "region_id": "content_region",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "unitless_distance",
    },
    "clip_pair_distance_mean_crop": {
        "metric_family": "feature_pairwise",
        "metric_name": "pairwise_clip_cosine_distance",
        "region_id": "mask_bbox_crop",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "unitless_distance",
    },
    "dino_pair_distance_mean_content": {
        "metric_family": "feature_pairwise",
        "metric_name": "pairwise_dinov2_cosine_distance",
        "region_id": "content_region",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "unitless_distance",
    },
    "dino_pair_distance_mean_crop": {
        "metric_family": "feature_pairwise",
        "metric_name": "pairwise_dinov2_cosine_distance",
        "region_id": "mask_bbox_crop",
        "summary_statistic": "mean_unordered_seed_pair",
        "value_unit": "unitless_distance",
    },
}

UNCERTAINTY_ROWS = []

for row in CANONICAL_UNCERTAINTY_SOURCE.itertuples(index=False):
    record = row._asdict()

    for source_column, metric in (
        CANONICAL_UNCERTAINTY_METRICS.items()
    ):
        UNCERTAINTY_ROWS.append(
            {
                "uncertainty_row_id": stable_id(
                    "uncertainty",
                    "canonical",
                    record["uncertainty_group_id"],
                    source_column,
                ),
                "page_id": "robustness_uncertainty",
                "section_id": "canonical_uncertainty",
                "display_order": 0,
                "record_type": "canonical_group_summary",
                "population_id": "canonical_repeated_seed",
                "experiment_id": str(record["experiment_id"]),
                "damage_or_degradation_type": str(
                    record["damage_or_degradation_type"]
                ),
                "prompt_variant_id": str(
                    record["prompt_variant_id"]
                ),
                "uncertainty_group_id": str(
                    record["uncertainty_group_id"]
                ),
                "painting_id": str(record["painting_id"]),
                "category": str(record["category"]),
                "metric_family": metric["metric_family"],
                "metric_name": metric["metric_name"],
                "region_id": metric["region_id"],
                "summary_statistic": metric[
                    "summary_statistic"
                ],
                "value": float(record[source_column]),
                "value_unit": metric["value_unit"],
                "seed_count": int(float(record["seed_count"])),
                "group_count": 1,
                "case_count": 1,
                "applicability_status": (
                    "supported_complete_seed_coverage"
                ),
                "is_calibrated_confidence": False,
                "source_notebook_ids_json": json_list(["18"]),
                "source_paths_json": json_list(
                    [
                        SETTINGS["inputs"][
                            "canonical_uncertainty_path"
                        ]
                    ]
                ),
                "interpretation": (
                    "Higher repeated-seed distance or variability means "
                    "the diffusion outputs are less consistent with one "
                    "another for this case and prompt arm."
                ),
                "limitation": (
                    "Repeated-seed variability is not calibrated "
                    "confidence and does not establish correctness."
                ),
                "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
                "status": "ok",
                "issue": "",
            }
        )

print(
    "Canonical uncertainty rows:",
    len(UNCERTAINTY_ROWS),
)

Canonical uncertainty rows: 1300


In [28]:
DAMAGE_SIZE_UNCERTAINTY_SOURCE = (
    INPUT_TABLES["damage_size_uncertainty_path"]
    .copy()
    .reset_index(drop=True)
)

DAMAGE_SIZE_GROUP_SUMMARIES = (
    DAMAGE_SIZE_UNCERTAINTY_SOURCE.loc[
        DAMAGE_SIZE_UNCERTAINTY_SOURCE[
            "observation_level"
        ].eq("group_summary")
    ]
    .copy()
    .reset_index(drop=True)
)

for row in DAMAGE_SIZE_GROUP_SUMMARIES.itertuples(index=False):
    UNCERTAINTY_ROWS.append(
        {
            "uncertainty_row_id": stable_id(
                "uncertainty",
                "damage_size",
                row.uncertainty_metric_id,
            ),
            "page_id": "robustness_uncertainty",
            "section_id": "damage_size_uncertainty",
            "display_order": 0,
            "record_type": "damage_size_group_summary",
            "population_id": "damage_size_repeated_seed",
            "experiment_id": str(row.experiment_id),
            "damage_or_degradation_type": str(
                row.damage_or_degradation_type
            ),
            "prompt_variant_id": str(row.prompt_variant_id),
            "uncertainty_group_id": str(
                row.uncertainty_group_id
            ),
            "painting_id": str(row.painting_id),
            "category": str(row.category),
            "metric_family": str(row.metric_family),
            "metric_name": str(row.metric_name),
            "region_id": str(row.region_id),
            "summary_statistic": str(row.summary_statistic),
            "value": float(row.value),
            "value_unit": str(row.value_unit),
            "seed_count": int(float(row.seed_count)),
            "group_count": 1,
            "case_count": 1,
            "applicability_status": (
                "supported_complete_seed_coverage"
            ),
            "is_calibrated_confidence": False,
            "source_notebook_ids_json": json_list(["22"]),
            "source_paths_json": json_list(
                [
                    SETTINGS["inputs"][
                        "damage_size_uncertainty_path"
                    ]
                ]
            ),
            "interpretation": (
                "This value measures pixelwise variation across "
                "repeated Stable Diffusion candidates at one "
                "damage-size level."
            ),
            "limitation": (
                "The value is an empirical seed-variability measure, "
                "not calibrated confidence."
            ),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

UNCERTAINTY_BOUNDARIES = [
    {
        "population_id": "opencv_telea_deterministic",
        "seed_count": 0,
        "applicability_status": "not_applicable_deterministic",
        "source_notebook_ids": ["09", "18"],
        "interpretation": (
            "OpenCV Telea is deterministic under the approved setup, "
            "so repeated-seed diffusion uncertainty does not apply."
        ),
    },
    {
        "population_id": "lama_deterministic",
        "seed_count": 0,
        "applicability_status": "not_applicable_deterministic",
        "source_notebook_ids": ["10", "18"],
        "interpretation": (
            "LaMa is deterministic under the approved setup, so "
            "repeated-seed diffusion uncertainty does not apply."
        ),
    },
    {
        "population_id": "sdxl_single_seed_subset",
        "seed_count": 1,
        "applicability_status": (
            "not_applicable_insufficient_seed_coverage"
        ),
        "source_notebook_ids": ["12", "18"],
        "interpretation": (
            "SDXL has one completed seed per bounded case, which is "
            "insufficient for empirical repeated-seed uncertainty."
        ),
    },
]

for boundary in UNCERTAINTY_BOUNDARIES:
    UNCERTAINTY_ROWS.append(
        {
            "uncertainty_row_id": stable_id(
                "uncertainty",
                "applicability",
                boundary["population_id"],
            ),
            "page_id": "robustness_uncertainty",
            "section_id": "uncertainty_applicability",
            "display_order": 0,
            "record_type": "applicability_boundary",
            "population_id": boundary["population_id"],
            "experiment_id": "",
            "damage_or_degradation_type": "",
            "prompt_variant_id": "",
            "uncertainty_group_id": "",
            "painting_id": "",
            "category": "",
            "metric_family": "",
            "metric_name": "",
            "region_id": "",
            "summary_statistic": "",
            "value": float("nan"),
            "value_unit": "not_applicable",
            "seed_count": boundary["seed_count"],
            "group_count": 0,
            "case_count": 0,
            "applicability_status": boundary[
                "applicability_status"
            ],
            "is_calibrated_confidence": False,
            "source_notebook_ids_json": json_list(
                boundary["source_notebook_ids"]
            ),
            "source_paths_json": json_list(
                [
                    SETTINGS["inputs"]["model_cards_path"],
                    SETTINGS["inputs"][
                        "canonical_uncertainty_path"
                    ],
                ]
            ),
            "interpretation": boundary["interpretation"],
            "limitation": (
                "No numeric uncertainty value is assigned where "
                "repeated stochastic evidence is unavailable."
            ),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

UNCERTAINTY_SUMMARY = pd.DataFrame(UNCERTAINTY_ROWS)

UNCERTAINTY_SUMMARY = (
    UNCERTAINTY_SUMMARY
    .sort_values(
        [
            "section_id",
            "population_id",
            "painting_id",
            "uncertainty_group_id",
            "metric_family",
            "metric_name",
            "region_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

UNCERTAINTY_SUMMARY["display_order"] = (
    UNCERTAINTY_SUMMARY.index + 1
)

UNCERTAINTY_SUMMARY = validate_output_frame(
    UNCERTAINTY_SUMMARY,
    "uncertainty_summary",
)

UNCERTAINTY_COVERAGE_VIEW = (
    UNCERTAINTY_SUMMARY.groupby(
        [
            "record_type",
            "population_id",
            "applicability_status",
        ],
        dropna=False,
        sort=True,
    )
    .agg(
        rows=("uncertainty_row_id", "size"),
        groups=("uncertainty_group_id", "nunique"),
        paintings=("painting_id", "nunique"),
        metrics=("metric_name", "nunique"),
        regions=("region_id", "nunique"),
    )
    .reset_index()
)

display(UNCERTAINTY_COVERAGE_VIEW)

print("Uncertainty-summary rows:", len(UNCERTAINTY_SUMMARY))

,record_type,population_id,applicability_status,rows,groups,paintings,metrics,regions
0,applicability_boundary,lama_deterministic,not_applicable_deterministic,1,1,1,1,1
1,applicability_boundary,opencv_telea_deterministic,not_applicable_deterministic,1,1,1,1,1
2,applicability_boundary,sdxl_single_seed_subset,not_applicable_insufficient_seed_coverage,1,1,1,1,1
3,canonical_group_summary,canonical_repeated_seed,supported_complete_seed_coverage,1300,130,50,7,3
4,damage_size_group_summary,damage_size_repeated_seed,supported_complete_seed_coverage,420,35,5,2,6


Uncertainty-summary rows: 1723


In [29]:
EXPECTED_SENSITIVITY_COUNTS = {
    "damage_size_sensitivity": 1901,
    "mask_robustness": 5373,
    "synthetic_degradation": 4695,
}

OBSERVED_SENSITIVITY_COUNTS = (
    SENSITIVITY_SUMMARY["analysis_family"]
    .value_counts()
    .to_dict()
)

EXPECTED_UNCERTAINTY_RECORD_COUNTS = {
    "canonical_group_summary": 1300,
    "damage_size_group_summary": 420,
    "applicability_boundary": 3,
}

OBSERVED_UNCERTAINTY_RECORD_COUNTS = (
    UNCERTAINTY_SUMMARY["record_type"]
    .value_counts()
    .to_dict()
)

MEASURED_UNCERTAINTY = UNCERTAINTY_SUMMARY.loc[
    UNCERTAINTY_SUMMARY["record_type"].ne(
        "applicability_boundary"
    )
]

BOUNDARY_UNCERTAINTY = UNCERTAINTY_SUMMARY.loc[
    UNCERTAINTY_SUMMARY["record_type"].eq(
        "applicability_boundary"
    )
]

BATCH_6_CHECKS = (
    (
        "sensitivity_row_count",
        "All upstream sensitivity results are retained",
        11969,
        len(SENSITIVITY_SUMMARY),
        len(SENSITIVITY_SUMMARY) == 11969,
    ),
    (
        "sensitivity_family_counts",
        "Every sensitivity family has complete row coverage",
        EXPECTED_SENSITIVITY_COUNTS,
        OBSERVED_SENSITIVITY_COUNTS,
        (
            OBSERVED_SENSITIVITY_COUNTS
            == EXPECTED_SENSITIVITY_COUNTS
        ),
    ),
    (
        "sensitivity_identifier_uniqueness",
        "Sensitivity identifiers are unique",
        len(SENSITIVITY_SUMMARY),
        SENSITIVITY_SUMMARY[
            "sensitivity_row_id"
        ].nunique(),
        (
            SENSITIVITY_SUMMARY[
                "sensitivity_row_id"
            ].nunique()
            == len(SENSITIVITY_SUMMARY)
        ),
    ),
    (
        "sensitivity_estimates_finite",
        "Every retained sensitivity result has an estimate",
        True,
        bool(
            pd.to_numeric(
                SENSITIVITY_SUMMARY["estimate"],
                errors="coerce",
            ).notna().all()
        ),
        bool(
            pd.to_numeric(
                SENSITIVITY_SUMMARY["estimate"],
                errors="coerce",
            ).notna().all()
        ),
    ),
    (
        "uncertainty_row_count",
        "The uncertainty table contains 1,723 rows",
        1723,
        len(UNCERTAINTY_SUMMARY),
        len(UNCERTAINTY_SUMMARY) == 1723,
    ),
    (
        "uncertainty_record_counts",
        "Measured and applicability uncertainty records are complete",
        EXPECTED_UNCERTAINTY_RECORD_COUNTS,
        OBSERVED_UNCERTAINTY_RECORD_COUNTS,
        (
            OBSERVED_UNCERTAINTY_RECORD_COUNTS
            == EXPECTED_UNCERTAINTY_RECORD_COUNTS
        ),
    ),
    (
        "canonical_uncertainty_groups",
        "All 130 canonical uncertainty groups are represented",
        130,
        UNCERTAINTY_SUMMARY.loc[
            UNCERTAINTY_SUMMARY["record_type"].eq(
                "canonical_group_summary"
            ),
            "uncertainty_group_id",
        ].nunique(),
        (
            UNCERTAINTY_SUMMARY.loc[
                UNCERTAINTY_SUMMARY["record_type"].eq(
                    "canonical_group_summary"
                ),
                "uncertainty_group_id",
            ].nunique()
            == 130
        ),
    ),
    (
        "damage_size_uncertainty_groups",
        "All 35 damage-size uncertainty groups are represented",
        35,
        UNCERTAINTY_SUMMARY.loc[
            UNCERTAINTY_SUMMARY["record_type"].eq(
                "damage_size_group_summary"
            ),
            "uncertainty_group_id",
        ].nunique(),
        (
            UNCERTAINTY_SUMMARY.loc[
                UNCERTAINTY_SUMMARY["record_type"].eq(
                    "damage_size_group_summary"
                ),
                "uncertainty_group_id",
            ].nunique()
            == 35
        ),
    ),
    (
        "measured_uncertainty_finite",
        "Every measured uncertainty value is finite",
        True,
        bool(
            pd.to_numeric(
                MEASURED_UNCERTAINTY["value"],
                errors="coerce",
            ).notna().all()
        ),
        bool(
            pd.to_numeric(
                MEASURED_UNCERTAINTY["value"],
                errors="coerce",
            ).notna().all()
        ),
    ),
    (
        "boundary_values_empty",
        "Non-applicable populations have no artificial values",
        3,
        int(BOUNDARY_UNCERTAINTY["value"].isna().sum()),
        int(BOUNDARY_UNCERTAINTY["value"].isna().sum()) == 3,
    ),
    (
        "uncertainty_not_confidence",
        "No uncertainty value is labelled calibrated confidence",
        {False},
        set(
            UNCERTAINTY_SUMMARY[
                "is_calibrated_confidence"
            ]
        ),
        (
            set(
                UNCERTAINTY_SUMMARY[
                    "is_calibrated_confidence"
                ]
            )
            == {False}
        ),
    ),
    (
        "sensitivity_sources_relative",
        "Sensitivity source paths are repository-relative",
        True,
        all_source_paths_are_relative(
            SENSITIVITY_SUMMARY["source_paths_json"]
        ),
        all_source_paths_are_relative(
            SENSITIVITY_SUMMARY["source_paths_json"]
        ),
    ),
    (
        "uncertainty_sources_relative",
        "Uncertainty source paths are repository-relative",
        True,
        all_source_paths_are_relative(
            UNCERTAINTY_SUMMARY["source_paths_json"]
        ),
        all_source_paths_are_relative(
            UNCERTAINTY_SUMMARY["source_paths_json"]
        ),
    ),
    (
        "batch_6_status",
        "All Batch 6 records have valid status",
        {"ok"},
        (
            set(SENSITIVITY_SUMMARY["status"])
            | set(UNCERTAINTY_SUMMARY["status"])
        ),
        (
            set(SENSITIVITY_SUMMARY["status"])
            | set(UNCERTAINTY_SUMMARY["status"])
            == {"ok"}
        ),
    ),
    (
        "no_canonical_files_written",
        "Batch 6 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_6_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_6_robustness_uncertainty",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Robustness or uncertainty evidence differs from its approved source."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_6_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_6_robustness_uncertainty"
        )
    ]
    .reset_index(drop=True)
)

display(SENSITIVITY_INFERENCE_SUMMARY)
display(UNCERTAINTY_COVERAGE_VIEW)
display(
    BATCH_6_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 6 completed.")
print("Sensitivity rows:", len(SENSITIVITY_SUMMARY))
print("Uncertainty rows:", len(UNCERTAINTY_SUMMARY))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,analysis_family,rows,estimates,interval_rows,effect_size_rows,p_value_rows,q_value_rows
0,damage_size_sensitivity,1901,1901,929,667,646,646
1,mask_robustness,5373,5373,573,669,429,429
2,synthetic_degradation,4695,4695,939,801,627,627


,record_type,population_id,applicability_status,rows,groups,paintings,metrics,regions
0,applicability_boundary,lama_deterministic,not_applicable_deterministic,1,1,1,1,1
1,applicability_boundary,opencv_telea_deterministic,not_applicable_deterministic,1,1,1,1,1
2,applicability_boundary,sdxl_single_seed_subset,not_applicable_insufficient_seed_coverage,1,1,1,1,1
3,canonical_group_summary,canonical_repeated_seed,supported_complete_seed_coverage,1300,130,50,7,3
4,damage_size_group_summary,damage_size_repeated_seed,supported_complete_seed_coverage,420,35,5,2,6


,check_id,severity,passed,observed
0,sensitivity_row_count,blocking,True,11969
1,sensitivity_family_counts,blocking,True,"{""damage_size_sensitivity"": 1901, ""mask_robust..."
2,sensitivity_identifier_uniqueness,blocking,True,11969
3,sensitivity_estimates_finite,blocking,True,True
4,uncertainty_row_count,blocking,True,1723
5,uncertainty_record_counts,blocking,True,"{""applicability_boundary"": 3, ""canonical_group..."
6,canonical_uncertainty_groups,blocking,True,130
7,damage_size_uncertainty_groups,blocking,True,35
8,measured_uncertainty_finite,blocking,True,True
9,boundary_values_empty,blocking,True,3


Batch 6 completed.
Sensitivity rows: 11969
Uncertainty rows: 1723
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 7 — Trustworthiness, XAI support, model cards, and compute

This batch prepares compact dashboard summaries for:

- the 14-category failure taxonomy;
- candidate recommendation outcomes;
- diagnostic flag states;
- failure-category assignments;
- metric, region, threshold, and aggregation ablations;
- flag-stability evidence;
- CLIP and DINOv2 retrieval lanes;
- model roles, strengths, weaknesses, runtime, storage, and scaling evidence.

Complete candidate-level flags and evidence paths remain available through the case index. These summaries support navigation and interpretation without replacing the underlying 65,000-plus diagnostic records.

Computational flags are conservative review aids. They are not expert annotations, calibrated risk scores, or conservation decisions.

In [30]:
FAILURE_TAXONOMY_SOURCE = (
    INPUT_TABLES["failure_taxonomy_path"]
    .copy()
    .reset_index(drop=True)
)

FAILURE_ASSIGNMENTS_SOURCE = (
    INPUT_TABLES["failure_assignments_path"]
    .copy()
    .reset_index(drop=True)
)

TRUSTWORTHINESS_FLAGS_SOURCE = (
    INPUT_TABLES["trustworthiness_flags_path"]
    .copy()
    .reset_index(drop=True)
)

ABLATION_RESULTS_SOURCE = (
    INPUT_TABLES["ablation_results_path"]
    .copy()
    .reset_index(drop=True)
)

FLAG_STABILITY_SOURCE = (
    INPUT_TABLES["flag_stability_path"]
    .copy()
    .reset_index(drop=True)
)

CASE_NEIGHBORS_SOURCE = (
    INPUT_TABLES["case_neighbors_path"]
    .copy()
    .reset_index(drop=True)
)


def union_json_lists(values: pd.Series) -> str:
    """Return the sorted union of JSON-list values."""

    return json_list(
        sorted(
            {
                item
                for value in values
                for item in parse_json_list(value)
            }
        )
    )


RECOMMENDATION_ACTIONS = {
    "do_not_rely_automatically": (
        "Do not rely on this candidate automatically; inspect the "
        "restoration and its diagnostic evidence."
    ),
    "specialist_review_required": (
        "Require specialist review before drawing a restoration conclusion."
    ),
    "suitable_for_preliminary_inspection": (
        "Use as a preliminary inspection candidate while retaining "
        "the displayed limitations."
    ),
    "unstable_candidate": (
        "Inspect all repeated seeds together and do not select one "
        "apparently attractive output in isolation."
    ),
}

print("Failure categories:", len(FAILURE_TAXONOMY_SOURCE))
print("Failure assignments:", len(FAILURE_ASSIGNMENTS_SOURCE))
print("Trustworthiness flags:", len(TRUSTWORTHINESS_FLAGS_SOURCE))
print("Ablation results:", len(ABLATION_RESULTS_SOURCE))
print("Flag-stability rows:", len(FLAG_STABILITY_SOURCE))
print("Retrieval-neighbour rows:", len(CASE_NEIGHBORS_SOURCE))

Failure categories: 14
Failure assignments: 24990
Trustworthiness flags: 19635
Ablation results: 7710
Flag-stability rows: 41055
Retrieval-neighbour rows: 100


In [31]:
TRUST_ROWS = []


def append_trust_row(
    *,
    section_id: str,
    record_type: str,
    entity_id: str,
    display_name: str,
    value: object,
    value_unit: str,
    candidate_count: int,
    case_count: int,
    painting_count: int,
    source_notebook_ids: list[str],
    source_paths: list[str],
    interpretation: str,
    limitation: str,
    model_id: str = "",
    experiment_id: str = "",
    category: str = "",
    damage_or_degradation_type: str = "",
    recommendation_category: str = "",
    flag_id: str = "",
    flag_status: str = "",
    failure_category_id: str = "",
    failure_status: str = "",
    affected_regions_json: str = "[]",
    recommended_action: str = "",
) -> None:
    """Append one exact-schema trustworthiness summary row."""

    TRUST_ROWS.append(
        {
            "trust_row_id": stable_id(
                "trust",
                section_id,
                record_type,
                entity_id,
                model_id,
                flag_status,
                failure_status,
            ),
            "page_id": "trustworthiness_xai",
            "section_id": section_id,
            "display_order": 0,
            "record_type": record_type,
            "entity_id": entity_id,
            "display_name": display_name,
            "model_id": model_id,
            "experiment_id": experiment_id,
            "category": category,
            "damage_or_degradation_type": (
                damage_or_degradation_type
            ),
            "recommendation_category": (
                recommendation_category
            ),
            "flag_id": flag_id,
            "flag_status": flag_status,
            "failure_category_id": failure_category_id,
            "failure_status": failure_status,
            "value": value,
            "value_unit": value_unit,
            "candidate_count": int(candidate_count),
            "case_count": int(case_count),
            "painting_count": int(painting_count),
            "affected_regions_json": affected_regions_json,
            "recommended_action": recommended_action,
            "source_notebook_ids_json": json_list(
                source_notebook_ids
            ),
            "source_paths_json": json_list(source_paths),
            "interpretation": interpretation,
            "limitation": limitation,
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )


for row in FAILURE_TAXONOMY_SOURCE.itertuples(index=False):
    append_trust_row(
        section_id="failure_taxonomy",
        record_type="failure_category_definition",
        entity_id=str(row.category_id),
        display_name=str(row.display_name),
        value=1,
        value_unit="defined proxy category",
        candidate_count=0,
        case_count=0,
        painting_count=0,
        source_notebook_ids=["27"],
        source_paths=[
            SETTINGS["inputs"]["failure_taxonomy_path"]
        ],
        interpretation=str(row.definition),
        limitation=str(row.limitations),
        failure_category_id=str(row.category_id),
        failure_status="definition",
        affected_regions_json=str(row.affected_regions_json),
        recommended_action=str(row.recommended_action),
    )


for recommendation, group in CASE_INDEX.groupby(
    "recommendation_category",
    sort=True,
):
    append_trust_row(
        section_id="candidate_recommendations",
        record_type="recommendation_summary",
        entity_id=f"overall::{recommendation}",
        display_name=readable_identifier(recommendation),
        value=len(group),
        value_unit="candidates",
        candidate_count=len(group),
        case_count=group["case_id"].nunique(),
        painting_count=group["painting_id"].nunique(),
        source_notebook_ids=["27", "29"],
        source_paths=[
            SETTINGS["inputs"]["trustworthiness_flags_path"],
            SETTINGS["inputs"]["explanation_cases_path"],
        ],
        interpretation=(
            f"{len(group):,} approved candidates receive the "
            f"{readable_identifier(recommendation).lower()} outcome."
        ),
        limitation=(
            "Recommendation outcomes are conservative computational "
            "guidance, not expert ground truth."
        ),
        recommendation_category=str(recommendation),
        affected_regions_json=union_json_lists(
            group["affected_regions_json"]
        ),
        recommended_action=RECOMMENDATION_ACTIONS[
            str(recommendation)
        ],
    )


for (model_id, recommendation), group in CASE_INDEX.groupby(
    ["model_id", "recommendation_category"],
    sort=True,
):
    append_trust_row(
        section_id="candidate_recommendations_by_model",
        record_type="model_recommendation_summary",
        entity_id=f"{model_id}::{recommendation}",
        display_name=(
            f"{MODEL_DISPLAY_NAMES.get(model_id, model_id)} — "
            f"{readable_identifier(recommendation)}"
        ),
        value=len(group),
        value_unit="candidates",
        candidate_count=len(group),
        case_count=group["case_id"].nunique(),
        painting_count=group["painting_id"].nunique(),
        source_notebook_ids=["27", "29"],
        source_paths=[
            SETTINGS["inputs"]["trustworthiness_flags_path"],
            SETTINGS["inputs"]["explanation_cases_path"],
        ],
        interpretation=(
            f"{len(group):,} {MODEL_DISPLAY_NAMES.get(model_id, model_id)} "
            f"candidates receive this review outcome."
        ),
        limitation=(
            "Counts depend on model coverage and must not be compared "
            "without considering each model's candidate population."
        ),
        model_id=str(model_id),
        recommendation_category=str(recommendation),
        affected_regions_json=union_json_lists(
            group["affected_regions_json"]
        ),
        recommended_action=RECOMMENDATION_ACTIONS[
            str(recommendation)
        ],
    )

print("Taxonomy and recommendation rows:", len(TRUST_ROWS))

Taxonomy and recommendation rows: 30


In [32]:
for (
    model_id,
    flag_id,
    flag_status,
), group in TRUSTWORTHINESS_FLAGS_SOURCE.groupby(
    ["model_id", "flag_id", "flag_status"],
    sort=True,
):
    append_trust_row(
        section_id="diagnostic_flags",
        record_type="flag_status_summary",
        entity_id=f"{model_id}::{flag_id}",
        display_name=str(group["flag_name"].iloc[0]),
        value=len(group),
        value_unit="flag assignments",
        candidate_count=group["candidate_id"].nunique(),
        case_count=group["case_id"].nunique(),
        painting_count=group["painting_id"].nunique(),
        source_notebook_ids=["27"],
        source_paths=[
            SETTINGS["inputs"]["trustworthiness_flags_path"]
        ],
        interpretation=(
            f"{len(group):,} candidate-level assignments for "
            f"{MODEL_DISPLAY_NAMES.get(model_id, model_id)} have "
            f"status {readable_identifier(flag_status).lower()}."
        ),
        limitation=(
            "A triggered flag identifies evidence requiring review; "
            "it does not prove restoration failure."
        ),
        model_id=str(model_id),
        recommendation_category=blank_safe_text(
            group["recommendation_category"].iloc[0]
        ),
        flag_id=str(flag_id),
        flag_status=str(flag_status),
        affected_regions_json=union_json_lists(
            group["affected_regions_json"]
        ),
        recommended_action=blank_safe_text(
            group["recommended_action"].iloc[0]
        ),
    )


for (
    model_id,
    category_id,
    assignment_status,
), group in FAILURE_ASSIGNMENTS_SOURCE.groupby(
    ["model_id", "category_id", "assignment_status"],
    sort=True,
):
    append_trust_row(
        section_id="failure_assignments",
        record_type="failure_assignment_summary",
        entity_id=f"{model_id}::{category_id}",
        display_name=str(group["category_name"].iloc[0]),
        value=len(group),
        value_unit="category assignments",
        candidate_count=group["candidate_id"].nunique(),
        case_count=group["case_id"].nunique(),
        painting_count=group["painting_id"].nunique(),
        source_notebook_ids=["27"],
        source_paths=[
            SETTINGS["inputs"]["failure_assignments_path"]
        ],
        interpretation=(
            f"{len(group):,} assignments for "
            f"{MODEL_DISPLAY_NAMES.get(model_id, model_id)} have "
            f"status {readable_identifier(assignment_status).lower()}."
        ),
        limitation=(
            "Failure categories are transparent metric-derived "
            "proxies and are not expert-labelled defects."
        ),
        model_id=str(model_id),
        failure_category_id=str(category_id),
        failure_status=str(assignment_status),
        affected_regions_json=union_json_lists(
            group["affected_regions_json"]
        ),
        recommended_action=blank_safe_text(
            group["recommended_action"].iloc[0]
        ),
    )

print("Rows after flag and failure summaries:", len(TRUST_ROWS))

Rows after flag and failure summaries: 243


In [33]:
ABLATION_SCENARIO_SUMMARIES = (
    ABLATION_RESULTS_SOURCE.loc[
        ABLATION_RESULTS_SOURCE["result_kind"].eq(
            "scenario_summary"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

for row in ABLATION_SCENARIO_SUMMARIES.itertuples(
    index=False
):
    append_trust_row(
        section_id="framework_ablation",
        record_type="ablation_scenario_summary",
        entity_id=str(row.scenario_id),
        display_name=readable_identifier(row.scenario_id),
        value=float(row.changed_fraction),
        value_unit="changed_candidate_fraction",
        candidate_count=int(float(row.n_candidates)),
        case_count=int(float(row.n_cases)),
        painting_count=int(float(row.n_paintings)),
        source_notebook_ids=["28"],
        source_paths=[
            SETTINGS["inputs"]["ablation_results_path"]
        ],
        interpretation=(
            f"{100.0 * float(row.changed_fraction):.1f}% of candidates "
            f"change flag state under this "
            f"{readable_identifier(row.scenario_family).lower()} "
            "ablation scenario."
        ),
        limitation=(
            "Ablation sensitivity shows dependence on evaluation "
            "choices; it is not a restoration-quality score."
        ),
        failure_status=str(row.applicability_status),
        recommended_action=(
            "Inspect whether the conclusion remains stable when "
            "metric, region, threshold, or aggregation choices change."
        ),
    )


for (
    scenario_family,
    model_id,
), group in FLAG_STABILITY_SOURCE.groupby(
    ["scenario_family", "model_id"],
    sort=True,
):
    mean_agreement = float(
        pd.to_numeric(
            group["flag_state_agreement_fraction"],
            errors="raise",
        ).mean()
    )

    append_trust_row(
        section_id="flag_stability",
        record_type="flag_stability_summary",
        entity_id=f"{scenario_family}::{model_id}",
        display_name=(
            f"{MODEL_DISPLAY_NAMES.get(model_id, model_id)} — "
            f"{readable_identifier(scenario_family)}"
        ),
        value=mean_agreement,
        value_unit="mean_flag_state_agreement_fraction",
        candidate_count=group["candidate_id"].nunique(),
        case_count=group["case_id"].nunique(),
        painting_count=group["painting_id"].nunique(),
        source_notebook_ids=["28"],
        source_paths=[
            SETTINGS["inputs"]["flag_stability_path"]
        ],
        interpretation=(
            f"Mean candidate-level flag-state agreement is "
            f"{100.0 * mean_agreement:.1f}% across "
            f"{readable_identifier(scenario_family).lower()} "
            f"scenarios for "
            f"{MODEL_DISPLAY_NAMES.get(model_id, model_id)}."
        ),
        limitation=(
            "High agreement means the computational flag state is "
            "stable under tested policies, not that it is correct."
        ),
        model_id=str(model_id),
        failure_status="policy_sensitivity_summary",
        recommended_action=(
            "Review candidates whose flag states change across "
            "reasonable evaluation policies."
        ),
    )


for lane, group in CASE_NEIGHBORS_SOURCE.groupby(
    "lane",
    sort=True,
):
    mean_similarity = float(
        pd.to_numeric(
            group["cosine_similarity"],
            errors="raise",
        ).mean()
    )

    append_trust_row(
        section_id="case_retrieval",
        record_type="retrieval_lane_summary",
        entity_id=str(lane),
        display_name=f"{readable_identifier(lane)} retrieval",
        value=mean_similarity,
        value_unit="mean_cosine_similarity",
        candidate_count=len(group),
        case_count=group["query_case_id"].nunique(),
        painting_count=group["query_painting_id"].nunique(),
        source_notebook_ids=["29"],
        source_paths=[
            SETTINGS["inputs"]["case_neighbors_path"]
        ],
        interpretation=(
            f"The {readable_identifier(lane).lower()} lane contains "
            f"{len(group)} ranked neighbours across CLIP and DINOv2 "
            "retrieval evidence."
        ),
        limitation=(
            "Retrieval similarity supports comparison and explanation; "
            "it does not establish restoration correctness."
        ),
        failure_status="retrieval_evidence",
        recommended_action=(
            "Compare the query with retrieved neighbours and inspect "
            "whether similar features correspond to similar failures."
        ),
    )

print("Rows after ablation and retrieval summaries:", len(TRUST_ROWS))

Rows after ablation and retrieval summaries: 284


In [34]:
TRUSTWORTHINESS_SUMMARY = pd.DataFrame(TRUST_ROWS)

TRUST_SECTION_ORDER = {
    "candidate_recommendations": 1,
    "candidate_recommendations_by_model": 2,
    "diagnostic_flags": 3,
    "failure_assignments": 4,
    "failure_taxonomy": 5,
    "framework_ablation": 6,
    "flag_stability": 7,
    "case_retrieval": 8,
}

TRUSTWORTHINESS_SUMMARY["_section_order"] = (
    TRUSTWORTHINESS_SUMMARY["section_id"].map(
        TRUST_SECTION_ORDER
    )
)

TRUSTWORTHINESS_SUMMARY = (
    TRUSTWORTHINESS_SUMMARY
    .sort_values(
        [
            "_section_order",
            "model_id",
            "display_name",
            "flag_status",
            "failure_status",
        ],
        kind="stable",
    )
    .drop(columns="_section_order")
    .reset_index(drop=True)
)

TRUSTWORTHINESS_SUMMARY["display_order"] = (
    TRUSTWORTHINESS_SUMMARY.index + 1
)

TRUSTWORTHINESS_SUMMARY = validate_output_frame(
    TRUSTWORTHINESS_SUMMARY,
    "trustworthiness_summary",
)

TRUSTWORTHINESS_COVERAGE_VIEW = (
    TRUSTWORTHINESS_SUMMARY.groupby(
        ["section_id", "record_type"],
        sort=False,
    )
    .agg(
        rows=("trust_row_id", "size"),
        represented_candidates=("candidate_count", "max"),
        represented_cases=("case_count", "max"),
        represented_paintings=("painting_count", "max"),
    )
    .reset_index()
)

display(TRUSTWORTHINESS_COVERAGE_VIEW)

print(
    "Trustworthiness-summary rows:",
    len(TRUSTWORTHINESS_SUMMARY),
)

,section_id,record_type,rows,represented_candidates,represented_cases,represented_paintings
0,candidate_recommendations,recommendation_summary,4,1376,360,50
1,candidate_recommendations_by_model,model_recommendation_summary,12,794,333,50
2,diagnostic_flags,flag_status_summary,88,954,410,50
3,failure_assignments,failure_assignment_summary,125,850,410,50
4,failure_taxonomy,failure_category_definition,14,0,0,0
5,framework_ablation,ablation_scenario_summary,23,1785,410,50
6,flag_stability,flag_stability_summary,16,955,410,50
7,case_retrieval,retrieval_lane_summary,2,50,10,10


Trustworthiness-summary rows: 284


In [36]:
COMPUTE_SOURCE = (
    INPUT_TABLES["compute_scalability_path"]
    .copy()
    .reset_index(drop=True)
)

MODEL_CARD_LOOKUP = (
    MODEL_DIMENSION
    .set_index("model_id", verify_integrity=True)
)

COMPUTE_ROWS = []

for row in COMPUTE_SOURCE.itertuples(index=False):
    model_id = str(row.model_id)
    card = MODEL_CARD_LOOKUP.loc[model_id]

    is_executed = as_bool(row.is_executed)
    is_projected = as_bool(row.is_projected)

    runtime_seconds = (
        optional_float(row.total_runtime_seconds)
        if is_executed
        else optional_float(row.runtime_central_seconds)
    )

    output_file_count = (
        optional_float(row.output_file_count)
        if is_executed
        else optional_float(row.projected_output_file_count)
    )

    output_storage_bytes = (
        optional_float(row.output_storage_bytes)
        if is_executed
        else optional_float(
            row.projected_output_storage_bytes
        )
    )

    COMPUTE_ROWS.append(
        {
            "compute_row_id": stable_id(
                "compute",
                row.compute_row_id,
            ),
            "page_id": "model_performance",
            "section_id": (
                "executed_compute"
                if is_executed
                else "scaling_projection"
            ),
            "display_order": 0,
            "record_type": str(row.record_type),
            "model_id": model_id,
            "display_name": str(card["display_name"]),
            "evaluation_status": str(
                card["evaluation_status"]
            ),
            "scenario_id": blank_safe_text(
                row.scenario_id
            ),
            "experiment_id": blank_safe_text(
                row.experiment_id
            ),
            "painting_count": optional_float(
                row.painting_count
            ),
            "case_count": optional_float(
                row.case_count
            ),
            "candidate_count": optional_float(
                row.candidate_count
            ),
            "inference_count": optional_float(
                row.inference_count
            ),
            "runtime_seconds": runtime_seconds,
            "runtime_lower_seconds": optional_float(
                row.runtime_lower_seconds
            ),
            "runtime_upper_seconds": optional_float(
                row.runtime_upper_seconds
            ),
            "throughput_candidates_per_second": optional_float(
                row.throughput_candidates_per_second
            ),
            "gpu_peak_memory_bytes": optional_float(
                row.gpu_peak_memory_bytes
            ),
            "output_file_count": output_file_count,
            "output_storage_bytes": output_storage_bytes,
            "is_executed": is_executed,
            "is_projected": is_projected,
            "projection_basis": blank_safe_text(
                row.projection_basis
            ),
            "applicability_status": str(
                row.applicability_status
            ),
            "strengths_json": str(card["strengths_json"]),
            "weaknesses_json": str(card["weaknesses_json"]),
            "limitations_json": str(
                card["known_limitations_json"]
            ),
            "source_notebook_ids_json": json_list(["30"]),
            "source_paths_json": json_list(
                [
                    SETTINGS["inputs"][
                        "compute_scalability_path"
                    ],
                    SETTINGS["inputs"]["model_cards_path"],
                ]
            ),
            "interpretation": (
                "Observed runtime and storage from the completed "
                "workstation execution."
                if is_executed
                else (
                    "Linear planning projection derived from observed "
                    "workstation evidence; this row was not executed."
                    if is_projected
                    else
                    "No defensible full-design projection is available "
                    "for this model and scenario."
                )
            ),
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

COMPUTE_SUMMARY = pd.DataFrame(COMPUTE_ROWS)

COMPUTE_SUMMARY = (
    COMPUTE_SUMMARY
    .sort_values(
        [
            "section_id",
            "model_id",
            "scenario_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

COMPUTE_SUMMARY["display_order"] = (
    COMPUTE_SUMMARY.index + 1
)

COMPUTE_SUMMARY = validate_output_frame(
    COMPUTE_SUMMARY,
    "compute_summary",
)

COMPUTE_COVERAGE_VIEW = (
    COMPUTE_SUMMARY.groupby(
        [
            "model_id",
            "record_type",
            "applicability_status",
        ],
        sort=True,
    )
    .agg(
        rows=("compute_row_id", "size"),
        maximum_candidates=("candidate_count", "max"),
        maximum_runtime_seconds=("runtime_seconds", "max"),
        maximum_storage_bytes=("output_storage_bytes", "max"),
    )
    .reset_index()
)

display(COMPUTE_COVERAGE_VIEW)

print("Compute-summary rows:", len(COMPUTE_SUMMARY))
print(
    "Rows with recorded painting counts:",
    int(COMPUTE_SUMMARY["painting_count"].notna().sum()),
)
print(
    "Rows with recorded runtime:",
    int(COMPUTE_SUMMARY["runtime_seconds"].notna().sum()),
)
print(
    "Rows with recorded storage:",
    int(COMPUTE_SUMMARY["output_storage_bytes"].notna().sum()),
)

,model_id,record_type,applicability_status,rows,maximum_candidates,maximum_runtime_seconds,maximum_storage_bytes
0,lama,observed,applicable_executed,5,410.0,655.093108,2.985141e+08
1,lama,projection,applicable_projection,2,2460.0,3930.558647,1.755531e+09
2,opencv_telea,observed,applicable_executed,5,410.0,185.613548,3.014544e+08
3,opencv_telea,projection,applicable_projection,2,2460.0,1113.681287,1.774485e+09
4,sdxl_inpainting,observed,applicable_executed,3,10.0,3784.687000,7.195725e+06
5,sdxl_inpainting,projection,applicable_projection,1,1500.0,479861.100000,1.063792e+09
6,sdxl_inpainting,projection,not_applicable_no_full_design_basis,1,NaN,NaN,NaN
7,stable_diffusion_inpainting,observed,applicable_executed,14,1330.0,12269.500548,9.709300e+08
8,stable_diffusion_inpainting,projection,applicable_projection,2,7980.0,73617.003286,5.731725e+09


Compute-summary rows: 35
Rows with recorded painting counts: 12
Rows with recorded runtime: 34
Rows with recorded storage: 11


In [37]:
EXPECTED_TRUST_RECORD_COUNTS = {
    "failure_category_definition": 14,
    "recommendation_summary": 4,
    "model_recommendation_summary": 12,
    "flag_status_summary": 88,
    "failure_assignment_summary": 125,
    "ablation_scenario_summary": 23,
    "flag_stability_summary": 16,
    "retrieval_lane_summary": 2,
}

OBSERVED_TRUST_RECORD_COUNTS = (
    TRUSTWORTHINESS_SUMMARY["record_type"]
    .value_counts()
    .to_dict()
)

OVERALL_RECOMMENDATION_ROWS = (
    TRUSTWORTHINESS_SUMMARY.loc[
        TRUSTWORTHINESS_SUMMARY["record_type"].eq(
            "recommendation_summary"
        )
    ]
)

FLAG_SUMMARY_ROWS = (
    TRUSTWORTHINESS_SUMMARY.loc[
        TRUSTWORTHINESS_SUMMARY["record_type"].eq(
            "flag_status_summary"
        )
    ]
)

FAILURE_SUMMARY_ROWS = (
    TRUSTWORTHINESS_SUMMARY.loc[
        TRUSTWORTHINESS_SUMMARY["record_type"].eq(
            "failure_assignment_summary"
        )
    ]
)

EXPECTED_COMPUTE_RECORD_COUNTS = {
    "observed": 27,
    "projection": 8,
}

OBSERVED_COMPUTE_RECORD_COUNTS = (
    COMPUTE_SUMMARY["record_type"]
    .value_counts()
    .to_dict()
)

BATCH_7_CHECKS = (
    (
        "trustworthiness_row_count",
        "The trustworthiness summary contains 284 rows",
        284,
        len(TRUSTWORTHINESS_SUMMARY),
        len(TRUSTWORTHINESS_SUMMARY) == 284,
    ),
    (
        "trust_record_counts",
        "Every trustworthiness record type has expected coverage",
        EXPECTED_TRUST_RECORD_COUNTS,
        OBSERVED_TRUST_RECORD_COUNTS,
        (
            OBSERVED_TRUST_RECORD_COUNTS
            == EXPECTED_TRUST_RECORD_COUNTS
        ),
    ),
    (
        "recommendation_population",
        "Overall recommendation counts cover every candidate once",
        len(CASE_INDEX),
        int(OVERALL_RECOMMENDATION_ROWS["value"].sum()),
        (
            int(OVERALL_RECOMMENDATION_ROWS["value"].sum())
            == len(CASE_INDEX)
        ),
    ),
    (
        "flag_assignment_population",
        "Flag summaries retain all source assignments",
        len(TRUSTWORTHINESS_FLAGS_SOURCE),
        int(FLAG_SUMMARY_ROWS["value"].sum()),
        (
            int(FLAG_SUMMARY_ROWS["value"].sum())
            == len(TRUSTWORTHINESS_FLAGS_SOURCE)
        ),
    ),
    (
        "failure_assignment_population",
        "Failure summaries retain all source assignments",
        len(FAILURE_ASSIGNMENTS_SOURCE),
        int(FAILURE_SUMMARY_ROWS["value"].sum()),
        (
            int(FAILURE_SUMMARY_ROWS["value"].sum())
            == len(FAILURE_ASSIGNMENTS_SOURCE)
        ),
    ),
    (
        "taxonomy_population",
        "All 14 failure-category definitions are retained",
        14,
        int(
            TRUSTWORTHINESS_SUMMARY["record_type"]
            .eq("failure_category_definition")
            .sum()
        ),
        int(
            TRUSTWORTHINESS_SUMMARY["record_type"]
            .eq("failure_category_definition")
            .sum()
        ) == 14,
    ),
    (
        "ablation_scenario_population",
        "All 23 scenario summaries are retained",
        23,
        int(
            TRUSTWORTHINESS_SUMMARY["record_type"]
            .eq("ablation_scenario_summary")
            .sum()
        ),
        int(
            TRUSTWORTHINESS_SUMMARY["record_type"]
            .eq("ablation_scenario_summary")
            .sum()
        ) == 23,
    ),
    (
        "retrieval_population",
        "Both retrieval lanes retain all 100 neighbours",
        100,
        int(
            TRUSTWORTHINESS_SUMMARY.loc[
                TRUSTWORTHINESS_SUMMARY["record_type"].eq(
                    "retrieval_lane_summary"
                ),
                "candidate_count",
            ].sum()
        ),
        (
            int(
                TRUSTWORTHINESS_SUMMARY.loc[
                    TRUSTWORTHINESS_SUMMARY[
                        "record_type"
                    ].eq("retrieval_lane_summary"),
                    "candidate_count",
                ].sum()
            )
            == 100
        ),
    ),
    (
        "trust_identifier_uniqueness",
        "Trustworthiness identifiers are unique",
        len(TRUSTWORTHINESS_SUMMARY),
        TRUSTWORTHINESS_SUMMARY["trust_row_id"].nunique(),
        (
            TRUSTWORTHINESS_SUMMARY[
                "trust_row_id"
            ].nunique()
            == len(TRUSTWORTHINESS_SUMMARY)
        ),
    ),
    (
        "compute_row_count",
        "All 35 compute and scaling rows are retained",
        35,
        len(COMPUTE_SUMMARY),
        len(COMPUTE_SUMMARY) == 35,
    ),
    (
        "compute_record_counts",
        "Observed and projected compute rows are separated",
        EXPECTED_COMPUTE_RECORD_COUNTS,
        OBSERVED_COMPUTE_RECORD_COUNTS,
        (
            OBSERVED_COMPUTE_RECORD_COUNTS
            == EXPECTED_COMPUTE_RECORD_COUNTS
        ),
    ),
    (
        "compute_model_population",
        "Compute evidence covers all four model identities",
        4,
        COMPUTE_SUMMARY["model_id"].nunique(),
        COMPUTE_SUMMARY["model_id"].nunique() == 4,
    ),
    (
        "compute_identifier_uniqueness",
        "Compute identifiers are unique",
        len(COMPUTE_SUMMARY),
        COMPUTE_SUMMARY["compute_row_id"].nunique(),
        (
            COMPUTE_SUMMARY["compute_row_id"].nunique()
            == len(COMPUTE_SUMMARY)
        ),
    ),
    (
        "trust_sources_relative",
        "Trustworthiness source paths are repository-relative",
        True,
        all_source_paths_are_relative(
            TRUSTWORTHINESS_SUMMARY["source_paths_json"]
        ),
        all_source_paths_are_relative(
            TRUSTWORTHINESS_SUMMARY["source_paths_json"]
        ),
    ),
    (
        "compute_sources_relative",
        "Compute source paths are repository-relative",
        True,
        all_source_paths_are_relative(
            COMPUTE_SUMMARY["source_paths_json"]
        ),
        all_source_paths_are_relative(
            COMPUTE_SUMMARY["source_paths_json"]
        ),
    ),
    (
        "batch_7_status",
        "All Batch 7 records have valid status",
        {"ok"},
        (
            set(TRUSTWORTHINESS_SUMMARY["status"])
            | set(COMPUTE_SUMMARY["status"])
        ),
        (
            set(TRUSTWORTHINESS_SUMMARY["status"])
            | set(COMPUTE_SUMMARY["status"])
            == {"ok"}
        ),
    ),
    (
        "no_canonical_files_written",
        "Batch 7 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_7_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_7_trustworthiness_compute",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Trustworthiness or compute evidence differs from its source."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_7_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_7_trustworthiness_compute"
        )
    ]
    .reset_index(drop=True)
)

display(TRUSTWORTHINESS_COVERAGE_VIEW)
display(COMPUTE_COVERAGE_VIEW)
display(
    BATCH_7_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 7 completed.")
print(
    "Trustworthiness rows:",
    len(TRUSTWORTHINESS_SUMMARY),
)
print("Compute rows:", len(COMPUTE_SUMMARY))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,section_id,record_type,rows,represented_candidates,represented_cases,represented_paintings
0,candidate_recommendations,recommendation_summary,4,1376,360,50
1,candidate_recommendations_by_model,model_recommendation_summary,12,794,333,50
2,diagnostic_flags,flag_status_summary,88,954,410,50
3,failure_assignments,failure_assignment_summary,125,850,410,50
4,failure_taxonomy,failure_category_definition,14,0,0,0
5,framework_ablation,ablation_scenario_summary,23,1785,410,50
6,flag_stability,flag_stability_summary,16,955,410,50
7,case_retrieval,retrieval_lane_summary,2,50,10,10


,model_id,record_type,applicability_status,rows,maximum_candidates,maximum_runtime_seconds,maximum_storage_bytes
0,lama,observed,applicable_executed,5,410.0,655.093108,2.985141e+08
1,lama,projection,applicable_projection,2,2460.0,3930.558647,1.755531e+09
2,opencv_telea,observed,applicable_executed,5,410.0,185.613548,3.014544e+08
3,opencv_telea,projection,applicable_projection,2,2460.0,1113.681287,1.774485e+09
4,sdxl_inpainting,observed,applicable_executed,3,10.0,3784.687000,7.195725e+06
5,sdxl_inpainting,projection,applicable_projection,1,1500.0,479861.100000,1.063792e+09
6,sdxl_inpainting,projection,not_applicable_no_full_design_basis,1,NaN,NaN,NaN
7,stable_diffusion_inpainting,observed,applicable_executed,14,1330.0,12269.500548,9.709300e+08
8,stable_diffusion_inpainting,projection,applicable_projection,2,7980.0,73617.003286,5.731725e+09


,check_id,severity,passed,observed
0,trustworthiness_row_count,blocking,True,284
1,trust_record_counts,blocking,True,"{""ablation_scenario_summary"": 23, ""failure_ass..."
2,recommendation_population,blocking,True,1785
3,flag_assignment_population,blocking,True,19635
4,failure_assignment_population,blocking,True,24990
5,taxonomy_population,blocking,True,14
6,ablation_scenario_population,blocking,True,23
7,retrieval_population,blocking,True,100
8,trust_identifier_uniqueness,blocking,True,284
9,compute_row_count,blocking,True,35


Batch 7 completed.
Trustworthiness rows: 284
Compute rows: 35
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 8 — Complete visual-asset index

This batch constructs the visual index consumed by the dashboard.

It retains every manifest record for:

- difference maps and spatial diagnostics;
- texture, colour, and seam maps;
- uncertainty maps and overlays;
- semantic and structural maps;
- damage-size uncertainty maps.

Numeric archive records remain separate because several records can reference different archive keys inside the same NPZ file.

The index also includes:

- all 76 validated representative restorations;
- all 18 thesis figures;
- all six publication figures.

Representative defaults control initial presentation only. They do not restrict access to the complete case, map, or figure population.

In [38]:
from PIL import Image

from restoration_eval.dashboard_assets import sha256_file


SPATIAL_MAPS_SOURCE = (
    INPUT_TABLES["spatial_map_manifest_path"]
    .copy()
    .reset_index(drop=True)
)

LOCAL_MAPS_SOURCE = (
    INPUT_TABLES["local_map_manifest_path"]
    .copy()
    .reset_index(drop=True)
)

UNCERTAINTY_MAPS_SOURCE = (
    INPUT_TABLES["uncertainty_map_manifest_path"]
    .copy()
    .reset_index(drop=True)
)

SEMANTIC_MAPS_SOURCE = (
    INPUT_TABLES["semantic_map_manifest_path"]
    .copy()
    .reset_index(drop=True)
)

DAMAGE_SIZE_MAPS_SOURCE = (
    INPUT_TABLES["damage_size_map_manifest_path"]
    .copy()
    .reset_index(drop=True)
)

REPRESENTATIVE_CASES_SOURCE = (
    INPUT_TABLES["representative_cases_path"]
    .copy()
    .reset_index(drop=True)
)

CASE_EXPERIMENT_LOOKUP = (
    CASE_INDEX[
        ["case_id", "experiment_id"]
    ]
    .drop_duplicates()
    .set_index("case_id", verify_integrity=True)[
        "experiment_id"
    ]
    .to_dict()
)


def image_file_metadata(relative_path: str) -> dict:
    """Read required metadata for an existing raster image."""

    full_path = PROJECT_ROOT / relative_path

    if not full_path.is_file():
        raise FileNotFoundError(full_path)

    with Image.open(full_path) as image:
        width, height = image.size
        image_format = image.format or full_path.suffix.lstrip(".")

    return {
        "sha256": sha256_file(full_path),
        "size_bytes": int(full_path.stat().st_size),
        "width": int(width),
        "height": int(height),
        "format": str(image_format).upper(),
    }


FINAL_THESIS_FIGURES = sorted(
    (
        PROJECT_ROOT
        / SETTINGS["inputs"]["final_thesis_figures_dir"]
    ).glob("*")
)

FINAL_PUBLICATION_FIGURES = sorted(
    (
        PROJECT_ROOT
        / SETTINGS["inputs"]["final_publication_figures_dir"]
    ).glob("*")
)

print("Spatial manifest rows:", len(SPATIAL_MAPS_SOURCE))
print("Local map rows:", len(LOCAL_MAPS_SOURCE))
print("Uncertainty map rows:", len(UNCERTAINTY_MAPS_SOURCE))
print("Semantic map rows:", len(SEMANTIC_MAPS_SOURCE))
print("Damage-size map rows:", len(DAMAGE_SIZE_MAPS_SOURCE))
print(
    "Representative restorations:",
    len(REPRESENTATIVE_CASES_SOURCE),
)
print("Thesis figures:", len(FINAL_THESIS_FIGURES))
print("Publication figures:", len(FINAL_PUBLICATION_FIGURES))

Spatial manifest rows: 10062
Local map rows: 3282
Uncertainty map rows: 1055
Semantic map rows: 9430
Damage-size map rows: 35
Representative restorations: 76
Thesis figures: 18
Publication figures: 6


In [44]:
VISUAL_ROWS = []


def add_manifest_visual(
    *,
    source_notebook_id: str,
    source_row_id: str,
    asset_role: str,
    asset_type: str,
    page_id: str,
    candidate_id: str,
    case_id: str,
    painting_id: str,
    model_id: str,
    experiment_id: str,
    uncertainty_group_id: str,
    feature_model_id: str,
    map_type: str,
    region_id: str,
    selection_role: str,
    relative_path: str,
    sha256: str,
    size_bytes: object,
    width: object,
    height: object,
    file_format: str,
    source_artifact_key: str,
    is_default_visual: bool,
) -> None:
    """Append one exact-schema visual asset record."""

    VISUAL_ROWS.append(
        {
            "visual_asset_id": stable_id(
                "visual",
                source_notebook_id,
                source_row_id,
            ),
            "asset_role": asset_role,
            "asset_type": asset_type,
            "page_id": page_id,
            "display_order": 0,
            "candidate_id": candidate_id,
            "case_id": case_id,
            "painting_id": painting_id,
            "model_id": model_id,
            "experiment_id": experiment_id,
            "uncertainty_group_id": uncertainty_group_id,
            "feature_model_id": feature_model_id,
            "map_type": map_type,
            "region_id": region_id,
            "selection_role": selection_role,
            "relative_path": relative_path,
            "sha256": sha256,
            "size_bytes": int(float(size_bytes)),
            "width": int(float(width)),
            "height": int(float(height)),
            "format": file_format,
            "source_notebook_id": source_notebook_id,
            "source_artifact_key": source_artifact_key,
            "is_default_visual": bool(is_default_visual),
            "applicability_status": "applicable",
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )


for row in SPATIAL_MAPS_SOURCE.itertuples(index=False):
    selection_role = blank_safe_text(row.selection_role)

    add_manifest_visual(
        source_notebook_id="16",
        source_row_id=str(row.map_image_id),
        asset_role="spatial_diagnostic",
        asset_type=str(row.asset_kind),
        page_id="trustworthiness_xai",
        candidate_id=blank_safe_text(row.candidate_id),
        case_id=blank_safe_text(row.case_id),
        painting_id=blank_safe_text(row.painting_id),
        model_id=blank_safe_text(row.model_id),
        experiment_id=CASE_EXPERIMENT_LOOKUP.get(
            blank_safe_text(row.case_id),
            "",
        ),
        uncertainty_group_id="",
        feature_model_id="",
        map_type=str(row.map_type),
        region_id="",
        selection_role=selection_role,
        relative_path=normalize_relative_paths(
            pd.Series([row.relative_path])
        ).iloc[0],
        sha256=str(row.sha256),
        size_bytes=row.size_bytes,
        width=row.width,
        height=row.height,
        file_format=str(row.format),
        source_artifact_key=blank_safe_text(row.map_id),
        is_default_visual=bool(selection_role),
    )


for row in LOCAL_MAPS_SOURCE.itertuples(index=False):
    selection_role = blank_safe_text(row.selection_role)

    is_curated = bool(
        selection_role
        and selection_role != "primary_nonzero_candidate"
    )

    add_manifest_visual(
        source_notebook_id="17",
        source_row_id=str(row.map_image_id),
        asset_role="local_consistency_diagnostic",
        asset_type=str(row.asset_kind),
        page_id="trustworthiness_xai",
        candidate_id=blank_safe_text(row.candidate_id),
        case_id=blank_safe_text(row.case_id),
        painting_id=blank_safe_text(row.painting_id),
        model_id=blank_safe_text(row.model_id),
        experiment_id=CASE_EXPERIMENT_LOOKUP.get(
            blank_safe_text(row.case_id),
            "",
        ),
        uncertainty_group_id="",
        feature_model_id="",
        map_type=str(row.map_type),
        region_id="",
        selection_role=selection_role,
        relative_path=normalize_relative_paths(
            pd.Series([row.relative_path])
        ).iloc[0],
        sha256=str(row.sha256),
        size_bytes=row.size_bytes,
        width=row.width,
        height=row.height,
        file_format=str(row.format),
        source_artifact_key=blank_safe_text(row.map_id),
        is_default_visual=is_curated,
    )


for row in UNCERTAINTY_MAPS_SOURCE.itertuples(index=False):
    map_type = str(row.map_type)

    add_manifest_visual(
        source_notebook_id="19",
        source_row_id=str(row.map_asset_id),
        asset_role="uncertainty_explanation",
        asset_type=str(row.asset_kind),
        page_id="robustness_uncertainty",
        candidate_id=blank_safe_text(row.candidate_id),
        case_id=blank_safe_text(row.case_id),
        painting_id=blank_safe_text(row.painting_id),
        model_id=blank_safe_text(row.model_id),
        experiment_id=CASE_EXPERIMENT_LOOKUP.get(
            blank_safe_text(row.case_id),
            "",
        ),
        uncertainty_group_id=blank_safe_text(
            row.uncertainty_group_id
        ),
        feature_model_id="",
        map_type=map_type,
        region_id=blank_safe_text(row.region_scope),
        selection_role=blank_safe_text(row.selection_role),
        relative_path=normalize_relative_paths(
            pd.Series([row.relative_path])
        ).iloc[0],
        sha256=str(row.sha256),
        size_bytes=row.size_bytes,
        width=row.width,
        height=row.height,
        file_format=str(row.format),
        source_artifact_key=(
            blank_safe_text(row.archive_key)
            or blank_safe_text(row.source_artifact_key)
        ),
        is_default_visual=map_type.startswith("selected_"),
    )


for row in SEMANTIC_MAPS_SOURCE.itertuples(index=False):
    add_manifest_visual(
        source_notebook_id="20",
        source_row_id=str(row.semantic_map_asset_id),
        asset_role="semantic_structural_diagnostic",
        asset_type=str(row.asset_kind),
        page_id="trustworthiness_xai",
        candidate_id=blank_safe_text(row.candidate_id),
        case_id=blank_safe_text(row.case_id),
        painting_id=blank_safe_text(row.painting_id),
        model_id=blank_safe_text(row.model_id),
        experiment_id=CASE_EXPERIMENT_LOOKUP.get(
            blank_safe_text(row.case_id),
            "",
        ),
        uncertainty_group_id="",
        feature_model_id=blank_safe_text(
            row.feature_model_id
        ),
        map_type=str(row.map_type),
        region_id=blank_safe_text(row.region_id),
        selection_role=blank_safe_text(row.selection_role),
        relative_path=normalize_relative_paths(
            pd.Series([row.relative_path])
        ).iloc[0],
        sha256=str(row.sha256),
        size_bytes=row.size_bytes,
        width=row.width,
        height=row.height,
        file_format=str(row.format),
        source_artifact_key=blank_safe_text(row.archive_key),
        is_default_visual=False,
    )


for row in DAMAGE_SIZE_MAPS_SOURCE.itertuples(index=False):
    add_manifest_visual(
        source_notebook_id="22",
        source_row_id=str(row.map_image_id),
        asset_role="damage_size_uncertainty",
        asset_type="uncertainty_map",
        page_id="robustness_uncertainty",
        candidate_id="",
        case_id=str(row.case_id),
        painting_id=str(row.painting_id),
        model_id="stable_diffusion_inpainting",
        experiment_id="damage_size_sensitivity",
        uncertainty_group_id=str(row.uncertainty_group_id),
        feature_model_id="",
        map_type=str(row.map_metric_name),
        region_id="full_image",
        selection_role="damage_size_group_map",
        relative_path=(
            Path(
                SETTINGS["inputs"][
                    "damage_size_map_manifest_path"
                ]
            )
            .parent
            .parent
            .joinpath(str(row.relative_path))
            .as_posix()
        ),
        sha256=str(row.sha256),
        size_bytes=row.size_bytes,
        width=row.width,
        height=row.height,
        file_format=str(row.format),
        source_artifact_key=blank_safe_text(row.raw_map_key),
        is_default_visual=False,
    )

print("Map-manifest visual rows:", len(VISUAL_ROWS))

Map-manifest visual rows: 23864


In [45]:
for row in REPRESENTATIVE_CASES_SOURCE.itertuples(
    index=False
):
    relative_path = normalize_relative_paths(
        pd.Series([row.restored_path])
    ).iloc[0]

    metadata = image_file_metadata(relative_path)

    recorded_sha256 = str(row.restored_sha256)

    if metadata["sha256"] != recorded_sha256:
        raise ValueError(
            "Representative restoration checksum mismatch: "
            f"{relative_path}"
        )

    add_manifest_visual(
        source_notebook_id="21",
        source_row_id=str(row.representative_row_id),
        asset_role="representative_restoration",
        asset_type="restoration_image",
        page_id="model_performance",
        candidate_id=str(row.candidate_id),
        case_id=str(row.case_id),
        painting_id=str(row.painting_id),
        model_id=str(row.model_id),
        experiment_id=str(row.experiment_id),
        uncertainty_group_id="",
        feature_model_id="",
        map_type="restored_candidate",
        region_id="full_image",
        selection_role=str(row.selection_role),
        relative_path=relative_path,
        sha256=metadata["sha256"],
        size_bytes=metadata["size_bytes"],
        width=metadata["width"],
        height=metadata["height"],
        file_format=metadata["format"],
        source_artifact_key=(
            "multi_model_comparison.representative_cases"
        ),
        is_default_visual=True,
    )

print(
    "Rows after representative restorations:",
    len(VISUAL_ROWS),
)

Rows after representative restorations: 23940


In [46]:
THESIS_FIGURE_PAGE_MAP = {
    1: "overview",
    2: "study_design",
    3: "model_performance",
    4: "model_performance",
    5: "metric_framework",
    6: "robustness_uncertainty",
    7: "robustness_uncertainty",
    8: "robustness_uncertainty",
    9: "model_performance",
    10: "metric_framework",
    11: "model_performance",
    12: "robustness_uncertainty",
    13: "robustness_uncertainty",
    14: "trustworthiness_xai",
    15: "metric_framework",
    16: "trustworthiness_xai",
    17: "reports_reproducibility",
    18: "reports_reproducibility",
}

PUBLICATION_FIGURE_PAGE_MAP = {
    1: "overview",
    2: "robustness_uncertainty",
    3: "robustness_uncertainty",
    4: "trustworthiness_xai",
    5: "trustworthiness_xai",
    6: "reports_reproducibility",
}


def add_final_figure(
    path: Path,
    *,
    figure_family: str,
    page_map: dict[int, str],
) -> None:
    relative_path = path.relative_to(PROJECT_ROOT).as_posix()
    figure_number = int(path.stem.split("_", 1)[0])
    metadata = image_file_metadata(relative_path)

    add_manifest_visual(
        source_notebook_id="33",
        source_row_id=f"{figure_family}::{path.name}",
        asset_role=figure_family,
        asset_type="summary_figure",
        page_id=page_map[figure_number],
        candidate_id="",
        case_id="",
        painting_id="",
        model_id="",
        experiment_id="",
        uncertainty_group_id="",
        feature_model_id="",
        map_type=path.stem,
        region_id="",
        selection_role=f"default_{figure_family}",
        relative_path=relative_path,
        sha256=metadata["sha256"],
        size_bytes=metadata["size_bytes"],
        width=metadata["width"],
        height=metadata["height"],
        file_format=metadata["format"],
        source_artifact_key=(
            f"final_report.{figure_family}s"
        ),
        is_default_visual=True,
    )


for figure_path in FINAL_THESIS_FIGURES:
    add_final_figure(
        figure_path,
        figure_family="thesis_figure",
        page_map=THESIS_FIGURE_PAGE_MAP,
    )

for figure_path in FINAL_PUBLICATION_FIGURES:
    add_final_figure(
        figure_path,
        figure_family="publication_figure",
        page_map=PUBLICATION_FIGURE_PAGE_MAP,
    )

print("All visual rows constructed:", len(VISUAL_ROWS))

All visual rows constructed: 23964


In [47]:
VISUAL_ASSET_INDEX = pd.DataFrame(VISUAL_ROWS)

VISUAL_PAGE_ORDER = dict(
    zip(
        PAGE_PLAN["page_id"],
        PAGE_PLAN["display_order"],
    )
)

VISUAL_ASSET_INDEX["_page_order"] = (
    VISUAL_ASSET_INDEX["page_id"].map(VISUAL_PAGE_ORDER)
)

VISUAL_ASSET_INDEX = (
    VISUAL_ASSET_INDEX
    .sort_values(
        [
            "_page_order",
            "is_default_visual",
            "asset_role",
            "painting_id",
            "case_id",
            "candidate_id",
            "map_type",
            "relative_path",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
        kind="stable",
    )
    .drop(columns="_page_order")
    .reset_index(drop=True)
)

VISUAL_ASSET_INDEX["display_order"] = (
    VISUAL_ASSET_INDEX.index + 1
)

VISUAL_ASSET_INDEX = validate_output_frame(
    VISUAL_ASSET_INDEX,
    "visual_asset_index",
)

UNIQUE_VISUAL_PATHS = sorted(
    set(VISUAL_ASSET_INDEX["relative_path"])
)

MISSING_VISUAL_PATHS = [
    path
    for path in UNIQUE_VISUAL_PATHS
    if not (PROJECT_ROOT / path).is_file()
]

VISUAL_COVERAGE_VIEW = (
    VISUAL_ASSET_INDEX.groupby(
        [
            "source_notebook_id",
            "asset_role",
            "asset_type",
            "format",
        ],
        dropna=False,
        sort=True,
    )
    .agg(
        records=("visual_asset_id", "size"),
        physical_files=("relative_path", "nunique"),
        default_visuals=("is_default_visual", "sum"),
    )
    .reset_index()
)

VISUAL_PAGE_VIEW = (
    VISUAL_ASSET_INDEX.groupby(
        "page_id",
        sort=True,
    )
    .agg(
        records=("visual_asset_id", "size"),
        physical_files=("relative_path", "nunique"),
        default_visuals=("is_default_visual", "sum"),
    )
    .reset_index()
)

display(VISUAL_COVERAGE_VIEW)
display(VISUAL_PAGE_VIEW)

print("Visual index records:", len(VISUAL_ASSET_INDEX))
print("Unique physical paths:", len(UNIQUE_VISUAL_PATHS))
print("Missing physical paths:", len(MISSING_VISUAL_PATHS))

,source_notebook_id,asset_role,asset_type,format,records,physical_files,default_visuals
0,16,spatial_diagnostic,candidate_map,PNG,10050,10050,0
1,16,spatial_diagnostic,selected_panel,PNG,12,12,12
2,17,local_consistency_diagnostic,candidate_map,PNG,3270,3270,0
3,17,local_consistency_diagnostic,selected_panel,PNG,12,12,12
4,19,uncertainty_explanation,component_map,PNG,650,650,0
5,19,uncertainty_explanation,raw_numeric_map,NPZ,130,1,0
6,19,uncertainty_explanation,selected_panel,PNG,15,15,15
7,19,uncertainty_explanation,uncertainty_overlay,PNG,130,130,0
8,19,uncertainty_explanation,uncertainty_panel,PNG,130,130,0
9,20,semantic_structural_diagnostic,numeric_map_bundle,NPZ,8340,1,0


,page_id,records,physical_files,default_visuals
0,metric_framework,3,3,3
1,model_performance,80,80,80
2,overview,2,2,2
3,reports_reproducibility,3,3,3
4,robustness_uncertainty,1097,968,22
5,study_design,1,1,1
6,trustworthiness_xai,22778,14439,28


Visual index records: 23964
Unique physical paths: 14996
Missing physical paths: 0


In [48]:
# Remove the previous failed Batch 8 checks before validating the fix.
PRE_BATCH_8_CHECKS = [
    check
    for check in VALIDATION.checks
    if check.validation_stage != "batch_8_visual_asset_index"
]

VALIDATION = ValidationCollector()
VALIDATION.extend(PRE_BATCH_8_CHECKS)

EXPECTED_VISUAL_SOURCE_COUNTS = {
    "16": 10062,
    "17": 3282,
    "19": 1055,
    "20": 9430,
    "21": 76,
    "22": 35,
    "33": 24,
}

OBSERVED_VISUAL_SOURCE_COUNTS = (
    VISUAL_ASSET_INDEX["source_notebook_id"]
    .value_counts()
    .sort_index()
    .to_dict()
)

EXPECTED_VISUAL_ROLE_COUNTS = {
    "spatial_diagnostic": 10062,
    "local_consistency_diagnostic": 3282,
    "uncertainty_explanation": 1055,
    "semantic_structural_diagnostic": 9430,
    "representative_restoration": 76,
    "damage_size_uncertainty": 35,
    "thesis_figure": 18,
    "publication_figure": 6,
}

OBSERVED_VISUAL_ROLE_COUNTS = (
    VISUAL_ASSET_INDEX["asset_role"]
    .value_counts()
    .to_dict()
)

BATCH_8_CHECKS = (
    (
        "visual_asset_row_count",
        "The complete visual index contains 23,964 records",
        23964,
        len(VISUAL_ASSET_INDEX),
        len(VISUAL_ASSET_INDEX) == 23964,
    ),
    (
        "visual_source_counts",
        "Every visual source has complete manifest coverage",
        EXPECTED_VISUAL_SOURCE_COUNTS,
        OBSERVED_VISUAL_SOURCE_COUNTS,
        (
            OBSERVED_VISUAL_SOURCE_COUNTS
            == EXPECTED_VISUAL_SOURCE_COUNTS
        ),
    ),
    (
        "visual_role_counts",
        "Every visual role has expected coverage",
        EXPECTED_VISUAL_ROLE_COUNTS,
        OBSERVED_VISUAL_ROLE_COUNTS,
        (
            OBSERVED_VISUAL_ROLE_COUNTS
            == EXPECTED_VISUAL_ROLE_COUNTS
        ),
    ),
    (
        "visual_identifier_uniqueness",
        "Visual asset identifiers are unique",
        len(VISUAL_ASSET_INDEX),
        VISUAL_ASSET_INDEX["visual_asset_id"].nunique(),
        (
            VISUAL_ASSET_INDEX["visual_asset_id"].nunique()
            == len(VISUAL_ASSET_INDEX)
        ),
    ),
    (
        "visual_unique_physical_paths",
        "The index resolves to 14,996 physical files",
        14996,
        len(UNIQUE_VISUAL_PATHS),
        len(UNIQUE_VISUAL_PATHS) == 14996,
    ),
    (
        "visual_path_existence",
        "Every indexed visual path exists",
        [],
        MISSING_VISUAL_PATHS,
        not MISSING_VISUAL_PATHS,
    ),
    (
        "visual_paths_relative",
        "Every visual path is repository-relative",
        True,
        bool(
            VISUAL_ASSET_INDEX["relative_path"]
            .map(is_repo_relative)
            .all()
        ),
        bool(
            VISUAL_ASSET_INDEX["relative_path"]
            .map(is_repo_relative)
            .all()
        ),
    ),
    (
        "visual_checksums_present",
        "Every visual record has a checksum",
        0,
        int(
            normalized_text(
                VISUAL_ASSET_INDEX["sha256"]
            ).eq("").sum()
        ),
        not normalized_text(
            VISUAL_ASSET_INDEX["sha256"]
        ).eq("").any(),
    ),
    (
        "visual_dimensions_positive",
        "Every indexed visual has positive recorded dimensions",
        True,
        bool(
            (
                VISUAL_ASSET_INDEX["width"] > 0
            ).all()
            and (
                VISUAL_ASSET_INDEX["height"] > 0
            ).all()
        ),
        bool(
            (
                VISUAL_ASSET_INDEX["width"] > 0
            ).all()
            and (
                VISUAL_ASSET_INDEX["height"] > 0
            ).all()
        ),
    ),
    (
        "representative_restoration_population",
        "All 76 representative restorations are indexed",
        76,
        int(
            VISUAL_ASSET_INDEX["asset_role"]
            .eq("representative_restoration")
            .sum()
        ),
        int(
            VISUAL_ASSET_INDEX["asset_role"]
            .eq("representative_restoration")
            .sum()
        ) == 76,
    ),
    (
        "final_figure_population",
        "All 24 final report figures are indexed",
        24,
        int(
            VISUAL_ASSET_INDEX["source_notebook_id"]
            .eq("33")
            .sum()
        ),
        int(
            VISUAL_ASSET_INDEX["source_notebook_id"]
            .eq("33")
            .sum()
        ) == 24,
    ),
    (
        "default_visual_population",
        "The curated initial presentation contains 139 visual records",
        139,
        int(VISUAL_ASSET_INDEX["is_default_visual"].sum()),
        int(VISUAL_ASSET_INDEX["is_default_visual"].sum())
        == 139,
    ),
    (
        "visual_status",
        "All visual-index records have valid status",
        {"ok"},
        set(VISUAL_ASSET_INDEX["status"]),
        set(VISUAL_ASSET_INDEX["status"]) == {"ok"},
    ),
    (
        "no_canonical_files_written",
        "Batch 8 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_8_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_8_visual_asset_index",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "The visual-asset index differs from validated upstream artifacts."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_8_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_8_visual_asset_index"
        )
    ]
    .reset_index(drop=True)
)

display(VISUAL_PAGE_VIEW)
display(
    BATCH_8_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 8 completed.")
print("Visual records:", len(VISUAL_ASSET_INDEX))
print("Physical files:", len(UNIQUE_VISUAL_PATHS))
print(
    "Default presentation records:",
    int(VISUAL_ASSET_INDEX["is_default_visual"].sum()),
)
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,page_id,records,physical_files,default_visuals
0,metric_framework,3,3,3
1,model_performance,80,80,80
2,overview,2,2,2
3,reports_reproducibility,3,3,3
4,robustness_uncertainty,1097,968,22
5,study_design,1,1,1
6,trustworthiness_xai,22778,14439,28


,check_id,severity,passed,observed
0,visual_asset_row_count,blocking,True,23964
1,visual_source_counts,blocking,True,"{""16"": 10062, ""17"": 3282, ""19"": 1055, ""20"": 94..."
2,visual_role_counts,blocking,True,"{""damage_size_uncertainty"": 35, ""local_consist..."
3,visual_identifier_uniqueness,blocking,True,23964
4,visual_unique_physical_paths,blocking,True,14996
5,visual_path_existence,blocking,True,[]
6,visual_paths_relative,blocking,True,True
7,visual_checksums_present,blocking,True,0
8,visual_dimensions_positive,blocking,True,True
9,representative_restoration_population,blocking,True,76


Batch 8 completed.
Visual records: 23964
Physical files: 14996
Default presentation records: 139
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 9 — Reports, research questions, and dashboard summary

This batch constructs:

- a complete index of all 104 upstream reports;
- model, case, painting, protocol, analytical, final, and limitation report roles;
- report checksums, sizes, section counts, and embedded-image counts;
- explicit coverage for the three research questions and practical output;
- the complete dashboard summary object.

The report index includes every report, not only the reports selected for initial display. The research-question table separates supported conclusions from interpretations that the completed evidence cannot justify.

In [49]:
import re


REPORT_PATTERN = (
    r"^outputs/"
    r"(?:0[1-9]|[12][0-9]|3[0-3])_[^/]+/"
    r"reports/.+\.(?:html|md)$"
)

REPORT_FILE_INVENTORY = (
    INVENTORY.loc[
        INVENTORY["relative_path"]
        .astype(str)
        .str.match(REPORT_PATTERN, case=False, na=False)
    ]
    .copy()
    .sort_values("relative_path")
    .reset_index(drop=True)
)

MODEL_REPORT_SOURCE = (
    INPUT_TABLES["model_report_index_path"]
    .copy()
    .reset_index(drop=True)
)

CASE_REPORT_SOURCE = (
    INPUT_TABLES["case_report_index_path"]
    .copy()
    .reset_index(drop=True)
)

PAINTING_REPORT_SOURCE = (
    INPUT_TABLES["painting_report_index_path"]
    .copy()
    .reset_index(drop=True)
)


def indexed_report_lookup(
    frame: pd.DataFrame,
) -> dict[str, dict]:
    """Build a normalized report-path lookup."""

    records = {}

    for record in frame.to_dict(orient="records"):
        path = (
            str(record["report_path"])
            .replace("\\", "/")
            .strip()
        )
        records[path] = record

    return records


MODEL_REPORT_LOOKUP = indexed_report_lookup(
    MODEL_REPORT_SOURCE
)

CASE_REPORT_LOOKUP_COMPLETE = indexed_report_lookup(
    CASE_REPORT_SOURCE
)

PAINTING_REPORT_LOOKUP_COMPLETE = indexed_report_lookup(
    PAINTING_REPORT_SOURCE
)

KNOWN_MODEL_IDS = set(
    MODEL_DIMENSION["model_id"].astype(str)
)


def classify_report(
    relative_path: str,
    source_notebook_id: str,
) -> tuple[str, str, str]:
    """Return report family, role, and scope."""

    path = relative_path.lower()

    if "/reports/cases/" in path:
        return "case", "case_report", "individual_case"

    if "/reports/paintings/" in path:
        return "painting", "painting_report", "individual_painting"

    if source_notebook_id == "31":
        return "model", "model_report", "individual_model"

    if (
        source_notebook_id == "30"
        and "/reports/model_cards/" in path
    ):
        return "model", "model_card", "individual_model"

    if path.endswith("/reports/index.html"):
        return (
            "report_collection",
            "report_collection_index",
            "case_and_painting_collection",
        )

    if path.endswith("/final_evaluation.html"):
        return "final", "final_evaluation_report", "complete_thesis"

    if path.endswith("/limitations_and_deviations.md"):
        return (
            "final",
            "limitations_and_deviations",
            "complete_thesis",
        )

    if source_notebook_id in {"03", "07", "08", "11"}:
        return "method", "method_protocol", "method_definition"

    if source_notebook_id == "12":
        return (
            "model",
            "partial_evaluation_report",
            "bounded_sdxl_evidence",
        )

    return (
        "analysis",
        "analytical_report",
        "notebook_analysis",
    )


def report_text_statistics(
    full_path: Path,
) -> dict:
    """Inspect report structure without rendering or modifying it."""

    text = full_path.read_text(encoding="utf-8-sig")
    suffix = full_path.suffix.lower()

    if suffix == ".html":
        section_count = len(
            re.findall(
                r"<section\b",
                text,
                flags=re.IGNORECASE,
            )
        )

        image_sources = re.findall(
            r"<img\b[^>]*\bsrc=[\"']([^\"']+)[\"']",
            text,
            flags=re.IGNORECASE,
        )

        embedded_image_count = sum(
            source.lower().startswith("data:image/")
            for source in image_sources
        )

        self_contained = all(
            source.lower().startswith("data:")
            for source in image_sources
        )

    else:
        section_count = len(
            re.findall(
                r"(?m)^##\s+",
                text,
            )
        )
        embedded_image_count = len(
            re.findall(r"!\[[^\]]*\]\(", text)
        )
        self_contained = True

    return {
        "section_count": int(section_count),
        "embedded_image_count": int(
            embedded_image_count
        ),
        "self_contained": bool(self_contained),
    }


print("Discovered report files:", len(REPORT_FILE_INVENTORY))
print(
    "Report formats:",
    REPORT_FILE_INVENTORY["format"]
    .value_counts()
    .to_dict(),
)

Discovered report files: 104
Report formats: {'html': 94, 'markdown': 10}


In [50]:
REPORT_ROWS = []

for inventory_row in REPORT_FILE_INVENTORY.itertuples(
    index=False
):
    relative_path = str(inventory_row.relative_path)
    full_path = PROJECT_ROOT / relative_path

    source_match = re.match(
        r"^outputs/(\d{2})_",
        relative_path,
    )

    if source_match is None:
        raise ValueError(
            f"Cannot identify report producer: {relative_path}"
        )

    source_notebook_id = source_match.group(1)

    report_family, report_role, scope = classify_report(
        relative_path,
        source_notebook_id,
    )

    indexed_record = (
        MODEL_REPORT_LOOKUP.get(relative_path)
        or CASE_REPORT_LOOKUP_COMPLETE.get(relative_path)
        or PAINTING_REPORT_LOOKUP_COMPLETE.get(relative_path)
    )

    parsed_statistics = report_text_statistics(full_path)

    model_id = ""
    case_id = ""
    painting_id = ""
    title = readable_identifier(full_path.stem)

    if relative_path in MODEL_REPORT_LOOKUP:
        indexed_record = MODEL_REPORT_LOOKUP[relative_path]
        model_id = str(indexed_record["model_id"])
        title = str(indexed_record["display_name"])

    elif relative_path in CASE_REPORT_LOOKUP_COMPLETE:
        indexed_record = CASE_REPORT_LOOKUP_COMPLETE[
            relative_path
        ]
        case_id = str(indexed_record["case_id"])
        painting_id = str(indexed_record["painting_id"])
        title = f"Case report — {case_id}"

    elif relative_path in PAINTING_REPORT_LOOKUP_COMPLETE:
        indexed_record = PAINTING_REPORT_LOOKUP_COMPLETE[
            relative_path
        ]
        painting_id = str(indexed_record["painting_id"])
        painting_title = PAINTING_INDEX.loc[
            PAINTING_INDEX["painting_id"].eq(painting_id),
            "title",
        ].iloc[0]
        title = f"Painting report — {painting_title}"

    elif (
        source_notebook_id == "30"
        and full_path.stem in KNOWN_MODEL_IDS
    ):
        model_id = full_path.stem
        title = (
            f"{MODEL_DISPLAY_NAMES.get(model_id, model_id)} "
            "model card"
        )

    if indexed_record is not None:
        section_count = optional_count(
            indexed_record.get("section_count")
        )
        embedded_image_count = optional_count(
            indexed_record.get("embedded_image_count")
        )
        embedded_tile_count = optional_count(
            indexed_record.get("embedded_tile_count")
        )
        self_contained = as_bool(
            indexed_record.get("self_contained")
        )
    else:
        section_count = parsed_statistics["section_count"]
        embedded_image_count = parsed_statistics[
            "embedded_image_count"
        ]
        embedded_tile_count = 0
        self_contained = parsed_statistics[
            "self_contained"
        ]

    applicability_status = (
        "available_partial_evaluation"
        if model_id == "sdxl_inpainting"
        or report_role == "partial_evaluation_report"
        else "available"
    )

    description = {
        "case_report": (
            "Self-contained evidence report for one selected case."
        ),
        "painting_report": (
            "Self-contained evidence report for one controlled painting."
        ),
        "model_report": (
            "Self-contained report covering one restoration model."
        ),
        "model_card": (
            "Model identity, role, coverage, limitations, and compute summary."
        ),
        "report_collection_index": (
            "Entry point for all case and painting reports."
        ),
        "final_evaluation_report": (
            "Complete thesis-oriented evaluation report."
        ),
        "limitations_and_deviations": (
            "Canonical statement of study limitations and deviations."
        ),
        "method_protocol": (
            "Method or evaluation protocol supporting reproducibility."
        ),
        "partial_evaluation_report": (
            "Bounded report for the ten completed SDXL cases."
        ),
        "analytical_report": (
            "Notebook-owned report for a completed analytical stage."
        ),
    }[report_role]

    REPORT_ROWS.append(
        {
            "dashboard_report_id": stable_id(
                "report",
                relative_path,
            ),
            "report_family": report_family,
            "report_role": report_role,
            "display_order": 0,
            "title": title,
            "description": description,
            "model_id": model_id,
            "case_id": case_id,
            "painting_id": painting_id,
            "report_path": relative_path,
            "report_sha256": sha256_file(full_path),
            "size_bytes": int(full_path.stat().st_size),
            "format": (
                "HTML"
                if full_path.suffix.lower() == ".html"
                else "MARKDOWN"
            ),
            "self_contained": self_contained,
            "section_count": section_count,
            "embedded_image_count": embedded_image_count,
            "embedded_tile_count": embedded_tile_count,
            "source_notebook_id": source_notebook_id,
            "source_artifact_key": (
                f"notebook_{source_notebook_id}.report"
            ),
            "scope": scope,
            "applicability_status": applicability_status,
            "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
            "status": "ok",
            "issue": "",
        }
    )

REPORT_INDEX = pd.DataFrame(REPORT_ROWS)

REPORT_ROLE_ORDER = {
    "final_evaluation_report": 1,
    "limitations_and_deviations": 2,
    "report_collection_index": 3,
    "model_report": 4,
    "model_card": 5,
    "case_report": 6,
    "painting_report": 7,
    "analytical_report": 8,
    "partial_evaluation_report": 9,
    "method_protocol": 10,
}

REPORT_INDEX["_role_order"] = (
    REPORT_INDEX["report_role"].map(REPORT_ROLE_ORDER)
)

REPORT_INDEX = (
    REPORT_INDEX
    .sort_values(
        [
            "_role_order",
            "model_id",
            "painting_id",
            "case_id",
            "report_path",
        ],
        kind="stable",
    )
    .drop(columns="_role_order")
    .reset_index(drop=True)
)

REPORT_INDEX["display_order"] = REPORT_INDEX.index + 1

REPORT_INDEX = validate_output_frame(
    REPORT_INDEX,
    "report_index",
)

REPORT_COVERAGE_VIEW = (
    REPORT_INDEX.groupby(
        [
            "report_family",
            "report_role",
            "format",
            "self_contained",
        ],
        sort=True,
    )
    .agg(
        reports=("dashboard_report_id", "size"),
        total_sections=("section_count", "sum"),
        embedded_images=("embedded_image_count", "sum"),
        embedded_tiles=("embedded_tile_count", "sum"),
    )
    .reset_index()
)

display(REPORT_COVERAGE_VIEW)

print("Report-index rows:", len(REPORT_INDEX))

,report_family,report_role,format,self_contained,reports,total_sections,embedded_images,embedded_tiles
0,analysis,analytical_report,HTML,True,8,103,250,0
1,case,case_report,HTML,True,30,390,330,810
2,final,final_evaluation_report,HTML,True,1,19,68,0
3,final,limitations_and_deviations,MARKDOWN,True,1,2,0,0
4,method,method_protocol,MARKDOWN,True,4,41,0,0
5,model,model_card,MARKDOWN,True,4,52,0,0
6,model,model_report,HTML,True,4,60,63,298
7,model,partial_evaluation_report,MARKDOWN,True,1,5,0,0
8,painting,painting_report,HTML,True,50,550,300,1334
9,report_collection,report_collection_index,HTML,True,1,6,2,0


Report-index rows: 104


In [51]:
RQ_DEFINITIONS = {
    item["id"]: item["title"]
    for item in SETTINGS["research_questions"]
}

RQ_ROWS = [
    {
        "coverage_row_id": stable_id("coverage", "rq1"),
        "research_question_id": "rq1",
        "research_question": RQ_DEFINITIONS["rq1"],
        "display_order": 1,
        "page_id": "metric_framework",
        "evidence_role": (
            "Metric families, canonical regions, disagreement, "
            "correlations, and ablation"
        ),
        "coverage_status": "addressed_by_completed_evidence",
        "source_notebook_ids_json": json_list(
            ["08", "13", "14", "15", "16", "17", "20", "21", "26", "28", "33"]
        ),
        "source_paths_json": json_list(
            [
                SETTINGS["inputs"]["region_policy_path"],
                SETTINGS["inputs"]["model_comparison_path"],
                SETTINGS["inputs"]["metric_disagreement_path"],
                SETTINGS["inputs"]["metric_correlations_path"],
                SETTINGS["inputs"]["ablation_results_path"],
                SETTINGS["inputs"]["thesis_tables_path"],
            ]
        ),
        "supported_interpretation": (
            "Restoration quality requires complementary pixel, "
            "perceptual, feature, texture, colour, seam, spatial, "
            "and structural evidence evaluated in valid regions."
        ),
        "prohibited_interpretation": (
            "No individual metric or combined universal score is "
            "treated as conservation truth."
        ),
        "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
        "status": "ok",
        "issue": "",
    },
    {
        "coverage_row_id": stable_id("coverage", "rq2"),
        "research_question_id": "rq2",
        "research_question": RQ_DEFINITIONS["rq2"],
        "display_order": 2,
        "page_id": "model_performance",
        "evidence_role": (
            "Overall model comparison and controlled conditional "
            "sensitivity analyses"
        ),
        "coverage_status": "addressed_by_completed_evidence",
        "source_notebook_ids_json": json_list(
            ["21", "23", "24", "25", "26", "30", "33"]
        ),
        "source_paths_json": json_list(
            [
                SETTINGS["inputs"]["model_comparison_path"],
                SETTINGS["inputs"]["damage_size_analysis_path"],
                SETTINGS["inputs"]["mask_robustness_analysis_path"],
                SETTINGS["inputs"]["degradation_analysis_path"],
                SETTINGS["inputs"]["statistical_results_path"],
                SETTINGS["inputs"]["model_cards_path"],
            ]
        ),
        "supported_interpretation": (
            "LaMa is the strongest general baseline in this controlled "
            "benchmark, while performance still changes by metric, "
            "damage condition, painting, and experimental scope."
        ),
        "prohibited_interpretation": (
            "The result does not establish a universally best model, "
            "real-world treatment suitability, or full SDXL performance."
        ),
        "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
        "status": "ok",
        "issue": "",
    },
    {
        "coverage_row_id": stable_id("coverage", "rq3"),
        "research_question_id": "rq3",
        "research_question": RQ_DEFINITIONS["rq3"],
        "display_order": 3,
        "page_id": "robustness_uncertainty",
        "evidence_role": (
            "Repeated-seed variability, uncertainty maps, diagnostic "
            "flags, and case-level explanations"
        ),
        "coverage_status": "addressed_by_completed_evidence",
        "source_notebook_ids_json": json_list(
            ["18", "19", "22", "27", "28", "29", "33"]
        ),
        "source_paths_json": json_list(
            [
                SETTINGS["inputs"]["canonical_uncertainty_path"],
                SETTINGS["inputs"]["uncertainty_map_manifest_path"],
                SETTINGS["inputs"]["damage_size_uncertainty_path"],
                SETTINGS["inputs"]["damage_size_map_manifest_path"],
                SETTINGS["inputs"]["trustworthiness_flags_path"],
                SETTINGS["inputs"]["explanation_cases_path"],
            ]
        ),
        "supported_interpretation": (
            "Repeated-seed variability identifies diffusion cases "
            "whose outputs are less consistent and should receive "
            "closer visual review."
        ),
        "prohibited_interpretation": (
            "Empirical variability is not calibrated confidence, "
            "correctness, historical plausibility, or expert approval."
        ),
        "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
        "status": "ok",
        "issue": "",
    },
    {
        "coverage_row_id": stable_id(
            "coverage",
            "practical_output",
        ),
        "research_question_id": "practical_output",
        "research_question": RQ_DEFINITIONS[
            "practical_output"
        ],
        "display_order": 4,
        "page_id": "reports_reproducibility",
        "evidence_role": (
            "Complete case, painting, model, visual, report, and "
            "provenance indexes"
        ),
        "coverage_status": "addressed_by_completed_evidence",
        "source_notebook_ids_json": json_list(
            ["27", "29", "30", "31", "32", "33", "34"]
        ),
        "source_paths_json": json_list(
            [
                SETTINGS["inputs"]["explanation_cases_path"],
                SETTINGS["inputs"]["model_report_index_path"],
                SETTINGS["inputs"]["case_report_index_path"],
                SETTINGS["inputs"]["painting_report_index_path"],
                SETTINGS["inputs"]["final_report_path"],
                (
                    f"{OUTPUT_SPEC['root']}/"
                    f"{OUTPUT_SPEC['visual_asset_index_path']}"
                ),
            ]
        ),
        "supported_interpretation": (
            "The completed evidence is packaged as traceable "
            "decision support with complete indexes and selected "
            "human-readable reports."
        ),
        "prohibited_interpretation": (
            "The dashboard is not a restoration tool, expert-review "
            "replacement, or physical-treatment recommendation."
        ),
        "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
        "status": "ok",
        "issue": "",
    },
]

RESEARCH_QUESTION_COVERAGE = validate_output_frame(
    pd.DataFrame(RQ_ROWS),
    "research_question_coverage",
)

display(
    RESEARCH_QUESTION_COVERAGE[
        [
            "research_question_id",
            "research_question",
            "coverage_status",
            "supported_interpretation",
            "prohibited_interpretation",
        ]
    ]
)

print(
    "Research-question coverage rows:",
    len(RESEARCH_QUESTION_COVERAGE),
)

,research_question_id,research_question,coverage_status,supported_interpretation,prohibited_interpretation
0,rq1,Multi-metric evaluation beyond traditional ima...,addressed_by_completed_evidence,Restoration quality requires complementary pix...,No individual metric or combined universal sco...
1,rq2,Conditional model differences across paintings...,addressed_by_completed_evidence,LaMa is the strongest general baseline in this...,The result does not establish a universally be...
2,rq3,Repeated-candidate uncertainty and speculative...,addressed_by_completed_evidence,Repeated-seed variability identifies diffusion...,Empirical variability is not calibrated confid...
3,practical_output,Interpretable museum-oriented decision-support...,addressed_by_completed_evidence,The completed evidence is packaged as traceabl...,"The dashboard is not a restoration tool, exper..."


Research-question coverage rows: 4


In [52]:
DASHBOARD_TABLES = {
    "headline_findings": HEADLINE_FINDINGS,
    "study_design": STUDY_DESIGN,
    "metric_framework": METRIC_FRAMEWORK,
    "performance_summary": PERFORMANCE_SUMMARY,
    "sensitivity_summary": SENSITIVITY_SUMMARY,
    "uncertainty_summary": UNCERTAINTY_SUMMARY,
    "trustworthiness_summary": TRUSTWORTHINESS_SUMMARY,
    "compute_summary": COMPUTE_SUMMARY,
    "research_question_coverage": (
        RESEARCH_QUESTION_COVERAGE
    ),
}

DASHBOARD_INDEXES = {
    "case_index": CASE_INDEX,
    "painting_index": PAINTING_INDEX,
    "visual_asset_index": VISUAL_ASSET_INDEX,
    "report_index": REPORT_INDEX,
}

DASHBOARD_SUMMARY_GENERATED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

DASHBOARD_SUMMARY = {
    "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
    "notebook_id": SETTINGS["notebook_id"],
    "notebook_stem": SETTINGS["notebook_stem"],
    "dataset_id": SETTINGS["dataset_id"],
    "dataset_version": SETTINGS["dataset_version"],
    "dataset_scope": SETTINGS["dataset_scope"],
    "generated_at_utc": DASHBOARD_SUMMARY_GENERATED_AT_UTC,
    "creates_new_scientific_evidence": False,
    "inventory_run_id": INVENTORY_RUN["inventory_run_id"],
    "page_count": len(PAGE_PLAN),
    "pages": PAGE_PLAN.to_dict(orient="records"),
    "population": {
        "paintings": len(PAINTING_INDEX),
        "registered_cases": len(CASE_REGISTRY),
        "restoration_cases": CASE_INDEX["case_id"].nunique(),
        "approved_candidates": len(CASE_INDEX),
        "primary_three_model_candidates": (
            int(PRIMARY_CANDIDATE_COUNT)
        ),
        "stable_diffusion_candidates": (
            int(STABLE_DIFFUSION_COUNT)
        ),
        "sdxl_candidates": int(SDXL_COUNT),
        "canonical_uncertainty_groups": 130,
        "damage_size_uncertainty_groups": 35,
        "reports": len(REPORT_INDEX),
        "visual_records": len(VISUAL_ASSET_INDEX),
        "visual_physical_files": len(UNIQUE_VISUAL_PATHS),
    },
    "table_rows": {
        name: len(frame)
        for name, frame in DASHBOARD_TABLES.items()
    },
    "index_rows": {
        **{
            name: len(frame)
            for name, frame in DASHBOARD_INDEXES.items()
        },
        "filter_options": len(FILTER_OPTIONS),
    },
    "headline_findings": (
        HEADLINE_FINDINGS[
            [
                "finding_id",
                "title",
                "value",
                "value_unit",
                "conclusion",
                "limitation",
            ]
        ].to_dict(orient="records")
    ),
    "model_coverage": (
        MODEL_DIMENSION[
            [
                "model_id",
                "display_name",
                "evaluation_status",
                "evaluated_case_count",
                "evaluated_candidate_count",
            ]
        ].to_dict(orient="records")
    ),
    "methodological_boundaries": [
        "No universal combined quality or trust score.",
        "Uncertainty is empirical seed variability, not calibrated confidence.",
        "Computational flags are not expert ground truth.",
        "Retrieval similarity is not restoration correctness.",
        "Feature similarity is not historical authenticity.",
        "SDXL evidence is limited to ten completed cases.",
        "The dashboard does not provide conservation approval.",
    ],
    "presentation": {
        "theme_id": PRESENTATION["theme_id"],
        "principal_page_count": int(
            PRESENTATION["principal_page_count"]
        ),
        "complete_population_filterable": bool(
            PRESENTATION[
                "complete_population_must_remain_filterable"
            ]
        ),
        "representative_defaults_only": bool(
            PRESENTATION[
                "default_case_selection_is_representative_only"
            ]
        ),
    },
    "persistence_status": "not_yet_persisted",
}

print("Dashboard summary assembled.")
print(
    "Dashboard table rows:",
    DASHBOARD_SUMMARY["table_rows"],
)
print(
    "Dashboard index rows:",
    DASHBOARD_SUMMARY["index_rows"],
)

Dashboard summary assembled.
Dashboard table rows: {'headline_findings': 8, 'study_design': 35, 'metric_framework': 178, 'performance_summary': 1241, 'sensitivity_summary': 11969, 'uncertainty_summary': 1723, 'trustworthiness_summary': 284, 'compute_summary': 35, 'research_question_coverage': 4}
Dashboard index rows: {'case_index': 1785, 'painting_index': 50, 'visual_asset_index': 23964, 'report_index': 104, 'filter_options': 16}


In [53]:
REPORT_ROLE_COUNTS = (
    REPORT_INDEX["report_role"]
    .value_counts()
    .to_dict()
)

REPORT_FORMAT_COUNTS = (
    REPORT_INDEX["format"]
    .value_counts()
    .to_dict()
)

MISSING_REPORT_PATHS = [
    path
    for path in REPORT_INDEX["report_path"]
    if not (PROJECT_ROOT / path).is_file()
]

SUMMARY_TABLE_COUNTS = {
    name: len(frame)
    for name, frame in DASHBOARD_TABLES.items()
}

SUMMARY_INDEX_COUNTS = {
    name: len(frame)
    for name, frame in DASHBOARD_INDEXES.items()
}

BATCH_9_CHECKS = (
    (
        "report_index_population",
        "All 104 upstream reports are indexed",
        104,
        len(REPORT_INDEX),
        len(REPORT_INDEX) == 104,
    ),
    (
        "report_format_population",
        "HTML and Markdown report counts match inventory",
        {"HTML": 94, "MARKDOWN": 10},
        REPORT_FORMAT_COUNTS,
        REPORT_FORMAT_COUNTS
        == {"HTML": 94, "MARKDOWN": 10},
    ),
    (
        "model_report_population",
        "All four self-contained model reports are indexed",
        4,
        int(
            REPORT_INDEX["report_role"]
            .eq("model_report")
            .sum()
        ),
        int(
            REPORT_INDEX["report_role"]
            .eq("model_report")
            .sum()
        ) == 4,
    ),
    (
        "case_report_population",
        "All 30 case reports are indexed",
        30,
        int(
            REPORT_INDEX["report_role"]
            .eq("case_report")
            .sum()
        ),
        int(
            REPORT_INDEX["report_role"]
            .eq("case_report")
            .sum()
        ) == 30,
    ),
    (
        "painting_report_population",
        "All 50 painting reports are indexed",
        50,
        int(
            REPORT_INDEX["report_role"]
            .eq("painting_report")
            .sum()
        ),
        int(
            REPORT_INDEX["report_role"]
            .eq("painting_report")
            .sum()
        ) == 50,
    ),
    (
        "final_report_population",
        "The final evaluation and limitations reports are indexed",
        {
            "final_evaluation_report": 1,
            "limitations_and_deviations": 1,
        },
        {
            role: REPORT_ROLE_COUNTS.get(role, 0)
            for role in [
                "final_evaluation_report",
                "limitations_and_deviations",
            ]
        },
        (
            REPORT_ROLE_COUNTS.get(
                "final_evaluation_report",
                0,
            ) == 1
            and REPORT_ROLE_COUNTS.get(
                "limitations_and_deviations",
                0,
            ) == 1
        ),
    ),
    (
        "report_identifier_uniqueness",
        "Report identifiers are unique",
        len(REPORT_INDEX),
        REPORT_INDEX["dashboard_report_id"].nunique(),
        (
            REPORT_INDEX["dashboard_report_id"].nunique()
            == len(REPORT_INDEX)
        ),
    ),
    (
        "report_paths_exist",
        "Every indexed report exists",
        [],
        MISSING_REPORT_PATHS,
        not MISSING_REPORT_PATHS,
    ),
    (
        "report_paths_relative",
        "Every report path is repository-relative",
        True,
        bool(
            REPORT_INDEX["report_path"]
            .map(is_repo_relative)
            .all()
        ),
        bool(
            REPORT_INDEX["report_path"]
            .map(is_repo_relative)
            .all()
        ),
    ),
    (
        "report_checksums_present",
        "Every report has a full checksum",
        0,
        int(
            normalized_text(
                REPORT_INDEX["report_sha256"]
            ).eq("").sum()
        ),
        not normalized_text(
            REPORT_INDEX["report_sha256"]
        ).eq("").any(),
    ),
    (
        "indexed_reports_self_contained",
        "All reports satisfy their recorded self-contained contract",
        {True},
        set(REPORT_INDEX["self_contained"]),
        set(REPORT_INDEX["self_contained"]) == {True},
    ),
    (
        "research_question_population",
        "All three research questions and practical output are covered",
        4,
        len(RESEARCH_QUESTION_COVERAGE),
        len(RESEARCH_QUESTION_COVERAGE) == 4,
    ),
    (
        "research_question_identifiers",
        "Research-question identifiers match the approved contract",
        {
            "rq1",
            "rq2",
            "rq3",
            "practical_output",
        },
        set(
            RESEARCH_QUESTION_COVERAGE[
                "research_question_id"
            ]
        ),
        (
            set(
                RESEARCH_QUESTION_COVERAGE[
                    "research_question_id"
                ]
            )
            == {
                "rq1",
                "rq2",
                "rq3",
                "practical_output",
            }
        ),
    ),
    (
        "summary_table_counts",
        "Dashboard summary records every table population",
        SUMMARY_TABLE_COUNTS,
        DASHBOARD_SUMMARY["table_rows"],
        (
            DASHBOARD_SUMMARY["table_rows"]
            == SUMMARY_TABLE_COUNTS
        ),
    ),
    (
        "summary_index_counts",
        "Dashboard summary records every dataframe index population",
        SUMMARY_INDEX_COUNTS,
        {
            key: DASHBOARD_SUMMARY["index_rows"][key]
            for key in SUMMARY_INDEX_COUNTS
        },
        (
            {
                key: DASHBOARD_SUMMARY[
                    "index_rows"
                ][key]
                for key in SUMMARY_INDEX_COUNTS
            }
            == SUMMARY_INDEX_COUNTS
        ),
    ),
    (
        "summary_scope_boundary",
        "Dashboard summary creates no new scientific evidence",
        False,
        DASHBOARD_SUMMARY[
            "creates_new_scientific_evidence"
        ],
        (
            DASHBOARD_SUMMARY[
                "creates_new_scientific_evidence"
            ]
            is False
        ),
    ),
    (
        "batch_9_status",
        "All Batch 9 dataframe records have valid status",
        {"ok"},
        (
            set(REPORT_INDEX["status"])
            | set(RESEARCH_QUESTION_COVERAGE["status"])
        ),
        (
            set(REPORT_INDEX["status"])
            | set(RESEARCH_QUESTION_COVERAGE["status"])
            == {"ok"}
        ),
    ),
    (
        "no_canonical_files_written",
        "Batch 9 has not persisted canonical dashboard files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_9_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_9_reports_summary",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else "Report, research-question, or dashboard-summary validation failed."
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_9_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_9_reports_summary"
        )
    ]
    .reset_index(drop=True)
)

display(REPORT_COVERAGE_VIEW)
display(RESEARCH_QUESTION_COVERAGE)
display(
    BATCH_9_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 9 completed.")
print("Indexed reports:", len(REPORT_INDEX))
print(
    "Research-question rows:",
    len(RESEARCH_QUESTION_COVERAGE),
)
print("Dashboard tables:", len(DASHBOARD_TABLES))
print("Dashboard dataframe indexes:", len(DASHBOARD_INDEXES))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,report_family,report_role,format,self_contained,reports,total_sections,embedded_images,embedded_tiles
0,analysis,analytical_report,HTML,True,8,103,250,0
1,case,case_report,HTML,True,30,390,330,810
2,final,final_evaluation_report,HTML,True,1,19,68,0
3,final,limitations_and_deviations,MARKDOWN,True,1,2,0,0
4,method,method_protocol,MARKDOWN,True,4,41,0,0
5,model,model_card,MARKDOWN,True,4,52,0,0
6,model,model_report,HTML,True,4,60,63,298
7,model,partial_evaluation_report,MARKDOWN,True,1,5,0,0
8,painting,painting_report,HTML,True,50,550,300,1334
9,report_collection,report_collection_index,HTML,True,1,6,2,0


,coverage_row_id,research_question_id,research_question,display_order,page_id,evidence_role,coverage_status,source_notebook_ids_json,source_paths_json,supported_interpretation,prohibited_interpretation,schema_version,status,issue
0,coverage_432aac6e25c8dee456703c02,rq1,Multi-metric evaluation beyond traditional ima...,1,metric_framework,"Metric families, canonical regions, disagreeme...",addressed_by_completed_evidence,"[""08"", ""13"", ""14"", ""15"", ""16"", ""17"", ""20"", ""21...","[""outputs/08_experiment_contracts_and_region_p...",Restoration quality requires complementary pix...,No individual metric or combined universal sco...,dashboard_package.v1,ok,
1,coverage_8cd2f439fb2eb96c80ebc845,rq2,Conditional model differences across paintings...,2,model_performance,Overall model comparison and controlled condit...,addressed_by_completed_evidence,"[""21"", ""23"", ""24"", ""25"", ""26"", ""30"", ""33""]","[""outputs/21_multi_model_comparison/metrics/mo...",LaMa is the strongest general baseline in this...,The result does not establish a universally be...,dashboard_package.v1,ok,
2,coverage_cf5184dac80235d4f1db99f7,rq3,Repeated-candidate uncertainty and speculative...,3,robustness_uncertainty,"Repeated-seed variability, uncertainty maps, d...",addressed_by_completed_evidence,"[""18"", ""19"", ""22"", ""27"", ""28"", ""29"", ""33""]","[""outputs/18_diffusion_uncertainty_analysis/me...",Repeated-seed variability identifies diffusion...,Empirical variability is not calibrated confid...,dashboard_package.v1,ok,
3,coverage_6af161c8ef5d217142ddc9e9,practical_output,Interpretable museum-oriented decision-support...,4,reports_reproducibility,"Complete case, painting, model, visual, report...",addressed_by_completed_evidence,"[""27"", ""29"", ""30"", ""31"", ""32"", ""33"", ""34""]","[""outputs/29_explainable_ai_and_case_retrieval...",The completed evidence is packaged as traceabl...,"The dashboard is not a restoration tool, exper...",dashboard_package.v1,ok,


,check_id,severity,passed,observed
0,report_index_population,blocking,True,104
1,report_format_population,blocking,True,"{""HTML"": 94, ""MARKDOWN"": 10}"
2,model_report_population,blocking,True,4
3,case_report_population,blocking,True,30
4,painting_report_population,blocking,True,50
5,final_report_population,blocking,True,"{""final_evaluation_report"": 1, ""limitations_an..."
6,report_identifier_uniqueness,blocking,True,104
7,report_paths_exist,blocking,True,[]
8,report_paths_relative,blocking,True,True
9,report_checksums_present,blocking,True,0


Batch 9 completed.
Indexed reports: 104
Research-question rows: 4
Dashboard tables: 9
Dashboard dataframe indexes: 4
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 9A — Grouped statistical, correlation, and ranking evidence

Package Notebook 26 evidence into the dashboard’s long-form sensitivity and analysis table.

This completion batch exposes:

- all grouped and statistical results;
- both correlation and rank-reversal evidence;
- baseline and sensitivity ranks;
- rank changes;
- winner retention and winner frequency;
- Kendall and Spearman ranking stability.

It does not recompute statistical evidence. It only normalizes validated Notebook 26 outputs for dashboard filtering and presentation.

In [54]:
STATISTICAL_RESULTS_SOURCE = (
    INPUT_TABLES["statistical_results_path"]
    .copy()
    .reset_index(drop=True)
)

METRIC_CORRELATIONS_SOURCE = (
    INPUT_TABLES["metric_correlations_path"]
    .copy()
    .reset_index(drop=True)
)

RANKING_STABILITY_SOURCE = (
    INPUT_TABLES["ranking_stability_path"]
    .copy()
    .reset_index(drop=True)
)


def n26_source_notebook_ids(record: dict) -> list[str]:
    """Preserve declared provenance and include Notebook 26."""

    declared = blank_safe_text(
        record.get("source_notebook_ids")
    )

    source_ids = ["26"]

    for token in declared.replace(",", "|").split("|"):
        notebook_id = token.strip()

        if notebook_id and notebook_id not in source_ids:
            source_ids.append(notebook_id)

    return source_ids


def optional_bool_number(value: object) -> float:
    """Encode a recorded Boolean as 1.0 or 0.0."""

    text = blank_safe_text(value).lower()

    if not text:
        return float("nan")

    if text in {"true", "1", "yes"}:
        return 1.0

    if text in {"false", "0", "no"}:
        return 0.0

    raise ValueError(
        f"Unsupported Boolean representation: {value!r}"
    )


def joined_dimension(
    left: object,
    right: object,
    separator: str = "__vs__",
) -> str:
    """Preserve both sides of a paired evidence dimension."""

    left_text = blank_safe_text(left)
    right_text = blank_safe_text(right)

    if left_text and right_text:
        return f"{left_text}{separator}{right_text}"

    return left_text or right_text


print(
    "Notebook 26 source rows:",
    {
        "statistical_results": len(
            STATISTICAL_RESULTS_SOURCE
        ),
        "metric_correlations": len(
            METRIC_CORRELATIONS_SOURCE
        ),
        "ranking_stability": len(
            RANKING_STABILITY_SOURCE
        ),
    },
)
print("Canonical files persisted by this cell: 0")

Notebook 26 source rows: {'statistical_results': 4174, 'metric_correlations': 504, 'ranking_stability': 258}
Canonical files persisted by this cell: 0


In [55]:
N26_ANALYSIS_ROWS = []

STATISTICAL_SOURCE_PATH = SETTINGS["inputs"][
    "statistical_results_path"
]
CORRELATION_SOURCE_PATH = SETTINGS["inputs"][
    "metric_correlations_path"
]
RANKING_SOURCE_PATH = SETTINGS["inputs"][
    "ranking_stability_path"
]


for row in STATISTICAL_RESULTS_SOURCE.itertuples(
    index=False
):
    record = row._asdict()

    condition_field, condition_value = source_condition(
        record,
        [
            "analysis_family_id",
            "scope_type",
            "scope_value",
            "population_id",
            "painting_id",
            "category",
            "damage_or_degradation_type",
            "target_damage_fraction",
        ],
    )

    result_kind = blank_safe_text(
        record.get("result_kind")
    )
    interpretation_status = blank_safe_text(
        record.get("interpretation_status")
    )

    N26_ANALYSIS_ROWS.append(
        {
            "sensitivity_row_id": stable_id(
                "dashboard_statistical",
                record["result_id"],
            ),
            "page_id": "model_performance",
            "section_id": "grouped_statistical_analysis",
            "display_order": 0,
            "analysis_family": (
                "grouped_statistical_analysis"
            ),
            "analysis_kind": result_kind,
            "experiment_id": blank_safe_text(
                record.get("experiment_id")
            ),
            "condition_field": condition_field,
            "condition_value": condition_value,
            "condition_order": source_condition_order(
                record
            ),
            "evidence_family": blank_safe_text(
                record.get("evidence_family")
            ),
            "metric_family": "",
            "metric_name": blank_safe_text(
                record.get("metric_name")
            ),
            "feature_model_id": blank_safe_text(
                record.get("feature_model_id")
            ),
            "region_id": blank_safe_text(
                record.get("region_id")
            ),
            "summary_statistic": blank_safe_text(
                record.get("summary_statistic")
            ),
            "comparison_direction": blank_safe_text(
                record.get("comparison_direction")
            ),
            "model_id": blank_safe_text(
                record.get("model_id")
            ),
            "comparison_model_id": blank_safe_text(
                record.get("comparison_model_id")
            ),
            "estimate_name": blank_safe_text(
                record.get("estimate_name")
            ),
            "estimate": float(record["estimate"]),
            "interval_low": optional_float(
                record.get("ci_lower")
            ),
            "interval_high": optional_float(
                record.get("ci_upper")
            ),
            "effect_size_name": blank_safe_text(
                record.get("effect_size_name")
            ),
            "effect_size": optional_float(
                record.get("effect_size")
            ),
            "p_value": optional_float(
                record.get("p_value")
            ),
            "q_value": optional_float(
                record.get("q_value")
            ),
            "independent_unit": blank_safe_text(
                record.get("independent_unit")
            ),
            "n_paintings": optional_count(
                record.get("n_paintings")
            ),
            "n_cases": optional_count(
                record.get("n_cases")
            ),
            "n_observations": optional_count(
                record.get("n_observations")
            ),
            "applicability_status": blank_safe_text(
                record.get("applicability_status")
            ),
            "source_notebook_ids_json": json_list(
                n26_source_notebook_ids(record)
            ),
            "source_paths_json": json_list(
                [STATISTICAL_SOURCE_PATH]
            ),
            "interpretation": (
                f"{readable_identifier(result_kind)}. "
                f"Notebook 26 interpretation: "
                f"{readable_identifier(interpretation_status)}."
            ),
            "schema_version": (
                DASHBOARD_PACKAGE_SCHEMA_VERSION
            ),
            "status": "ok",
            "issue": "",
        }
    )


for row in METRIC_CORRELATIONS_SOURCE.itertuples(
    index=False
):
    record = row._asdict()

    condition_field, condition_value = source_condition(
        record,
        [
            "analysis_family_id",
            "scope_type",
            "scope_value",
            "agreement_status",
        ],
    )

    correlation_kind = blank_safe_text(
        record.get("correlation_kind")
    )
    evidence_pair = joined_dimension(
        record.get("left_evidence_family"),
        record.get("right_evidence_family"),
    )
    metric_pair = joined_dimension(
        record.get("left_metric_name"),
        record.get("right_metric_name"),
    )
    region_pair = joined_dimension(
        record.get("left_region_id"),
        record.get("right_region_id"),
    )
    interpretation_status = blank_safe_text(
        record.get("interpretation_status")
    )

    correlation_measures = [
        {
            "estimate_name": (
                blank_safe_text(
                    record.get("correlation_method")
                )
                or "correlation"
            ),
            "estimate": float(record["correlation"]),
            "comparison_direction": (
                "higher_absolute_is_stronger"
            ),
            "p_value": optional_float(
                record.get("p_value")
            ),
            "q_value": optional_float(
                record.get("q_value")
            ),
        },
        {
            "estimate_name": "rank_reversal_fraction",
            "estimate": float(
                record["rank_reversal_fraction"]
            ),
            "comparison_direction": "lower_is_better",
            "p_value": float("nan"),
            "q_value": float("nan"),
        },
    ]

    for measure in correlation_measures:
        N26_ANALYSIS_ROWS.append(
            {
                "sensitivity_row_id": stable_id(
                    "dashboard_correlation",
                    record["correlation_id"],
                    measure["estimate_name"],
                ),
                "page_id": "metric_framework",
                "section_id": "metric_correlations",
                "display_order": 0,
                "analysis_family": "metric_correlations",
                "analysis_kind": correlation_kind,
                "experiment_id": blank_safe_text(
                    record.get("experiment_id")
                ),
                "condition_field": condition_field,
                "condition_value": condition_value,
                "condition_order": float("nan"),
                "evidence_family": evidence_pair,
                "metric_family": "",
                "metric_name": metric_pair,
                "feature_model_id": "",
                "region_id": region_pair,
                "summary_statistic": blank_safe_text(
                    record.get("correlation_method")
                ),
                "comparison_direction": measure[
                    "comparison_direction"
                ],
                "model_id": blank_safe_text(
                    record.get("model_id")
                ),
                "comparison_model_id": blank_safe_text(
                    record.get("comparison_model_id")
                ),
                "estimate_name": measure[
                    "estimate_name"
                ],
                "estimate": measure["estimate"],
                "interval_low": float("nan"),
                "interval_high": float("nan"),
                "effect_size_name": "",
                "effect_size": float("nan"),
                "p_value": measure["p_value"],
                "q_value": measure["q_value"],
                "independent_unit": blank_safe_text(
                    record.get("independent_unit")
                ),
                "n_paintings": optional_count(
                    record.get("n_paintings")
                ),
                "n_cases": optional_count(
                    record.get("n_cases")
                ),
                "n_observations": optional_count(
                    record.get("n_observations")
                ),
                "applicability_status": blank_safe_text(
                    record.get("applicability_status")
                ),
                "source_notebook_ids_json": json_list(
                    n26_source_notebook_ids(record)
                ),
                "source_paths_json": json_list(
                    [CORRELATION_SOURCE_PATH]
                ),
                "interpretation": (
                    f"{readable_identifier(correlation_kind)}: "
                    f"{readable_identifier(interpretation_status)}. "
                    f"The stored measure is "
                    f"{readable_identifier(measure['estimate_name'])}."
                ),
                "schema_version": (
                    DASHBOARD_PACKAGE_SCHEMA_VERSION
                ),
                "status": "ok",
                "issue": "",
            }
        )


RANKING_MEASURE_SPECS = [
    (
        "baseline_rank",
        "lower_rank_is_better",
    ),
    (
        "sensitivity_rank",
        "lower_rank_is_better",
    ),
    (
        "rank_change",
        "closer_to_zero_is_more_stable",
    ),
    (
        "winner_retained",
        "one_means_winner_retained",
    ),
    (
        "winner_frequency",
        "higher_is_more_stable",
    ),
    (
        "kendalls_tau",
        "higher_is_more_stable",
    ),
    (
        "spearman_rho",
        "higher_is_more_stable",
    ),
]


for row in RANKING_STABILITY_SOURCE.itertuples(
    index=False
):
    record = row._asdict()

    condition_field, condition_value = source_condition(
        record,
        [
            "analysis_family_id",
            "scope_type",
            "scope_value",
            "omitted_unit_type",
            "omitted_unit_id",
            "evidence_family",
            "region_policy_id",
        ],
    )

    ranking_kind = blank_safe_text(
        record.get("ranking_kind")
    )
    interpretation_status = blank_safe_text(
        record.get("interpretation_status")
    )

    for measure_name, comparison_direction in (
        RANKING_MEASURE_SPECS
    ):
        if measure_name == "winner_retained":
            estimate = optional_bool_number(
                record.get(measure_name)
            )
        else:
            estimate = optional_float(
                record.get(measure_name)
            )

        if pd.isna(estimate):
            continue

        N26_ANALYSIS_ROWS.append(
            {
                "sensitivity_row_id": stable_id(
                    "dashboard_ranking",
                    record["ranking_id"],
                    measure_name,
                ),
                "page_id": "model_performance",
                "section_id": "ranking_stability",
                "display_order": 0,
                "analysis_family": "ranking_stability",
                "analysis_kind": ranking_kind,
                "experiment_id": blank_safe_text(
                    record.get("experiment_id")
                ),
                "condition_field": condition_field,
                "condition_value": condition_value,
                "condition_order": float("nan"),
                "evidence_family": blank_safe_text(
                    record.get("evidence_family")
                ),
                "metric_family": "ranking_stability",
                "metric_name": "model_ranking",
                "feature_model_id": "",
                "region_id": blank_safe_text(
                    record.get("region_policy_id")
                ),
                "summary_statistic": measure_name,
                "comparison_direction": (
                    comparison_direction
                ),
                "model_id": blank_safe_text(
                    record.get("model_id")
                ),
                "comparison_model_id": blank_safe_text(
                    record.get("winner_model_id")
                ),
                "estimate_name": measure_name,
                "estimate": float(estimate),
                "interval_low": float("nan"),
                "interval_high": float("nan"),
                "effect_size_name": "",
                "effect_size": float("nan"),
                "p_value": float("nan"),
                "q_value": float("nan"),
                "independent_unit": blank_safe_text(
                    record.get("independent_unit")
                ),
                "n_paintings": optional_count(
                    record.get("n_paintings")
                ),
                "n_cases": optional_count(
                    record.get("n_cases")
                ),
                "n_observations": optional_count(
                    record.get("n_candidates")
                ),
                "applicability_status": blank_safe_text(
                    record.get("applicability_status")
                ),
                "source_notebook_ids_json": json_list(
                    n26_source_notebook_ids(record)
                ),
                "source_paths_json": json_list(
                    [RANKING_SOURCE_PATH]
                ),
                "interpretation": (
                    f"{readable_identifier(ranking_kind)}: "
                    f"{readable_identifier(interpretation_status)}. "
                    f"The stored measure is "
                    f"{readable_identifier(measure_name)}."
                ),
                "schema_version": (
                    DASHBOARD_PACKAGE_SCHEMA_VERSION
                ),
                "status": "ok",
                "issue": "",
            }
        )


N26_ANALYSIS_SUMMARY = pd.DataFrame(N26_ANALYSIS_ROWS)

N26_ANALYSIS_SUMMARY = validate_output_frame(
    N26_ANALYSIS_SUMMARY,
    "sensitivity_summary",
)

print(
    "Normalized Notebook 26 rows:",
    len(N26_ANALYSIS_SUMMARY),
)
print(
    "Normalized family counts:",
    N26_ANALYSIS_SUMMARY["analysis_family"]
    .value_counts()
    .to_dict(),
)
print("Canonical files persisted by this cell: 0")

Normalized Notebook 26 rows: 6982
Normalized family counts: {'grouped_statistical_analysis': 4174, 'ranking_stability': 1800, 'metric_correlations': 1008}
Canonical files persisted by this cell: 0


In [56]:
N26_ANALYSIS_FAMILIES = {
    "grouped_statistical_analysis",
    "metric_correlations",
    "ranking_stability",
}

BASE_SENSITIVITY_SUMMARY = (
    SENSITIVITY_SUMMARY.loc[
        ~SENSITIVITY_SUMMARY["analysis_family"].isin(
            N26_ANALYSIS_FAMILIES
        )
    ]
    .copy()
    .reset_index(drop=True)
)

SENSITIVITY_SUMMARY = pd.concat(
    [
        BASE_SENSITIVITY_SUMMARY,
        N26_ANALYSIS_SUMMARY,
    ],
    ignore_index=True,
)

SENSITIVITY_SUMMARY = (
    SENSITIVITY_SUMMARY
    .sort_values(
        [
            "analysis_family",
            "analysis_kind",
            "condition_order",
            "condition_value",
            "metric_name",
            "region_id",
            "model_id",
            "comparison_model_id",
            "estimate_name",
        ],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

SENSITIVITY_SUMMARY["display_order"] = (
    SENSITIVITY_SUMMARY.index + 1
)

SENSITIVITY_SUMMARY = validate_output_frame(
    SENSITIVITY_SUMMARY,
    "sensitivity_summary",
)

SENSITIVITY_KIND_SUMMARY = (
    SENSITIVITY_SUMMARY.groupby(
        [
            "analysis_family",
            "analysis_kind",
            "applicability_status",
        ],
        dropna=False,
        sort=True,
    )
    .agg(
        rows=("sensitivity_row_id", "size"),
        models=("model_id", "nunique"),
        metrics=("metric_name", "nunique"),
        regions=("region_id", "nunique"),
        maximum_paintings=("n_paintings", "max"),
        maximum_cases=("n_cases", "max"),
        maximum_observations=(
            "n_observations",
            "max",
        ),
    )
    .reset_index()
)

display(
    SENSITIVITY_KIND_SUMMARY.loc[
        SENSITIVITY_KIND_SUMMARY[
            "analysis_family"
        ].isin(N26_ANALYSIS_FAMILIES)
    ]
)

print(
    "Completed sensitivity/analysis rows:",
    len(SENSITIVITY_SUMMARY),
)
print(
    "Completed family counts:",
    SENSITIVITY_SUMMARY["analysis_family"]
    .value_counts()
    .to_dict(),
)
print("Canonical files persisted by this cell: 0")

,analysis_family,analysis_kind,applicability_status,rows,models,metrics,regions,maximum_paintings,maximum_cases,maximum_observations
12,grouped_statistical_analysis,applicability_audit,descriptive_only,1,1,1,1,0,0,10
13,grouped_statistical_analysis,applicability_audit,not_applicable,2,1,2,1,0,0,0
14,grouped_statistical_analysis,applicability_audit,not_applicable_single_dataset,1,1,1,1,0,0,1
15,grouped_statistical_analysis,canonical_category_summary,supported,165,3,11,4,10,40,40
16,grouped_statistical_analysis,canonical_category_test,supported_canonical_only,33,3,11,4,50,200,50
17,grouped_statistical_analysis,canonical_damage_summary,supported,132,3,11,4,50,50,50
18,grouped_statistical_analysis,canonical_damage_test,supported_canonical_only,33,3,11,4,50,200,50
19,grouped_statistical_analysis,evidence_availability,supported,5,1,5,1,0,0,369720
20,grouped_statistical_analysis,focused_experiment_summary,applicable,3239,7,35,10,5,75,165
21,grouped_statistical_analysis,focused_experiment_summary,applicable_computational_contrast,66,3,11,4,5,10,5


Completed sensitivity/analysis rows: 18951
Completed family counts: {'mask_robustness': 5373, 'synthetic_degradation': 4695, 'grouped_statistical_analysis': 4174, 'damage_size_sensitivity': 1901, 'ranking_stability': 1800, 'metric_correlations': 1008}
Canonical files persisted by this cell: 0


In [57]:
EXPECTED_N26_NORMALIZED_COUNTS = {
    "grouped_statistical_analysis": 4174,
    "metric_correlations": 1008,
    "ranking_stability": 1800,
}

OBSERVED_N26_NORMALIZED_COUNTS = (
    SENSITIVITY_SUMMARY.loc[
        SENSITIVITY_SUMMARY["analysis_family"].isin(
            N26_ANALYSIS_FAMILIES
        ),
        "analysis_family",
    ]
    .value_counts()
    .to_dict()
)

EXPECTED_COMPLETE_SENSITIVITY_ROWS = 18951

N26_ESTIMATES_FINITE = bool(
    pd.to_numeric(
        N26_ANALYSIS_SUMMARY["estimate"],
        errors="coerce",
    )
    .notna()
    .all()
)

N26_SOURCE_PATHS_RELATIVE = (
    all_source_paths_are_relative(
        N26_ANALYSIS_SUMMARY["source_paths_json"]
    )
)

N26_IDENTIFIER_COUNT = (
    N26_ANALYSIS_SUMMARY[
        "sensitivity_row_id"
    ].nunique()
)

BATCH_9A_CHECKS = (
    (
        "notebook_26_normalized_counts",
        "Every required Notebook 26 measure is normalized",
        EXPECTED_N26_NORMALIZED_COUNTS,
        OBSERVED_N26_NORMALIZED_COUNTS,
        (
            OBSERVED_N26_NORMALIZED_COUNTS
            == EXPECTED_N26_NORMALIZED_COUNTS
        ),
    ),
    (
        "complete_analysis_row_count",
        "The completed analysis table has 18,951 rows",
        EXPECTED_COMPLETE_SENSITIVITY_ROWS,
        len(SENSITIVITY_SUMMARY),
        (
            len(SENSITIVITY_SUMMARY)
            == EXPECTED_COMPLETE_SENSITIVITY_ROWS
        ),
    ),
    (
        "statistical_source_coverage",
        "All 4,174 statistical source rows are represented",
        4174,
        int(
            N26_ANALYSIS_SUMMARY[
                "analysis_family"
            ].eq(
                "grouped_statistical_analysis"
            ).sum()
        ),
        int(
            N26_ANALYSIS_SUMMARY[
                "analysis_family"
            ].eq(
                "grouped_statistical_analysis"
            ).sum()
        ) == 4174,
    ),
    (
        "correlation_measure_coverage",
        "Both measures from all 504 correlations are represented",
        1008,
        int(
            N26_ANALYSIS_SUMMARY[
                "analysis_family"
            ].eq("metric_correlations").sum()
        ),
        int(
            N26_ANALYSIS_SUMMARY[
                "analysis_family"
            ].eq("metric_correlations").sum()
        ) == 1008,
    ),
    (
        "ranking_measure_coverage",
        "All recorded ranking-stability measures are represented",
        1800,
        int(
            N26_ANALYSIS_SUMMARY[
                "analysis_family"
            ].eq("ranking_stability").sum()
        ),
        int(
            N26_ANALYSIS_SUMMARY[
                "analysis_family"
            ].eq("ranking_stability").sum()
        ) == 1800,
    ),
    (
        "notebook_26_identifiers_unique",
        "Notebook 26 dashboard identifiers are unique",
        len(N26_ANALYSIS_SUMMARY),
        N26_IDENTIFIER_COUNT,
        (
            N26_IDENTIFIER_COUNT
            == len(N26_ANALYSIS_SUMMARY)
        ),
    ),
    (
        "completed_analysis_identifiers_unique",
        "All completed analysis identifiers are unique",
        len(SENSITIVITY_SUMMARY),
        SENSITIVITY_SUMMARY[
            "sensitivity_row_id"
        ].nunique(),
        (
            SENSITIVITY_SUMMARY[
                "sensitivity_row_id"
            ].nunique()
            == len(SENSITIVITY_SUMMARY)
        ),
    ),
    (
        "notebook_26_estimates_finite",
        "Every normalized Notebook 26 measure is finite",
        True,
        N26_ESTIMATES_FINITE,
        N26_ESTIMATES_FINITE,
    ),
    (
        "notebook_26_sources_relative",
        "Notebook 26 source paths are repository-relative",
        True,
        N26_SOURCE_PATHS_RELATIVE,
        N26_SOURCE_PATHS_RELATIVE,
    ),
    (
        "notebook_26_status",
        "Every normalized Notebook 26 row has valid status",
        {"ok"},
        set(N26_ANALYSIS_SUMMARY["status"]),
        (
            set(N26_ANALYSIS_SUMMARY["status"])
            == {"ok"}
        ),
    ),
    (
        "no_canonical_files_written",
        "Batch 9A has not persisted canonical files",
        0,
        sum(
            1
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        ),
        not any(
            path.is_file()
            for path in OUTPUT_ROOT.rglob("*")
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    BATCH_9A_CHECKS
):
    VALIDATION.add(
        validation_stage=(
            "batch_9a_grouped_analysis_completion"
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else (
                "Notebook 26 evidence is not completely "
                "represented in the dashboard package."
            )
        ),
    )

VALIDATION.raise_for_blocking()

DASHBOARD_TABLES["sensitivity_summary"] = (
    SENSITIVITY_SUMMARY
)

DASHBOARD_SUMMARY_GENERATED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

DASHBOARD_SUMMARY["generated_at_utc"] = (
    DASHBOARD_SUMMARY_GENERATED_AT_UTC
)
DASHBOARD_SUMMARY["table_rows"] = {
    name: len(frame)
    for name, frame in DASHBOARD_TABLES.items()
}

BATCH_9A_VALIDATION = (
    VALIDATION.to_dataframe()
    .loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_9a_grouped_analysis_completion"
        )
    ]
    .reset_index(drop=True)
)

display(
    BATCH_9A_VALIDATION[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Batch 9A completed.")
print(
    "Notebook 26 normalized rows:",
    len(N26_ANALYSIS_SUMMARY),
)
print(
    "Final sensitivity/analysis rows:",
    len(SENSITIVITY_SUMMARY),
)
print(
    "Updated dashboard table rows:",
    DASHBOARD_SUMMARY["table_rows"],
)
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Canonical files persisted by this batch: 0")

,check_id,severity,passed,observed
0,notebook_26_normalized_counts,blocking,True,"{""grouped_statistical_analysis"": 4174, ""metric..."
1,complete_analysis_row_count,blocking,True,18951
2,statistical_source_coverage,blocking,True,4174
3,correlation_measure_coverage,blocking,True,1008
4,ranking_measure_coverage,blocking,True,1800
5,notebook_26_identifiers_unique,blocking,True,6982
6,completed_analysis_identifiers_unique,blocking,True,18951
7,notebook_26_estimates_finite,blocking,True,True
8,notebook_26_sources_relative,blocking,True,True
9,notebook_26_status,blocking,True,"""{'ok'}"""


Batch 9A completed.
Notebook 26 normalized rows: 6982
Final sensitivity/analysis rows: 18951
Updated dashboard table rows: {'headline_findings': 8, 'study_design': 35, 'metric_framework': 178, 'performance_summary': 1241, 'sensitivity_summary': 18951, 'uncertainty_summary': 1723, 'trustworthiness_summary': 284, 'compute_summary': 35, 'research_question_coverage': 4}
Blocking failures: 0
Canonical files persisted by this batch: 0


## Batch 10 — Persistence, manifests, and completion gate

Persist the complete dashboard asset package, reload every canonical output, validate checksums and schemas, and record roadmap traceability.

The package contains:

- nine normalized dashboard tables;
- four CSV indexes;
- one filter-options JSON index;
- one dashboard summary;
- one dashboard asset manifest;
- one consolidated validation table;
- one universal artifact manifest;
- one completed run manifest.

Notebook 34 packages existing validated evidence. It does not run restoration inference, calculate new scientific metrics, or perform new statistical tests.

In [58]:
def json_paths_have_content(
    frame: pd.DataFrame,
    column: str,
) -> int:
    """Count records containing at least one referenced path."""

    return int(
        frame[column]
        .map(lambda value: len(parse_json_list(value)) > 0)
        .sum()
    )


ROADMAP_TRACEABILITY_ROWS = []


def add_roadmap_trace(
    responsibility_id: str,
    responsibility: str,
    dashboard_evidence: str,
    observed_count: int,
    passed: bool,
) -> None:
    ROADMAP_TRACEABILITY_ROWS.append(
        {
            "responsibility_id": responsibility_id,
            "responsibility": responsibility,
            "dashboard_evidence": dashboard_evidence,
            "observed_count": int(observed_count),
            "passed": bool(passed),
        }
    )


add_roadmap_trace(
    "overview",
    "Overview and headline findings",
    "headline_findings",
    len(HEADLINE_FINDINGS),
    len(HEADLINE_FINDINGS) == 8,
)

add_roadmap_trace(
    "dataset_and_bias",
    "Dataset composition and bias evidence",
    "study_design and painting_index",
    len(PAINTING_INDEX),
    len(PAINTING_INDEX) == 50,
)

add_roadmap_trace(
    "canonical_damage",
    "Canonical damage design",
    "study_design and case_index",
    len(INPUT_TABLES["canonical_cases_path"]),
    len(INPUT_TABLES["canonical_cases_path"]) == 250,
)

add_roadmap_trace(
    "damage_size_sensitivity",
    "Damage-size sensitivity",
    "sensitivity_summary",
    int(
        SENSITIVITY_SUMMARY["analysis_family"]
        .eq("damage_size_sensitivity")
        .sum()
    ),
    int(
        SENSITIVITY_SUMMARY["analysis_family"]
        .eq("damage_size_sensitivity")
        .sum()
    ) == 1901,
)

add_roadmap_trace(
    "mask_robustness",
    "Mask robustness",
    "sensitivity_summary",
    int(
        SENSITIVITY_SUMMARY["analysis_family"]
        .eq("mask_robustness")
        .sum()
    ),
    int(
        SENSITIVITY_SUMMARY["analysis_family"]
        .eq("mask_robustness")
        .sum()
    ) == 5373,
)

add_roadmap_trace(
    "synthetic_degradation",
    "Synthetic-degradation sensitivity",
    "sensitivity_summary",
    int(
        SENSITIVITY_SUMMARY["analysis_family"]
        .eq("synthetic_degradation")
        .sum()
    ),
    int(
        SENSITIVITY_SUMMARY["analysis_family"]
        .eq("synthetic_degradation")
        .sum()
    ) == 4695,
)

add_roadmap_trace(
    "model_stack",
    "Complete evaluated model stack",
    "compute_summary and report_index",
    COMPUTE_SUMMARY["model_id"].nunique(),
    COMPUTE_SUMMARY["model_id"].nunique() == 4,
)

add_roadmap_trace(
    "model_comparison",
    "Overall and conditional model comparison",
    "performance_summary",
    len(PERFORMANCE_SUMMARY),
    len(PERFORMANCE_SUMMARY) == 1241,
)

add_roadmap_trace(
    "metric_region_policy",
    "Metric-region policy",
    "metric_framework",
    len(INPUT_TABLES["region_policy_path"]),
    len(INPUT_TABLES["region_policy_path"]) == 143,
)

add_roadmap_trace(
    "metric_ablation",
    "Metric and region-policy ablation",
    "trustworthiness_summary",
    len(INPUT_TABLES["ablation_results_path"]),
    len(INPUT_TABLES["ablation_results_path"]) == 7710,
)

add_roadmap_trace(
    "texture_diagnostics",
    "Texture diagnostics",
    "case_index and visual_asset_index",
    json_paths_have_content(
        CASE_INDEX,
        "texture_paths_json",
    ),
    json_paths_have_content(
        CASE_INDEX,
        "texture_paths_json",
    ) > 0,
)

add_roadmap_trace(
    "colour_diagnostics",
    "Colour diagnostics",
    "case_index and visual_asset_index",
    json_paths_have_content(
        CASE_INDEX,
        "colour_paths_json",
    ),
    json_paths_have_content(
        CASE_INDEX,
        "colour_paths_json",
    ) > 0,
)

add_roadmap_trace(
    "seam_diagnostics",
    "Seam and boundary diagnostics",
    "case_index and visual_asset_index",
    json_paths_have_content(
        CASE_INDEX,
        "seam_paths_json",
    ),
    json_paths_have_content(
        CASE_INDEX,
        "seam_paths_json",
    ) > 0,
)

add_roadmap_trace(
    "uncertainty_summaries",
    "Repeated-seed uncertainty summaries",
    "uncertainty_summary",
    len(UNCERTAINTY_SUMMARY),
    len(UNCERTAINTY_SUMMARY) == 1723,
)

UNCERTAINTY_VISUAL_COUNT = int(
    VISUAL_ASSET_INDEX["source_notebook_id"]
    .astype(str)
    .isin({"19", "22"})
    .sum()
)

add_roadmap_trace(
    "uncertainty_heatmaps",
    "Uncertainty heatmaps",
    "visual_asset_index",
    UNCERTAINTY_VISUAL_COUNT,
    UNCERTAINTY_VISUAL_COUNT > 0,
)

SEMANTIC_VISUAL_COUNT = int(
    VISUAL_ASSET_INDEX["source_notebook_id"]
    .astype(str)
    .eq("20")
    .sum()
)

add_roadmap_trace(
    "semantic_consistency",
    "Semantic and structural consistency",
    "case_index and visual_asset_index",
    SEMANTIC_VISUAL_COUNT,
    SEMANTIC_VISUAL_COUNT > 0,
)

XAI_VISUAL_COUNT = int(
    VISUAL_ASSET_INDEX["source_notebook_id"]
    .astype(str)
    .isin({"16", "17", "19", "20"})
    .sum()
)

add_roadmap_trace(
    "xai_maps",
    "Difference, local, uncertainty, and semantic maps",
    "visual_asset_index",
    XAI_VISUAL_COUNT,
    XAI_VISUAL_COUNT > 0,
)

FLAG_RECORD_COUNT = int(
    TRUSTWORTHINESS_SUMMARY["record_type"]
    .astype(str)
    .str.contains("flag", case=False)
    .sum()
)

add_roadmap_trace(
    "trustworthiness_flags",
    "Separate trustworthiness flags",
    "trustworthiness_summary",
    FLAG_RECORD_COUNT,
    FLAG_RECORD_COUNT > 0,
)

FAILURE_RECORD_COUNT = int(
    TRUSTWORTHINESS_SUMMARY["record_type"]
    .astype(str)
    .str.contains(
        "taxonomy|failure",
        case=False,
        regex=True,
    )
    .sum()
)

add_roadmap_trace(
    "failure_taxonomy",
    "Failure taxonomy and assignments",
    "trustworthiness_summary",
    FAILURE_RECORD_COUNT,
    FAILURE_RECORD_COUNT > 0,
)

GROUPED_ANALYSIS_COUNT = int(
    SENSITIVITY_SUMMARY["analysis_family"]
    .eq("grouped_statistical_analysis")
    .sum()
)

add_roadmap_trace(
    "grouped_statistical_analysis",
    "Grouped and statistical analysis",
    "sensitivity_summary",
    GROUPED_ANALYSIS_COUNT,
    GROUPED_ANALYSIS_COUNT == 4174,
)

add_roadmap_trace(
    "compute_scalability",
    "Compute and scalability evidence",
    "compute_summary",
    len(COMPUTE_SUMMARY),
    len(COMPUTE_SUMMARY) == 35,
)

MODEL_REPORT_COUNT = int(
    REPORT_INDEX["report_role"]
    .eq("model_report")
    .sum()
)

add_roadmap_trace(
    "model_cards",
    "Model cards and model reports",
    "compute_summary and report_index",
    MODEL_REPORT_COUNT,
    MODEL_REPORT_COUNT == 4,
)

CASE_AND_PAINTING_REPORT_COUNT = int(
    REPORT_INDEX["report_role"]
    .isin({"case_report", "painting_report"})
    .sum()
)

add_roadmap_trace(
    "case_and_painting_reports",
    "Case and painting reports",
    "report_index",
    CASE_AND_PAINTING_REPORT_COUNT,
    CASE_AND_PAINTING_REPORT_COUNT == 80,
)

add_roadmap_trace(
    "reproducibility",
    "Validation and reproducibility evidence",
    "report_index, dashboard_assets, and run_manifest",
    len(UPSTREAM_MANIFESTS),
    len(UPSTREAM_MANIFESTS) == 33,
)

FINAL_REPORT_COUNT = int(
    REPORT_INDEX["report_role"]
    .isin(
        {
            "final_evaluation_report",
            "limitations_and_deviations",
        }
    )
    .sum()
)

add_roadmap_trace(
    "final_reports",
    "Final evaluation and limitations reports",
    "report_index",
    FINAL_REPORT_COUNT,
    FINAL_REPORT_COUNT == 2,
)


ROADMAP_TRACEABILITY = pd.DataFrame(
    ROADMAP_TRACEABILITY_ROWS
)

if len(ROADMAP_TRACEABILITY) != 25:
    raise ValueError(
        "Expected 25 roadmap responsibilities, "
        f"observed {len(ROADMAP_TRACEABILITY)}."
    )

if not ROADMAP_TRACEABILITY["passed"].all():
    display(
        ROADMAP_TRACEABILITY.loc[
            ~ROADMAP_TRACEABILITY["passed"]
        ]
    )
    raise ValueError(
        "One or more Notebook 34 roadmap "
        "responsibilities are not covered."
    )

DASHBOARD_SUMMARY["roadmap_traceability"] = (
    ROADMAP_TRACEABILITY.to_dict(orient="records")
)

display(ROADMAP_TRACEABILITY)

print("Roadmap responsibilities covered: 25/25")
print("Canonical files persisted by this cell: 0")

,responsibility_id,responsibility,dashboard_evidence,observed_count,passed
0,overview,Overview and headline findings,headline_findings,8,True
1,dataset_and_bias,Dataset composition and bias evidence,study_design and painting_index,50,True
2,canonical_damage,Canonical damage design,study_design and case_index,250,True
3,damage_size_sensitivity,Damage-size sensitivity,sensitivity_summary,1901,True
4,mask_robustness,Mask robustness,sensitivity_summary,5373,True
5,synthetic_degradation,Synthetic-degradation sensitivity,sensitivity_summary,4695,True
6,model_stack,Complete evaluated model stack,compute_summary and report_index,4,True
7,model_comparison,Overall and conditional model comparison,performance_summary,1241,True
8,metric_region_policy,Metric-region policy,metric_framework,143,True
9,metric_ablation,Metric and region-policy ablation,trustworthiness_summary,7710,True


Roadmap responsibilities covered: 25/25
Canonical files persisted by this cell: 0


In [59]:
from restoration_eval.dashboard_assets import (
    atomic_write_csv,
    atomic_write_json,
    path_statistics as dashboard_path_statistics,
    sha256_file as dashboard_sha256_file,
)
from restoration_eval.manifests import (
    ARTIFACT_MANIFEST_SCHEMA_VERSION,
    MANIFESTS_MODULE_VERSION,
    artifact_records_dataframe,
    build_artifact_record,
    build_run_manifest,
    configuration_checksums,
    sha256_path as manifest_sha256_path,
)


TABLE_OUTPUT_KEYS = {
    "headline_findings": "headline_findings_path",
    "study_design": "study_design_path",
    "metric_framework": "metric_framework_path",
    "performance_summary": "performance_summary_path",
    "sensitivity_summary": "sensitivity_summary_path",
    "uncertainty_summary": "uncertainty_summary_path",
    "trustworthiness_summary": (
        "trustworthiness_summary_path"
    ),
    "compute_summary": "compute_summary_path",
    "research_question_coverage": (
        "research_question_coverage_path"
    ),
}

INDEX_OUTPUT_KEYS = {
    "case_index": "case_index_path",
    "painting_index": "painting_index_path",
    "visual_asset_index": "visual_asset_index_path",
    "report_index": "report_index_path",
}

EXPECTED_FINAL_COUNTS = {
    "dashboard_tables": 9,
    "dashboard_indexes": 5,
    "dashboard_data_assets": 15,
    "dashboard_asset_records": 15,
    "artifact_records": 17,
    "physical_output_files": 19,
    "roadmap_responsibilities": 25,
    "headline_findings": 8,
    "study_design_rows": 35,
    "metric_framework_rows": 178,
    "performance_summary_rows": 1241,
    "sensitivity_summary_rows": 18951,
    "uncertainty_summary_rows": 1723,
    "trustworthiness_summary_rows": 284,
    "compute_summary_rows": 35,
    "research_question_rows": 4,
    "case_index_rows": 1785,
    "painting_index_rows": 50,
    "visual_asset_index_rows": 23964,
    "report_index_rows": 104,
    "filter_option_groups": 16,
}

DASHBOARD_SUMMARY["persistence_status"] = "completed"
DASHBOARD_SUMMARY["dashboard_asset_count"] = 15
DASHBOARD_SUMMARY["physical_output_file_count"] = 19
DASHBOARD_SUMMARY["roadmap_responsibility_count"] = 25

for table_name, frame in DASHBOARD_TABLES.items():
    atomic_write_csv(
        frame,
        OUTPUT_PATHS[TABLE_OUTPUT_KEYS[table_name]],
    )

    print(
        f"Persisted table: {table_name} "
        f"({len(frame)} rows)"
    )

for index_name, frame in DASHBOARD_INDEXES.items():
    atomic_write_csv(
        frame,
        OUTPUT_PATHS[INDEX_OUTPUT_KEYS[index_name]],
    )

    print(
        f"Persisted index: {index_name} "
        f"({len(frame)} rows)"
    )

atomic_write_json(
    FILTER_OPTIONS,
    OUTPUT_PATHS["filter_options_path"],
)

atomic_write_json(
    DASHBOARD_SUMMARY,
    OUTPUT_PATHS["dashboard_summary_path"],
)


RELOADED_DASHBOARD_TABLES = {}

for table_name, path_key in TABLE_OUTPUT_KEYS.items():
    reloaded = pd.read_csv(
        OUTPUT_PATHS[path_key],
        low_memory=False,
    )

    RELOADED_DASHBOARD_TABLES[table_name] = (
        validate_output_frame(
            reloaded,
            table_name,
        )
    )

RELOADED_DASHBOARD_INDEXES = {}

for index_name, path_key in INDEX_OUTPUT_KEYS.items():
    reloaded = pd.read_csv(
        OUTPUT_PATHS[path_key],
        low_memory=False,
    )

    RELOADED_DASHBOARD_INDEXES[index_name] = (
        validate_output_frame(
            reloaded,
            index_name,
        )
    )

with OUTPUT_PATHS["filter_options_path"].open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_FILTER_OPTIONS = json.load(handle)

with OUTPUT_PATHS["dashboard_summary_path"].open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_DASHBOARD_SUMMARY = json.load(handle)


OBSERVED_RELOADED_TABLE_COUNTS = {
    name: len(frame)
    for name, frame in RELOADED_DASHBOARD_TABLES.items()
}

OBSERVED_RELOADED_INDEX_COUNTS = {
    name: len(frame)
    for name, frame in RELOADED_DASHBOARD_INDEXES.items()
}

print("Dashboard data assets persisted and reloaded.")
print(
    "Reloaded table rows:",
    OBSERVED_RELOADED_TABLE_COUNTS,
)
print(
    "Reloaded index rows:",
    OBSERVED_RELOADED_INDEX_COUNTS,
)
print(
    "Filter-option groups:",
    len(RELOADED_FILTER_OPTIONS),
)
print("Persisted dashboard data assets: 15")

Persisted table: headline_findings (8 rows)
Persisted table: study_design (35 rows)
Persisted table: metric_framework (178 rows)
Persisted table: performance_summary (1241 rows)
Persisted table: sensitivity_summary (18951 rows)
Persisted table: uncertainty_summary (1723 rows)
Persisted table: trustworthiness_summary (284 rows)
Persisted table: compute_summary (35 rows)
Persisted table: research_question_coverage (4 rows)
Persisted index: case_index (1785 rows)
Persisted index: painting_index (50 rows)
Persisted index: visual_asset_index (23964 rows)
Persisted index: report_index (104 rows)
Dashboard data assets persisted and reloaded.
Reloaded table rows: {'headline_findings': 8, 'study_design': 35, 'metric_framework': 178, 'performance_summary': 1241, 'sensitivity_summary': 18951, 'uncertainty_summary': 1723, 'trustworthiness_summary': 284, 'compute_summary': 35, 'research_question_coverage': 4}
Reloaded index rows: {'case_index': 1785, 'painting_index': 50, 'visual_asset_index': 2396

In [60]:
def unique_in_order(values: list[str]) -> list[str]:
    """Return non-empty unique strings without changing order."""

    output = []

    for value in values:
        text = str(value).strip()

        if text and text not in output:
            output.append(text)

    return output


def frame_provenance(
    frame: pd.DataFrame,
) -> tuple[list[str], list[str]]:
    """Collect notebook and path provenance from normalized rows."""

    notebook_ids = []
    source_paths = []

    for column in [
        "source_notebook_ids_json",
        "evidence_source_notebook_ids_json",
    ]:
        if column not in frame.columns:
            continue

        for value in frame[column]:
            notebook_ids.extend(parse_json_list(value))

    if "source_notebook_id" in frame.columns:
        notebook_ids.extend(
            normalized_text(
                frame["source_notebook_id"]
            ).tolist()
        )

    if "source_paths_json" in frame.columns:
        for value in frame["source_paths_json"]:
            source_paths.extend(parse_json_list(value))

    return (
        unique_in_order(notebook_ids),
        unique_in_order(source_paths),
    )


def frame_page_ids(
    frame: pd.DataFrame,
    fallback: list[str],
) -> list[str]:
    """Read declared page IDs or return an explicit fallback."""

    if "page_id" not in frame.columns:
        return fallback

    values = [
        value
        for value in normalized_text(
            frame["page_id"]
        ).tolist()
        if value
    ]

    return unique_in_order(values) or fallback


DASHBOARD_DATA_ASSET_SPECS = []

for table_name, frame in DASHBOARD_TABLES.items():
    notebook_ids, source_paths = frame_provenance(
        frame
    )

    DASHBOARD_DATA_ASSET_SPECS.append(
        {
            "asset_key": f"dashboard.table.{table_name}",
            "asset_group": "dashboard_table",
            "path_key": TABLE_OUTPUT_KEYS[table_name],
            "schema_version": SETTINGS[
                "expected_output_schemas"
            ][table_name],
            "row_count": len(frame),
            "page_ids": frame_page_ids(
                frame,
                EXPECTED_PAGE_IDS,
            ),
            "source_notebook_ids": notebook_ids,
            "source_paths": source_paths,
        }
    )

INDEX_PAGE_FALLBACKS = {
    "case_index": ["case_explorer"],
    "painting_index": ["case_explorer"],
    "visual_asset_index": EXPECTED_PAGE_IDS,
    "report_index": ["reports_reproducibility"],
}

INDEX_SOURCE_PATH_OVERRIDES = {
    "case_index": [
        SETTINGS["inputs"]["explanation_cases_path"],
        SETTINGS["inputs"]["case_registry_path"],
        SETTINGS["inputs"]["case_report_index_path"],
        SETTINGS["inputs"]["painting_report_index_path"],
    ],
    "painting_index": [
        SETTINGS["inputs"]["artworks_path"],
        SETTINGS["inputs"]["explanation_cases_path"],
        SETTINGS["inputs"]["painting_report_index_path"],
    ],
    "visual_asset_index": [
        SETTINGS["inputs"]["spatial_map_manifest_path"],
        SETTINGS["inputs"]["local_map_manifest_path"],
        SETTINGS["inputs"]["uncertainty_map_manifest_path"],
        SETTINGS["inputs"]["semantic_map_manifest_path"],
        SETTINGS["inputs"]["representative_cases_path"],
        SETTINGS["inputs"]["damage_size_map_manifest_path"],
    ],
    "report_index": [
        SETTINGS["inputs"]["model_report_index_path"],
        SETTINGS["inputs"]["case_report_index_path"],
        SETTINGS["inputs"]["painting_report_index_path"],
        SETTINGS["inputs"]["final_report_path"],
        SETTINGS["inputs"]["final_limitations_path"],
    ],
}

for index_name, frame in DASHBOARD_INDEXES.items():
    notebook_ids, source_paths = frame_provenance(
        frame
    )

    source_paths = unique_in_order(
        source_paths
        + INDEX_SOURCE_PATH_OVERRIDES[index_name]
    )

    DASHBOARD_DATA_ASSET_SPECS.append(
        {
            "asset_key": f"dashboard.index.{index_name}",
            "asset_group": "dashboard_index",
            "path_key": INDEX_OUTPUT_KEYS[index_name],
            "schema_version": SETTINGS[
                "expected_output_schemas"
            ][index_name],
            "row_count": len(frame),
            "page_ids": frame_page_ids(
                frame,
                INDEX_PAGE_FALLBACKS[index_name],
            ),
            "source_notebook_ids": (
                notebook_ids or ["34"]
            ),
            "source_paths": source_paths,
        }
    )

DASHBOARD_DATA_ASSET_SPECS.append(
    {
        "asset_key": "dashboard.index.filter_options",
        "asset_group": "dashboard_index",
        "path_key": "filter_options_path",
        "schema_version": SETTINGS[
            "expected_output_schemas"
        ]["filter_options"],
        "row_count": len(FILTER_OPTIONS),
        "page_ids": EXPECTED_PAGE_IDS,
        "source_notebook_ids": ["01", "08", "27", "34"],
        "source_paths": [
            SETTINGS["inputs"]["artworks_path"],
            SETTINGS["inputs"]["region_policy_path"],
            SETTINGS["inputs"]["failure_taxonomy_path"],
            SETTINGS["inputs"]["model_cards_path"],
        ],
    }
)

DASHBOARD_DATA_ASSET_SPECS.append(
    {
        "asset_key": "dashboard.summary",
        "asset_group": "dashboard_metadata",
        "path_key": "dashboard_summary_path",
        "schema_version": DASHBOARD_PACKAGE_SCHEMA_VERSION,
        "row_count": 1,
        "page_ids": EXPECTED_PAGE_IDS,
        "source_notebook_ids": ["01-33", "34"],
        "source_paths": [
            OUTPUT_PATHS[spec["path_key"]]
            .relative_to(PROJECT_ROOT)
            .as_posix()
            for spec in DASHBOARD_DATA_ASSET_SPECS
        ],
    }
)

if len(DASHBOARD_DATA_ASSET_SPECS) != 15:
    raise ValueError(
        "Expected 15 dashboard data assets, "
        f"observed {len(DASHBOARD_DATA_ASSET_SPECS)}."
    )


DASHBOARD_ASSET_ROWS = []

for spec in DASHBOARD_DATA_ASSET_SPECS:
    path = OUTPUT_PATHS[spec["path_key"]]
    statistics = dashboard_path_statistics(path)
    relative_path = path.relative_to(
        PROJECT_ROOT
    ).as_posix()

    DASHBOARD_ASSET_ROWS.append(
        {
            "dashboard_asset_id": stable_id(
                "dashboard_asset",
                spec["asset_key"],
                relative_path,
            ),
            "asset_key": spec["asset_key"],
            "asset_group": spec["asset_group"],
            "page_ids_json": json_list(
                spec["page_ids"]
            ),
            "relative_path": relative_path,
            "format": (
                path.suffix.lower().lstrip(".")
                or "file"
            ),
            "schema_version": spec["schema_version"],
            "row_count": int(spec["row_count"]),
            "file_count": int(
                statistics["file_count"]
            ),
            "size_bytes": int(
                statistics["size_bytes"]
            ),
            "sha256": dashboard_sha256_file(path),
            "source_notebook_ids_json": json_list(
                spec["source_notebook_ids"]
            ),
            "source_paths_json": json_list(
                spec["source_paths"]
            ),
            "validation_status": "passed",
            "status": "ok",
            "issue": "",
        }
    )

DASHBOARD_ASSET_INDEX = pd.DataFrame(
    DASHBOARD_ASSET_ROWS
)

DASHBOARD_ASSET_INDEX = validate_output_frame(
    DASHBOARD_ASSET_INDEX,
    "dashboard_assets",
)

atomic_write_csv(
    DASHBOARD_ASSET_INDEX,
    OUTPUT_PATHS["dashboard_assets_path"],
)

RELOADED_DASHBOARD_ASSET_INDEX = pd.read_csv(
    OUTPUT_PATHS["dashboard_assets_path"],
    low_memory=False,
)

RELOADED_DASHBOARD_ASSET_INDEX = (
    validate_output_frame(
        RELOADED_DASHBOARD_ASSET_INDEX,
        "dashboard_assets",
    )
)

display(RELOADED_DASHBOARD_ASSET_INDEX)

print("Dashboard asset manifest persisted.")
print(
    "Dashboard asset records:",
    len(RELOADED_DASHBOARD_ASSET_INDEX),
)
print(
    "Manifest groups:",
    RELOADED_DASHBOARD_ASSET_INDEX[
        "asset_group"
    ].value_counts().to_dict(),
)

,dashboard_asset_id,asset_key,asset_group,page_ids_json,relative_path,format,schema_version,row_count,file_count,size_bytes,sha256,source_notebook_ids_json,source_paths_json,validation_status,status,issue
0,dashboard_asset_1d1e69a41bdb66706729ac57,dashboard.table.headline_findings,dashboard_table,"[""overview""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_headline_findings.v1,8,1,4941,02d16b6cae9b7a6594d2270faf15103a94107b6d9c8e96...,"[""01"", ""29"", ""34"", ""21"", ""33"", ""18"", ""22"", ""12...","[""outputs/01_dataset_verification/data/artwork...",passed,ok,NaN
1,dashboard_asset_6eed32c5bf6e0e9f9e27c4b0,dashboard.table.study_design,dashboard_table,"[""study_design""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_study_design.v1,35,1,13382,3a9ed2b892582746ec1c8aa42172bdb6f291f8301fa22d...,"[""01"", ""33"", ""08"", ""03"", ""04"", ""05"", ""06"", ""07...","[""outputs/01_dataset_verification/data/artwork...",passed,ok,NaN
2,dashboard_asset_a2706ff9cea046ce6d48dbc6,dashboard.table.metric_framework,dashboard_table,"[""metric_framework""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_metric_framework.v1,178,1,103827,ce390b97d329e20db56f729ef7b5cf156db277b32620ba...,"[""08"", ""21"", ""33""]","[""outputs/08_experiment_contracts_and_region_p...",passed,ok,NaN
3,dashboard_asset_4933c8d019796ef6e2d2ef45,dashboard.table.performance_summary,dashboard_table,"[""model_performance""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_performance_summary.v1,1241,1,872837,43319aa6e72dfe2d7f9c41ec723d28bd5204480f8b5df7...,"[""21""]","[""outputs/21_multi_model_comparison/metrics/mo...",passed,ok,NaN
4,dashboard_asset_2d28da4e3e1a04b273c64e08,dashboard.table.sensitivity_summary,dashboard_table,"[""robustness_uncertainty"", ""model_performance""...",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_sensitivity_summary.v1,18951,1,15278770,7d8a09353433c08759e59677dba22e37f08687eba4a6dd...,"[""23"", ""26"", ""12"", ""01"", ""24"", ""25"", ""13"", ""17...","[""outputs/23_damage_size_sensitivity_analysis/...",passed,ok,NaN
5,dashboard_asset_72e358bf7022893cea40d60e,dashboard.table.uncertainty_summary,dashboard_table,"[""robustness_uncertainty""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_uncertainty_summary.v1,1723,1,1286329,068a7632f07fdad261c10c2b2c7f4a7a7e79291672f427...,"[""18"", ""22"", ""10"", ""09"", ""12""]","[""outputs/18_diffusion_uncertainty_analysis/me...",passed,ok,NaN
6,dashboard_asset_723fad9ebe2d83544bdf1653,dashboard.table.trustworthiness_summary,dashboard_table,"[""trustworthiness_xai""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_trustworthiness_summary.v1,284,1,191736,4c79651663460f41bd4f499f3174c3cbf179497ac67467...,"[""27"", ""29"", ""28""]","[""outputs/27_failure_taxonomy_and_trustworthin...",passed,ok,NaN
7,dashboard_asset_7d533bc2a1b9a064a3c954be,dashboard.table.compute_summary,dashboard_table,"[""model_performance""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_compute_summary.v1,35,1,91114,7ab3f1fccf42a8da147931f4a8d0a535af59682bee8831...,"[""30""]","[""outputs/30_model_cards_compute_and_scalabili...",passed,ok,NaN
8,dashboard_asset_ad077a8f43b29ce9e0eff9f3,dashboard.table.research_question_coverage,dashboard_table,"[""metric_framework"", ""model_performance"", ""rob...",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_research_question_coverage.v1,4,1,4447,a0f7f101147332add582d4f658f410d04c3e162f33a743...,"[""08"", ""13"", ""14"", ""15"", ""16"", ""17"", ""20"", ""21...","[""outputs/08_experiment_contracts_and_region_p...",passed,ok,NaN
9,dashboard_asset_66287626ca370306710b219c,dashboard.index.case_index,dashboard_index,"[""case_explorer""]",outputs/34_final_streamlit_dashboard_assets/da...,csv,dashboard_case_index.v1,1785,1,5526378,9fbfa714adb4eb05acfa1709a2d81a533384bdb3294e62...,"[""10"", ""15"", ""16"", ""17"", ""20"", ""27"", ""28"", ""09...","[""output

Dashboard asset manifest persisted.
Dashboard asset records: 15
Manifest groups: {'dashboard_table': 9, 'dashboard_index': 5, 'dashboard_metadata': 1}


In [61]:
EXPECTED_TABLE_ROW_COUNTS = {
    "headline_findings": 8,
    "study_design": 35,
    "metric_framework": 178,
    "performance_summary": 1241,
    "sensitivity_summary": 18951,
    "uncertainty_summary": 1723,
    "trustworthiness_summary": 284,
    "compute_summary": 35,
    "research_question_coverage": 4,
}

EXPECTED_INDEX_ROW_COUNTS = {
    "case_index": 1785,
    "painting_index": 50,
    "visual_asset_index": 23964,
    "report_index": 104,
}

DATA_ASSET_PATHS = {
    OUTPUT_PATHS[spec["path_key"]].resolve()
    for spec in DASHBOARD_DATA_ASSET_SPECS
}

OBSERVED_DATA_ASSET_PATHS = {
    path.resolve()
    for path in OUTPUT_ROOT.rglob("*")
    if (
        path.is_file()
        and path.resolve()
        != OUTPUT_PATHS[
            "dashboard_assets_path"
        ].resolve()
    )
}

DASHBOARD_ASSET_CHECKSUM_FAILURES = []

for record in RELOADED_DASHBOARD_ASSET_INDEX.to_dict(
    orient="records"
):
    asset_path = (
        PROJECT_ROOT
        / str(record["relative_path"])
    ).resolve()

    observed_checksum = dashboard_sha256_file(
        asset_path
    )

    if observed_checksum != str(record["sha256"]):
        DASHBOARD_ASSET_CHECKSUM_FAILURES.append(
            str(record["asset_key"])
        )

ASSET_SOURCE_PATH_ERRORS = []

for row_index, value in (
    RELOADED_DASHBOARD_ASSET_INDEX[
        "source_paths_json"
    ].items()
):
    for source_path in parse_json_list(value):
        if not is_repo_relative(source_path):
            ASSET_SOURCE_PATH_ERRORS.append(
                f"row {row_index}: non-relative "
                f"path {source_path}"
            )
            continue

        if not (PROJECT_ROOT / source_path).exists():
            ASSET_SOURCE_PATH_ERRORS.append(
                f"row {row_index}: missing "
                f"path {source_path}"
            )

FINAL_PERSISTENCE_CHECKS = (
    (
        "dashboard_table_population",
        "All nine dashboard tables were persisted",
        EXPECTED_TABLE_ROW_COUNTS,
        OBSERVED_RELOADED_TABLE_COUNTS,
        (
            OBSERVED_RELOADED_TABLE_COUNTS
            == EXPECTED_TABLE_ROW_COUNTS
        ),
    ),
    (
        "dashboard_index_population",
        "All four dataframe indexes were persisted",
        EXPECTED_INDEX_ROW_COUNTS,
        OBSERVED_RELOADED_INDEX_COUNTS,
        (
            OBSERVED_RELOADED_INDEX_COUNTS
            == EXPECTED_INDEX_ROW_COUNTS
        ),
    ),
    (
        "filter_options_population",
        "The filter index contains all 16 fields",
        16,
        len(RELOADED_FILTER_OPTIONS),
        len(RELOADED_FILTER_OPTIONS) == 16,
    ),
    (
        "filter_options_status",
        "The filter index has valid status",
        "ok",
        RELOADED_FILTER_OPTIONS.get("status"),
        RELOADED_FILTER_OPTIONS.get("status") == "ok",
    ),
    (
        "dashboard_summary_status",
        "The dashboard summary records completed persistence",
        "completed",
        RELOADED_DASHBOARD_SUMMARY.get(
            "persistence_status"
        ),
        (
            RELOADED_DASHBOARD_SUMMARY.get(
                "persistence_status"
            )
            == "completed"
        ),
    ),
    (
        "dashboard_summary_table_counts",
        "Summary table counts match persisted tables",
        EXPECTED_TABLE_ROW_COUNTS,
        RELOADED_DASHBOARD_SUMMARY[
            "table_rows"
        ],
        (
            RELOADED_DASHBOARD_SUMMARY[
                "table_rows"
            ]
            == EXPECTED_TABLE_ROW_COUNTS
        ),
    ),
    (
        "dashboard_summary_index_counts",
        "Summary index counts match persisted indexes",
        {
            **EXPECTED_INDEX_ROW_COUNTS,
            "filter_options": 16,
        },
        RELOADED_DASHBOARD_SUMMARY[
            "index_rows"
        ],
        (
            RELOADED_DASHBOARD_SUMMARY[
                "index_rows"
            ]
            == {
                **EXPECTED_INDEX_ROW_COUNTS,
                "filter_options": 16,
            }
        ),
    ),
    (
        "dashboard_asset_population",
        "Exactly 15 data assets are registered",
        15,
        len(RELOADED_DASHBOARD_ASSET_INDEX),
        len(RELOADED_DASHBOARD_ASSET_INDEX) == 15,
    ),
    (
        "dashboard_asset_identifiers",
        "Dashboard asset identifiers are unique",
        15,
        RELOADED_DASHBOARD_ASSET_INDEX[
            "dashboard_asset_id"
        ].nunique(),
        (
            RELOADED_DASHBOARD_ASSET_INDEX[
                "dashboard_asset_id"
            ].nunique()
            == 15
        ),
    ),
    (
        "dashboard_asset_checksums",
        "All data-asset checksums match",
        [],
        DASHBOARD_ASSET_CHECKSUM_FAILURES,
        not DASHBOARD_ASSET_CHECKSUM_FAILURES,
    ),
    (
        "dashboard_asset_source_paths",
        "All data-asset source paths exist and are relative",
        [],
        ASSET_SOURCE_PATH_ERRORS,
        not ASSET_SOURCE_PATH_ERRORS,
    ),
    (
        "dashboard_data_asset_set",
        "Only the 15 declared data assets were written",
        sorted(
            path.relative_to(PROJECT_ROOT).as_posix()
            for path in DATA_ASSET_PATHS
        ),
        sorted(
            path.relative_to(PROJECT_ROOT).as_posix()
            for path in OBSERVED_DATA_ASSET_PATHS
        ),
        OBSERVED_DATA_ASSET_PATHS
        == DATA_ASSET_PATHS,
    ),
    (
        "complete_case_population",
        "Every approved candidate remains filterable",
        1785,
        len(RELOADED_DASHBOARD_INDEXES["case_index"]),
        (
            len(
                RELOADED_DASHBOARD_INDEXES[
                    "case_index"
                ]
            )
            == 1785
        ),
    ),
    (
        "complete_visual_population",
        "All validated visual records remain indexed",
        23964,
        len(
            RELOADED_DASHBOARD_INDEXES[
                "visual_asset_index"
            ]
        ),
        (
            len(
                RELOADED_DASHBOARD_INDEXES[
                    "visual_asset_index"
                ]
            )
            == 23964
        ),
    ),
    (
        "grouped_analysis_packaged",
        "Notebook 26 evidence is persisted for filtering",
        {
            "grouped_statistical_analysis": 4174,
            "metric_correlations": 1008,
            "ranking_stability": 1800,
        },
        (
            RELOADED_DASHBOARD_TABLES[
                "sensitivity_summary"
            ]["analysis_family"]
            .value_counts()
            .loc[
                [
                    "grouped_statistical_analysis",
                    "metric_correlations",
                    "ranking_stability",
                ]
            ]
            .to_dict()
        ),
        (
            RELOADED_DASHBOARD_TABLES[
                "sensitivity_summary"
            ]["analysis_family"]
            .value_counts()
            .loc[
                [
                    "grouped_statistical_analysis",
                    "metric_correlations",
                    "ranking_stability",
                ]
            ]
            .to_dict()
            == {
                "grouped_statistical_analysis": 4174,
                "metric_correlations": 1008,
                "ranking_stability": 1800,
            }
        ),
    ),
    (
        "roadmap_traceability",
        "All 25 roadmap responsibilities are covered",
        25,
        int(ROADMAP_TRACEABILITY["passed"].sum()),
        (
            len(ROADMAP_TRACEABILITY) == 25
            and ROADMAP_TRACEABILITY["passed"].all()
        ),
    ),
    (
        "presentation_only_boundary",
        "The package creates no new scientific evidence",
        False,
        RELOADED_DASHBOARD_SUMMARY[
            "creates_new_scientific_evidence"
        ],
        (
            RELOADED_DASHBOARD_SUMMARY[
                "creates_new_scientific_evidence"
            ]
            is False
        ),
    ),
)

for check_id, description, expected, observed, passed in (
    FINAL_PERSISTENCE_CHECKS
):
    VALIDATION.add(
        validation_stage="batch_10_persistence",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else (
                "The persisted dashboard package differs "
                "from the approved contract."
            )
        ),
    )

VALIDATION.raise_for_blocking()

FINAL_VALIDATION = VALIDATION.to_dataframe()
FINAL_VALIDATION_SUMMARY = VALIDATION.summary()

atomic_write_csv(
    FINAL_VALIDATION,
    OUTPUT_PATHS["validation_path"],
)

RELOADED_FINAL_VALIDATION = pd.read_csv(
    OUTPUT_PATHS["validation_path"],
    low_memory=False,
)

display(
    RELOADED_FINAL_VALIDATION.tail(
        len(FINAL_PERSISTENCE_CHECKS)
    )[
        [
            "check_id",
            "severity",
            "passed",
            "observed",
        ]
    ]
)

print("Consolidated validation persisted.")
print("Validation rows:", len(RELOADED_FINAL_VALIDATION))
print(
    "Blocking failures:",
    FINAL_VALIDATION_SUMMARY[
        "blocking_failure_count"
    ],
)
print(
    "Overall validation passed:",
    FINAL_VALIDATION_SUMMARY["overall_passed"],
)

,check_id,severity,passed,observed
393,dashboard_table_population,blocking,True,"{""compute_summary"": 35, ""headline_findings"": 8..."
394,dashboard_index_population,blocking,True,"{""case_index"": 1785, ""painting_index"": 50, ""re..."
395,filter_options_population,blocking,True,16
396,filter_options_status,blocking,True,ok
397,dashboard_summary_status,blocking,True,completed
398,dashboard_summary_table_counts,blocking,True,"{""compute_summary"": 35, ""headline_findings"": 8..."
399,dashboard_summary_index_counts,blocking,True,"{""case_index"": 1785, ""filter_options"": 16, ""pa..."
400,dashboard_asset_population,blocking,True,15
401,dashboard_asset_identifiers,blocking,True,15
402,dashboard_asset_checksums,blocking,True,[]


Consolidated validation persisted.
Validation rows: 410
Blocking failures: 0
Overall validation passed: True


In [62]:
ARTIFACT_RECORD_PAYLOADS = []

for spec in DASHBOARD_DATA_ASSET_SPECS:
    path = OUTPUT_PATHS[spec["path_key"]]

    artifact_type = {
        "dashboard_table": "normalized_dashboard_table",
        "dashboard_index": "dashboard_index",
        "dashboard_metadata": "dashboard_metadata",
    }[spec["asset_group"]]

    ARTIFACT_RECORD_PAYLOADS.append(
        build_artifact_record(
            artifact_key=spec["asset_key"],
            producer_notebook=SETTINGS[
                "notebook_stem"
            ],
            path=path,
            artifact_type=artifact_type,
            artifact_role=spec["asset_key"].replace(
                ".",
                "_",
            ),
            schema_version=spec["schema_version"],
            dataset_scope=SETTINGS["dataset_scope"],
            experiment_id="dashboard_presentation",
            row_count=int(spec["row_count"]),
            project_root=PROJECT_ROOT,
        )
    )

ARTIFACT_RECORD_PAYLOADS.extend(
    [
        build_artifact_record(
            artifact_key="dashboard.asset_manifest",
            producer_notebook=SETTINGS[
                "notebook_stem"
            ],
            path=OUTPUT_PATHS[
                "dashboard_assets_path"
            ],
            artifact_type="dashboard_asset_manifest",
            artifact_role=(
                "dashboard_input_file_registry"
            ),
            schema_version=SETTINGS[
                "expected_output_schemas"
            ]["dashboard_assets"],
            dataset_scope=SETTINGS[
                "dataset_scope"
            ],
            experiment_id="dashboard_presentation",
            row_count=len(
                RELOADED_DASHBOARD_ASSET_INDEX
            ),
            project_root=PROJECT_ROOT,
        ),
        build_artifact_record(
            artifact_key="dashboard.validation",
            producer_notebook=SETTINGS[
                "notebook_stem"
            ],
            path=OUTPUT_PATHS["validation_path"],
            artifact_type="validation_table",
            artifact_role=(
                "consolidated_dashboard_asset_checks"
            ),
            schema_version="validation_checks.v1",
            dataset_scope=SETTINGS[
                "dataset_scope"
            ],
            experiment_id="dashboard_presentation",
            row_count=len(
                RELOADED_FINAL_VALIDATION
            ),
            project_root=PROJECT_ROOT,
        ),
    ]
)

FINAL_ARTIFACT_RECORDS = (
    artifact_records_dataframe(
        ARTIFACT_RECORD_PAYLOADS
    )
)

if len(FINAL_ARTIFACT_RECORDS) != 17:
    raise ValueError(
        "Expected 17 universal artifact records, "
        f"observed {len(FINAL_ARTIFACT_RECORDS)}."
    )

atomic_write_csv(
    FINAL_ARTIFACT_RECORDS,
    OUTPUT_PATHS["artifacts_path"],
)

RELOADED_ARTIFACT_RECORDS = pd.read_csv(
    OUTPUT_PATHS["artifacts_path"],
    low_memory=False,
)

if len(RELOADED_ARTIFACT_RECORDS) != 17:
    raise ValueError(
        "Reloaded artifact-record count changed."
    )

display(RELOADED_ARTIFACT_RECORDS)

print("Universal artifact manifest persisted.")
print(
    "Artifact records:",
    len(RELOADED_ARTIFACT_RECORDS),
)
print(
    "Artifact validation states:",
    RELOADED_ARTIFACT_RECORDS[
        "validation_status"
    ].value_counts().to_dict(),
)

,artifact_id,artifact_key,producer_notebook,artifact_type,artifact_role,relative_path,format,dataset_scope,experiment_id,schema_version,row_count,file_count,size_bytes,checksum,validation_status
0,artifact_3b74a8261018dc221f16,dashboard.table.headline_findings,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_headline_findings,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_headline_findings.v1,8,1,4941,02d16b6cae9b7a6594d2270faf15103a94107b6d9c8e96...,passed
1,artifact_cbd3412723d9ad8fb83f,dashboard.table.study_design,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_study_design,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_study_design.v1,35,1,13382,3a9ed2b892582746ec1c8aa42172bdb6f291f8301fa22d...,passed
2,artifact_b651a6b6f2d37174cfd5,dashboard.table.metric_framework,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_metric_framework,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_metric_framework.v1,178,1,103827,ce390b97d329e20db56f729ef7b5cf156db277b32620ba...,passed
3,artifact_118b28e9f20a02945d52,dashboard.table.performance_summary,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_performance_summary,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_performance_summary.v1,1241,1,872837,43319aa6e72dfe2d7f9c41ec723d28bd5204480f8b5df7...,passed
4,artifact_32361f20d3ece99b4944,dashboard.table.sensitivity_summary,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_sensitivity_summary,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_sensitivity_summary.v1,18951,1,15278770,7d8a09353433c08759e59677dba22e37f08687eba4a6dd...,passed
5,artifact_0da5ef77d7364d7ea6ac,dashboard.table.uncertainty_summary,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_uncertainty_summary,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_uncertainty_summary.v1,1723,1,1286329,068a7632f07fdad261c10c2b2c7f4a7a7e79291672f427...,passed
6,artifact_99db7592409f01c3f5ee,dashboard.table.trustworthiness_summary,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_trustworthiness_summary,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_trustworthiness_summary.v1,284,1,191736,4c79651663460f41bd4f499f3174c3cbf179497ac67467...,passed
7,artifact_3f6b64eeeb46a27ca31b,dashboard.table.compute_summary,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_compute_summary,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_compute_summary.v1,35,1,91114,7ab3f1fccf42a8da147931f4a8d0a535af59682bee8831...,passed
8,artifact_591f034e7d64f4da8e4d,dashboard.table.research_question_coverage,34_final_streamlit_dashboard_assets,normalized_dashboard_table,dashboard_table_research_question_coverage,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_research_question_coverage.v1,4,1,4447,a0f7f101147332add582d4f658f410d04c3e162f33a743...,passed
9,artifact_f44adb73cba0b9b9ede0,dashboard.index.case_index,34_final_streamlit_dashboard_assets,dashboard_index,dashboard_index_case_index,outputs/34_final_streamlit_dashboard_assets/da...,csv,controlled_50,dashboard_presentation,dashboard_case_index.v1,1785,1,5526378,9fbfa714adb4eb05acfa1709a2d81a533384bdb3294e62...,passed


Universal artifact manifest persisted.
Artifact records: 17
Artifact validation states: {'passed': 17}


In [63]:
MANIFEST_INPUT_PATHS = {
    "configuration.dashboard_assets": CONFIG_PATH,
    "configuration.evidence_coverage": (
        PROJECT_ROOT
        / SETTINGS["inputs"]["evidence_coverage_path"]
    ),
    "governance.implementation_guidelines": (
        PROJECT_ROOT
        / SETTINGS["inputs"][
            "implementation_guidelines_path"
        ]
    ),
    "governance.notebook_roadmap": (
        PROJECT_ROOT
        / SETTINGS["inputs"]["notebook_roadmap_path"]
    ),
    "governance.evidence_dependency_audit": (
        PROJECT_ROOT
        / SETTINGS["inputs"][
            "evidence_dependency_audit_path"
        ]
    ),
    "inventory.project_file_inventory": INVENTORY_PATH,
    "inventory.inventory_run": INVENTORY_RUN_PATH,
    "inventory.project_paths": PROJECT_PATHS_PATH,
}

for input_key in SETTINGS["input_table_contracts"]:
    MANIFEST_INPUT_PATHS[
        f"table.{input_key}"
    ] = (
        PROJECT_ROOT
        / SETTINGS["inputs"][input_key]
    )

for notebook_id, relative_path in (
    SETTINGS["upstream_manifests"].items()
):
    MANIFEST_INPUT_PATHS[
        f"upstream_manifest.{notebook_id}"
    ] = PROJECT_ROOT / relative_path


MANIFEST_INPUT_RECORDS = [
    {
        "input_key": input_key,
        "relative_path": path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "checksum": manifest_sha256_path(path),
    }
    for input_key, path in sorted(
        MANIFEST_INPUT_PATHS.items()
    )
]


EXPECTED_FINAL_OUTPUT_PATHS = {
    *{
        OUTPUT_PATHS[spec["path_key"]].resolve()
        for spec in DASHBOARD_DATA_ASSET_SPECS
    },
    OUTPUT_PATHS["dashboard_assets_path"].resolve(),
    OUTPUT_PATHS["validation_path"].resolve(),
    OUTPUT_PATHS["artifacts_path"].resolve(),
    OUTPUT_PATHS["run_manifest_path"].resolve(),
}

if len(EXPECTED_FINAL_OUTPUT_PATHS) != 19:
    raise ValueError(
        "Expected 19 physical output paths, "
        f"observed {len(EXPECTED_FINAL_OUTPUT_PATHS)}."
    )


MANIFEST_OUTPUT_RECORDS = []

for output_path in sorted(
    EXPECTED_FINAL_OUTPUT_PATHS,
    key=lambda path: path.as_posix(),
):
    relative_path = output_path.relative_to(
        PROJECT_ROOT
    ).as_posix()

    suffix = output_path.suffix.lower().lstrip(".")

    artifact_type = {
        "csv": "csv",
        "json": "json",
    }.get(suffix, suffix or "file")

    MANIFEST_OUTPUT_RECORDS.append(
        {
            "artifact_key": (
                "output."
                + output_path.relative_to(
                    OUTPUT_ROOT
                )
                .as_posix()
                .replace("/", ".")
            ),
            "relative_path": relative_path,
            "artifact_type": artifact_type,
        }
    )


UPSTREAM_RUN_IDS = {
    notebook_id: manifest.get("run_id", "")
    for notebook_id, manifest
    in UPSTREAM_MANIFESTS.items()
}

CONFIGURATION_CHECKSUMS = configuration_checksums(
    [
        CONFIG_PATH,
        PROJECT_ROOT
        / SETTINGS["inputs"]["evidence_coverage_path"],
    ],
    PROJECT_ROOT,
)

RUN_COMPLETED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

OBSERVED_FINAL_COUNTS = {
    **EXPECTED_FINAL_COUNTS,
    "validation_rows": len(
        RELOADED_FINAL_VALIDATION
    ),
    "manifest_inputs": len(
        MANIFEST_INPUT_RECORDS
    ),
    "manifest_outputs": len(
        MANIFEST_OUTPUT_RECORDS
    ),
}

FINAL_RUN_MANIFEST = build_run_manifest(
    notebook_id=SETTINGS["notebook_id"],
    notebook_name=SETTINGS["notebook_stem"],
    origin=(
        "Consolidates Existing Previous Versions "
        "of Notebooks 29 and 34, Pre-refactor"
    ),
    run_status="completed",
    started_at_utc=RUN_STARTED_AT_UTC,
    completed_at_utc=RUN_COMPLETED_AT_UTC,
    inventory_run_id=str(
        INVENTORY_RUN["inventory_run_id"]
    ),
    dataset_versions={
        "dataset_id": SETTINGS["dataset_id"],
        "dataset_version": SETTINGS[
            "dataset_version"
        ],
        "dataset_scope": SETTINGS[
            "dataset_scope"
        ],
        "configuration_version": CONFIG[
            "config_version"
        ],
        "configuration_schema_version": (
            CONFIG_SCHEMA_VERSION
        ),
        "dashboard_schema_version": (
            DASHBOARD_PACKAGE_SCHEMA_VERSION
        ),
        "upstream_run_ids": UPSTREAM_RUN_IDS,
    },
    configuration_paths=[
        path.relative_to(PROJECT_ROOT).as_posix()
        for path in [
            CONFIG_PATH,
            PROJECT_ROOT
            / SETTINGS["inputs"][
                "evidence_coverage_path"
            ],
        ]
    ],
    configuration_checksums_by_path=(
        CONFIGURATION_CHECKSUMS
    ),
    helper_versions={
        "dashboard_assets": (
            DASHBOARD_ASSETS_MODULE_VERSION
        ),
        "manifests": MANIFESTS_MODULE_VERSION,
        "validation": VALIDATION_MODULE_VERSION,
        "paths": PATHS_MODULE_VERSION,
    },
    inputs=MANIFEST_INPUT_RECORDS,
    outputs=MANIFEST_OUTPUT_RECORDS,
    expected_counts=EXPECTED_FINAL_COUNTS,
    observed_counts=OBSERVED_FINAL_COUNTS,
    validation_summary=FINAL_VALIDATION_SUMMARY,
    known_limitations=[
        str(value)
        for value in SETTINGS["limitations"]
    ],
    project_root=PROJECT_ROOT,
    package_names=(
        "pandas",
        "PyYAML",
    ),
    hardware={
        "analysis_mode": (
            "presentation_only_dashboard_asset_packaging"
        ),
        "restoration_inference_performed": False,
        "new_metric_computation_performed": False,
        "new_statistical_test_performed": False,
        "dashboard_application_executed": False,
    },
)

FINAL_RUN_MANIFEST.update(
    {
        "validation_status": "passed",
        "refactor_status": "refactored",
        "completion_gate_passed": True,
        "creates_new_scientific_evidence": False,
        "dashboard_page_count": 8,
        "dashboard_table_count": 9,
        "dashboard_index_count": 5,
        "dashboard_asset_count": 15,
        "roadmap_responsibility_count": 25,
        "artifact_manifest_checksum": (
            dashboard_sha256_file(
                OUTPUT_PATHS["artifacts_path"]
            )
        ),
        "output_checksums": {
            str(record["artifact_key"]): str(
                record["checksum"]
            )
            for record in (
                RELOADED_ARTIFACT_RECORDS.to_dict(
                    orient="records"
                )
            )
        },
    }
)

atomic_write_json(
    FINAL_RUN_MANIFEST,
    OUTPUT_PATHS["run_manifest_path"],
)

print("Completed run manifest persisted.")
print("Manifest inputs:", len(MANIFEST_INPUT_RECORDS))
print("Manifest outputs:", len(MANIFEST_OUTPUT_RECORDS))
print("Expected physical files: 19")
print("Run status: completed")

Completed run manifest persisted.
Manifest inputs: 82
Manifest outputs: 19
Expected physical files: 19
Run status: completed


In [64]:
with OUTPUT_PATHS["run_manifest_path"].open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_RUN_MANIFEST = json.load(handle)

with OUTPUT_PATHS["dashboard_summary_path"].open(
    "r",
    encoding="utf-8-sig",
) as handle:
    FINAL_RELOADED_DASHBOARD_SUMMARY = json.load(
        handle
    )

FINAL_RELOADED_VALIDATION = pd.read_csv(
    OUTPUT_PATHS["validation_path"],
    low_memory=False,
)

FINAL_RELOADED_ARTIFACTS = pd.read_csv(
    OUTPUT_PATHS["artifacts_path"],
    low_memory=False,
)

FINAL_RELOADED_DASHBOARD_ASSETS = pd.read_csv(
    OUTPUT_PATHS["dashboard_assets_path"],
    low_memory=False,
)

OBSERVED_FINAL_OUTPUT_PATHS = {
    path.resolve()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
}

TEMPORARY_OUTPUT_FILES = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OBSERVED_FINAL_OUTPUT_PATHS
    if (
        ".tmp" in path.name.lower()
        or ".building" in path.name.lower()
        or path.suffix.lower()
        in {".temp", ".part", ".partial"}
        or path.name.startswith(".")
    )
)

PASSED_VALUES = (
    FINAL_RELOADED_VALIDATION["passed"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

FINAL_BLOCKING_FAILURES = int(
    (
        ~PASSED_VALUES
        & FINAL_RELOADED_VALIDATION[
            "severity"
        ]
        .astype(str)
        .str.lower()
        .eq("blocking")
    ).sum()
)

FINAL_WARNING_FAILURES = int(
    (
        ~PASSED_VALUES
        & FINAL_RELOADED_VALIDATION[
            "severity"
        ]
        .astype(str)
        .str.lower()
        .eq("warning")
    ).sum()
)


FINAL_DASHBOARD_CHECKSUM_FAILURES = []

for record in (
    FINAL_RELOADED_DASHBOARD_ASSETS
    .to_dict(orient="records")
):
    path = (
        PROJECT_ROOT
        / str(record["relative_path"])
    ).resolve()

    if (
        dashboard_sha256_file(path)
        != str(record["sha256"])
    ):
        FINAL_DASHBOARD_CHECKSUM_FAILURES.append(
            str(record["asset_key"])
        )


FINAL_ARTIFACT_CHECKSUM_FAILURES = []

for record in (
    FINAL_RELOADED_ARTIFACTS
    .to_dict(orient="records")
):
    path = (
        PROJECT_ROOT
        / str(record["relative_path"])
    ).resolve()

    if (
        manifest_sha256_path(path)
        != str(record["checksum"])
    ):
        FINAL_ARTIFACT_CHECKSUM_FAILURES.append(
            str(record["artifact_key"])
        )


MANIFEST_OUTPUT_PATHS = {
    (
        PROJECT_ROOT
        / str(record["relative_path"])
    ).resolve()
    for record in RELOADED_RUN_MANIFEST["outputs"]
}

MANIFEST_COMPLETION_VALID = bool(
    RELOADED_RUN_MANIFEST["run_status"]
    == "completed"
    and RELOADED_RUN_MANIFEST[
        "validation_status"
    ]
    == "passed"
    and RELOADED_RUN_MANIFEST[
        "refactor_status"
    ]
    == "refactored"
    and RELOADED_RUN_MANIFEST[
        "completion_gate_passed"
    ]
    is True
    and RELOADED_RUN_MANIFEST[
        "creates_new_scientific_evidence"
    ]
    is False
)

FINAL_COMPLETION_FAILURES = []

if (
    OBSERVED_FINAL_OUTPUT_PATHS
    != EXPECTED_FINAL_OUTPUT_PATHS
):
    FINAL_COMPLETION_FAILURES.append(
        "physical output identity"
    )

if len(OBSERVED_FINAL_OUTPUT_PATHS) != 19:
    FINAL_COMPLETION_FAILURES.append(
        "physical output count"
    )

if TEMPORARY_OUTPUT_FILES:
    FINAL_COMPLETION_FAILURES.append(
        "temporary files"
    )

if FINAL_BLOCKING_FAILURES:
    FINAL_COMPLETION_FAILURES.append(
        "blocking validation failures"
    )

if FINAL_WARNING_FAILURES:
    FINAL_COMPLETION_FAILURES.append(
        "warning validation failures"
    )

if len(FINAL_RELOADED_DASHBOARD_ASSETS) != 15:
    FINAL_COMPLETION_FAILURES.append(
        "dashboard asset count"
    )

if len(FINAL_RELOADED_ARTIFACTS) != 17:
    FINAL_COMPLETION_FAILURES.append(
        "artifact record count"
    )

if FINAL_DASHBOARD_CHECKSUM_FAILURES:
    FINAL_COMPLETION_FAILURES.append(
        "dashboard asset checksums"
    )

if FINAL_ARTIFACT_CHECKSUM_FAILURES:
    FINAL_COMPLETION_FAILURES.append(
        "artifact checksums"
    )

if (
    RELOADED_RUN_MANIFEST[
        "artifact_manifest_checksum"
    ]
    != dashboard_sha256_file(
        OUTPUT_PATHS["artifacts_path"]
    )
):
    FINAL_COMPLETION_FAILURES.append(
        "artifact manifest checksum"
    )

if (
    MANIFEST_OUTPUT_PATHS
    != EXPECTED_FINAL_OUTPUT_PATHS
):
    FINAL_COMPLETION_FAILURES.append(
        "run manifest output identity"
    )

if len(RELOADED_RUN_MANIFEST["outputs"]) != 19:
    FINAL_COMPLETION_FAILURES.append(
        "run manifest output count"
    )

if (
    int(
        RELOADED_RUN_MANIFEST[
            "validation_summary"
        ]["check_count"]
    )
    != len(FINAL_RELOADED_VALIDATION)
):
    FINAL_COMPLETION_FAILURES.append(
        "run manifest validation count"
    )

if not MANIFEST_COMPLETION_VALID:
    FINAL_COMPLETION_FAILURES.append(
        "run manifest completion status"
    )

if (
    FINAL_RELOADED_DASHBOARD_SUMMARY[
        "persistence_status"
    ]
    != "completed"
):
    FINAL_COMPLETION_FAILURES.append(
        "dashboard summary persistence status"
    )

if not ROADMAP_TRACEABILITY["passed"].all():
    FINAL_COMPLETION_FAILURES.append(
        "roadmap traceability"
    )


FINAL_OUTPUT_PROFILE = pd.DataFrame(
    [
        {
            "format": (
                path.suffix.lower().lstrip(".")
                or "file"
            ),
            "file_count": 1,
            "size_bytes": int(path.stat().st_size),
        }
        for path in OBSERVED_FINAL_OUTPUT_PATHS
    ]
).groupby(
    "format",
    as_index=False,
).agg(
    file_count=("file_count", "sum"),
    total_size_bytes=("size_bytes", "sum"),
)

FINAL_OUTPUT_PROFILE["total_size_mib"] = (
    FINAL_OUTPUT_PROFILE["total_size_bytes"]
    / (1024 ** 2)
).round(2)

FINAL_STAGE_PROFILE = (
    FINAL_RELOADED_VALIDATION.groupby(
        "validation_stage",
        as_index=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=(
            "passed",
            lambda values: int(
                values.astype(str)
                .str.strip()
                .str.lower()
                .eq("true")
                .sum()
            ),
        ),
    )
)

COMPLETION_DIAGNOSTICS = pd.DataFrame(
    [
        {
            "contract": "physical_output_set",
            "failures": int(
                OBSERVED_FINAL_OUTPUT_PATHS
                != EXPECTED_FINAL_OUTPUT_PATHS
            ),
        },
        {
            "contract": "temporary_files",
            "failures": len(
                TEMPORARY_OUTPUT_FILES
            ),
        },
        {
            "contract": "blocking_validation",
            "failures": FINAL_BLOCKING_FAILURES,
        },
        {
            "contract": "warning_validation",
            "failures": FINAL_WARNING_FAILURES,
        },
        {
            "contract": "dashboard_checksums",
            "failures": len(
                FINAL_DASHBOARD_CHECKSUM_FAILURES
            ),
        },
        {
            "contract": "artifact_checksums",
            "failures": len(
                FINAL_ARTIFACT_CHECKSUM_FAILURES
            ),
        },
        {
            "contract": "manifest_completion",
            "failures": int(
                not MANIFEST_COMPLETION_VALID
            ),
        },
        {
            "contract": "roadmap_traceability",
            "failures": int(
                not ROADMAP_TRACEABILITY[
                    "passed"
                ].all()
            ),
        },
    ]
)

if FINAL_COMPLETION_FAILURES:
    display(COMPLETION_DIAGNOSTICS)

    raise RuntimeError(
        "Notebook 34 completion gate failed: "
        + ", ".join(FINAL_COMPLETION_FAILURES)
    )

display(FINAL_STAGE_PROFILE)
display(COMPLETION_DIAGNOSTICS)
display(ROADMAP_TRACEABILITY)
display(FINAL_OUTPUT_PROFILE)

print("Notebook 34 completion gate passed.")
print(
    "Validation rows:",
    len(FINAL_RELOADED_VALIDATION),
)
print("Blocking failures:", FINAL_BLOCKING_FAILURES)
print("Warning failures:", FINAL_WARNING_FAILURES)
print(
    "Physical files:",
    f"{len(OBSERVED_FINAL_OUTPUT_PATHS)}/19",
)
print(
    "Dashboard asset records:",
    f"{len(FINAL_RELOADED_DASHBOARD_ASSETS)}/15",
)
print(
    "Artifact records:",
    f"{len(FINAL_RELOADED_ARTIFACTS)}/17",
)
print(
    "Roadmap responsibilities:",
    f"{int(ROADMAP_TRACEABILITY['passed'].sum())}/25",
)
print(
    "Run status:",
    RELOADED_RUN_MANIFEST["run_status"],
)
print(
    "Completion gate:",
    RELOADED_RUN_MANIFEST[
        "completion_gate_passed"
    ],
)
print("Notebook 34 is complete.")

,validation_stage,checks,passed
0,batch_10_persistence,17,17
1,batch_1_contract,13,13
2,batch_1_final_preflight,11,11
3,batch_1_inventory_governance,11,11
4,batch_1_preflight,222,222
5,batch_2_population_normalization,16,16
6,batch_3_overview_study_design,17,17
7,batch_4_metric_framework,14,14
8,batch_5_model_performance,14,14
9,batch_6_robustness_uncertainty,15,15


,contract,failures
0,physical_output_set,0
1,temporary_files,0
2,blocking_validation,0
3,warning_validation,0
4,dashboard_checksums,0
5,artifact_checksums,0
6,manifest_completion,0
7,roadmap_traceability,0


,responsibility_id,responsibility,dashboard_evidence,observed_count,passed
0,overview,Overview and headline findings,headline_findings,8,True
1,dataset_and_bias,Dataset composition and bias evidence,study_design and painting_index,50,True
2,canonical_damage,Canonical damage design,study_design and case_index,250,True
3,damage_size_sensitivity,Damage-size sensitivity,sensitivity_summary,1901,True
4,mask_robustness,Mask robustness,sensitivity_summary,5373,True
5,synthetic_degradation,Synthetic-degradation sensitivity,sensitivity_summary,4695,True
6,model_stack,Complete evaluated model stack,compute_summary and report_index,4,True
7,model_comparison,Overall and conditional model comparison,performance_summary,1241,True
8,metric_region_policy,Metric-region policy,metric_framework,143,True
9,metric_ablation,Metric and region-policy ablation,trustworthiness_summary,7710,True


,format,file_count,total_size_bytes,total_size_mib
0,csv,16,36341679,34.66
1,json,3,54228,0.05


Notebook 34 completion gate passed.
Validation rows: 410
Blocking failures: 0
Warning failures: 0
Physical files: 19/19
Dashboard asset records: 15/15
Artifact records: 17/17
Roadmap responsibilities: 25/25
Run status: completed
Completion gate: True
Notebook 34 is complete.
